# The Clown Project — V0.1

## Notebook 02: Visible Book Reconstruction

**Notebook:** `02_VISIBLE_BOOK_RECONSTRUCTION.ipynb`  
**Pipeline version:** `V0.1`  
**Source run:** `BTCUSDT_spot_20260710T063746Z_c8b5bf12`  
**V0.1 run ID:** `v0_1_20260714T090616Z_e82325081a81`  
**Operating mode:** `ENGINEERING_REPRODUCTION_MODE`  
**Initial status:** `NOT EVALUATED`

### Purpose

This notebook independently reconstructs the visible Binance Spot BTCUSDT market-by-price order book from the immutable REST snapshot and differential-depth stream.

It will:

- Initialize the book from the validated REST snapshot
- Enforce the validated snapshot-to-stream bridge
- Apply depth updates in authoritative collector-sequence order
- Interpret zero quantity as a level-deletion instruction
- Preserve original update identifiers, collector sequence, exchange time, and local receipt time
- Retain the visible top ten bid and ask levels after each accepted update
- Calculate book-state diagnostics and observable market-state quantities
- Record rejected updates, continuity failures, resynchronization requirements, and reconstruction boundaries
- Reconcile the completed V0.1 reconstruction against V0.0 reference artifacts only after independent reconstruction is complete

### Input contract

Authoritative inputs must be loaded from saved disk artifacts produced or verified by Notebooks 00 and 01.

Required inputs include:

- The frozen V0.1 run contract
- The Notebook 01 output manifest
- The Notebook 01-to-Notebook 02 handoff
- The canonical REST-snapshot representation
- The canonical raw depth-update table
- The validated schema and timestamp contracts
- The validated snapshot-bridge and update-continuity results

The reconstruction begins from:

- First reconstruction collector sequence: `10`
- First accepted update range: `U = 97,233,590,167`, `u = 97,233,590,170`

Primary ordering authority:

`collector_sequence`

All V0.0 files remain read-only. V0.0 reconstructed-book artifacts are reference inputs only and may not replace the V0.1 reconstruction.

### Output contract

This notebook must save:

- Reconstructed visible market-by-price state table
- Visible top-ten bid-and-ask table
- Rejected-update ledger
- Update-continuity report
- Resynchronization ledger
- Book-quality summary
- Spread and depth-completeness summaries
- V0.0 reconstruction-reconciliation table
- Notebook 02 output manifest
- Controlled handoff for Notebook 03

Every authoritative output must record its source files, checksums, run identities, schema version, row counts, producing notebook, and acceptance status.

### Acceptance gate

Every retained authoritative state must:

- Belong to a continuous and valid depth-update sequence
- Have a valid collector sequence
- Have a valid local receipt timestamp
- Preserve the original update identifiers
- Contain correctly ordered bid and ask levels
- Contain only strictly positive resting quantities
- Have a defensible best bid and best ask
- Contain no silently accepted continuity failure
- Contain no silently repaired locked, crossed, incomplete, or malformed state

Any failed update or state must be rejected, quarantined, or explicitly recorded with its reason.

Notebook 03 is not authorized until this notebook finishes with no unresolved critical reconstruction failure.

### Scope limitation

This notebook reconstructs a visible market-by-price book.

It does not reconstruct:

- Individual orders or order IDs
- Queue position or time priority
- Hidden, iceberg, or undisplayed liquidity
- The complete exchange matching engine

It does not align trades, construct point-process events, estimate Hawkes models, generate quotes, simulate executions, or evaluate market-making performance.

In [1]:
from __future__ import annotations

import hashlib
import json
import platform
import sys
from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation
from pathlib import Path
from time import perf_counter
from typing import Any, Final

import numpy as np
import pandas as pd


# ---------------------------------------------------------------------------
# Notebook identity
# ---------------------------------------------------------------------------

NOTEBOOK_NAME: Final[str] = "02_VISIBLE_BOOK_RECONSTRUCTION"
NOTEBOOK_FILE: Final[str] = f"{NOTEBOOK_NAME}.ipynb"
NOTEBOOK_SCHEMA_VERSION: Final[str] = "v0.1.0"

SOURCE_RUN_PREFIX: Final[str] = (
    "BTCUSDT_spot_20260710T063746Z_c8b5bf12"
)
V0_1_RUN_ID: Final[str] = "v0_1_20260714T090616Z_e82325081a81"
OUTPUT_PREFIX: Final[str] = f"{SOURCE_RUN_PREFIX}__{V0_1_RUN_ID}"

OPERATING_MODE: Final[str] = "ENGINEERING_REPRODUCTION_MODE"
CANONICAL_TIMEZONE: Final[str] = "UTC"
TOP_N_LEVELS: Final[int] = 10


# ---------------------------------------------------------------------------
# Frozen run identities
# ---------------------------------------------------------------------------

SOURCE_SET_SHA256: Final[str] = (
    "132c83531eec615d279408b5c06f402973114ba3058dfadd2fe58e2e67184c4b"
)
RUN_CONFIG_SHA256: Final[str] = (
    "14aea0efb3c7b6a193b8c4575a440babe8265d2935c6a770bee995f688d59617"
)
RUN_IDENTITY_SHA256: Final[str] = (
    "5eb89cf073c036d70e3767df4b35ace7c31822e52e9b9df19204c42e37fe4198"
)
NOTEBOOK_01_HANDOFF_SHA256: Final[str] = (
    "80f3e2df95b6f2daebdf7b85324cbfcabdb64d22740e328962d1af078420b2e1"
)
NOTEBOOK_01_OUTPUT_MANIFEST_SHA256: Final[str] = (
    "2bb561126bbfc20fd25aa80131c1c11307e651d38ed3d6ee26a11b2ff0f41a4f"
)


# ---------------------------------------------------------------------------
# Project roots
# ---------------------------------------------------------------------------

V0_0_ROOT: Final[Path] = Path(r"D:\Clown Project\V0.0")
V0_1_ROOT: Final[Path] = Path(r"D:\Clown Project\V0.1")

V0_0_RAW_ROOT: Final[Path] = V0_0_ROOT / "data" / "raw"
V0_0_REFERENCE_ROOT: Final[Path] = V0_0_ROOT / "data" / "processed"

V0_1_CONFIG_ROOT: Final[Path] = V0_1_ROOT / "config"
V0_1_DATA_ROOT: Final[Path] = V0_1_ROOT / "data"
V0_1_ARTIFACT_ROOT: Final[Path] = V0_1_ROOT / "artifacts"
V0_1_LOG_ROOT: Final[Path] = V0_1_ROOT / "logs"
V0_1_SRC_ROOT: Final[Path] = V0_1_ROOT / "src"
V0_1_TEST_ROOT: Final[Path] = V0_1_ROOT / "tests"


# ---------------------------------------------------------------------------
# Reconstruction contract
# ---------------------------------------------------------------------------

SNAPSHOT_LAST_UPDATE_ID: Final[int] = 97_233_590_166
REQUIRED_SUCCESSOR_UPDATE_ID: Final[int] = 97_233_590_167
FIRST_APPLICABLE_DEPTH_POSITION: Final[int] = 9
FIRST_RECONSTRUCTION_COLLECTOR_SEQUENCE: Final[int] = 10
FIRST_APPLIED_UPDATE_ID: Final[int] = 97_233_590_167
FIRST_APPLIED_FINAL_UPDATE_ID: Final[int] = 97_233_590_170

NOTEBOOK_STARTED_AT_UTC: Final[str] = datetime.now(
    timezone.utc
).isoformat()


# ---------------------------------------------------------------------------
# Display defaults
# ---------------------------------------------------------------------------

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.10f}")


{
    "notebook": NOTEBOOK_NAME,
    "schema_version": NOTEBOOK_SCHEMA_VERSION,
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "v0_1_run_id": V0_1_RUN_ID,
    "operating_mode": OPERATING_MODE,
    "top_n_levels": TOP_N_LEVELS,
    "first_reconstruction_collector_sequence": (
        FIRST_RECONSTRUCTION_COLLECTOR_SEQUENCE
    ),
    "started_at_utc": NOTEBOOK_STARTED_AT_UTC,
    "python_version": sys.version.split()[0],
    "platform": platform.platform(),
}

{'notebook': '02_VISIBLE_BOOK_RECONSTRUCTION',
 'schema_version': 'v0.1.0',
 'source_run_prefix': 'BTCUSDT_spot_20260710T063746Z_c8b5bf12',
 'v0_1_run_id': 'v0_1_20260714T090616Z_e82325081a81',
 'operating_mode': 'ENGINEERING_REPRODUCTION_MODE',
 'top_n_levels': 10,
 'first_reconstruction_collector_sequence': 10,
 'started_at_utc': '2026-07-14T13:37:08.161717+00:00',
 'python_version': '3.11.9',
 'platform': 'Windows-10-10.0.26200-SP0'}

In [2]:
from IPython.display import display


# ---------------------------------------------------------------------------
# Frozen manifest and payload hashes from the authority map
# ---------------------------------------------------------------------------

EXPECTED_NOTEBOOK_00_MANIFEST_SHA256: Final[str] = (
    "e12966301d92e61656715f2a6d397786820de836d8156dd70092250619cbb00e"
)

EXPECTED_RAW_DATA_AUDIT_PAYLOAD_SHA256: Final[str] = (
    "fdfd1bc3ca907b1bf14286ca8f849d49a997bd5655f8793260838a34b1c9f56d"
)


# ---------------------------------------------------------------------------
# Exact authoritative manifest paths
# ---------------------------------------------------------------------------

MANIFEST_ROOT: Final[Path] = (
    V0_1_ARTIFACT_ROOT / "manifests"
)

NOTEBOOK_00_OUTPUT_MANIFEST_PATH: Final[Path] = (
    MANIFEST_ROOT
    / (
        f"{OUTPUT_PREFIX}"
        "__00_V01_RUN_CONTRACT"
        "__notebook_00_output_manifest.json"
    )
)

NOTEBOOK_01_OUTPUT_MANIFEST_PATH: Final[Path] = (
    MANIFEST_ROOT
    / (
        f"{OUTPUT_PREFIX}"
        "__01_RAW_DATA_AUDIT"
        "__notebook_01_output_manifest.json"
    )
)


# ---------------------------------------------------------------------------
# Integrity and artifact-loading helpers
# ---------------------------------------------------------------------------

def require_file(path: Path, label: str) -> Path:
    """Require an existing regular file and return its resolved path."""
    resolved = path.resolve(strict=False)

    if not resolved.exists():
        raise FileNotFoundError(
            f"{label} does not exist:\n{resolved}"
        )

    if not resolved.is_file():
        raise RuntimeError(
            f"{label} is not a regular file:\n{resolved}"
        )

    return resolved


def sha256_file(
    path: Path,
    chunk_size: int = 1 << 20,
) -> str:
    """Return the SHA-256 digest of the exact bytes stored on disk."""
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(chunk_size),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def canonical_json_sha256(value: Any) -> str:
    """Hash a JSON-compatible value using the pipeline's canonical form."""
    encoded = json.dumps(
        value,
        sort_keys=True,
        ensure_ascii=False,
        allow_nan=False,
        separators=(",", ":"),
    ).encode("utf-8")

    return hashlib.sha256(encoded).hexdigest()


def load_json_mapping(
    path: Path,
    label: str,
) -> dict[str, Any]:
    """Load a UTF-8 JSON document whose root must be a mapping."""
    try:
        with path.open("r", encoding="utf-8") as handle:
            value = json.load(handle)
    except json.JSONDecodeError as exc:
        raise RuntimeError(
            f"{label} is not valid JSON:\n{path}\n{exc}"
        ) from exc

    if not isinstance(value, dict):
        raise TypeError(
            f"{label} must contain a JSON object; "
            f"observed {type(value).__name__}."
        )

    return value


def path_is_within(
    candidate: Path,
    root: Path,
) -> bool:
    """Return True when candidate is equal to or below root."""
    candidate_resolved = candidate.resolve(strict=False)
    root_resolved = root.resolve(strict=False)

    try:
        candidate_resolved.relative_to(root_resolved)
        return True
    except ValueError:
        return False


def unwrap_saved_artifact(
    document: dict[str, Any],
    expected_artifact_type: str,
) -> tuple[dict[str, Any], dict[str, Any]]:
    """
    Validate and unwrap one pipeline JSON artifact.

    File hashes and canonical payload hashes are intentionally distinct.
    """
    metadata = document.get("artifact_metadata")
    payload = document.get("payload")

    if not isinstance(metadata, dict):
        raise RuntimeError(
            f"{expected_artifact_type} lacks artifact_metadata."
        )

    if not isinstance(payload, dict):
        raise RuntimeError(
            f"{expected_artifact_type} lacks a mapping payload."
        )

    observed_type = metadata.get("artifact_type")

    if observed_type != expected_artifact_type:
        raise RuntimeError(
            "Artifact-type mismatch: "
            f"expected {expected_artifact_type!r}, "
            f"observed {observed_type!r}."
        )

    if metadata.get("source_run_prefix") != SOURCE_RUN_PREFIX:
        raise RuntimeError(
            f"Source-run mismatch for {expected_artifact_type}."
        )

    if metadata.get("v0_1_run_id") != V0_1_RUN_ID:
        raise RuntimeError(
            f"V0.1 run-ID mismatch for {expected_artifact_type}."
        )

    expected_payload_hash = metadata.get("payload_sha256")

    if not isinstance(expected_payload_hash, str):
        raise RuntimeError(
            f"{expected_artifact_type} lacks payload_sha256."
        )

    observed_payload_hash = canonical_json_sha256(payload)

    if observed_payload_hash != expected_payload_hash.lower():
        raise RuntimeError(
            f"Canonical payload-hash mismatch for "
            f"{expected_artifact_type}."
        )

    return metadata, payload


def build_artifact_registry(
    manifest_payload: dict[str, Any],
    manifest_label: str,
) -> pd.DataFrame:
    """Construct and validate an artifact registry from a manifest."""
    records = manifest_payload.get("artifacts")

    if not isinstance(records, list):
        raise RuntimeError(
            f"{manifest_label} lacks an artifacts list."
        )

    registry = pd.DataFrame(records)

    required_columns = {
        "artifact_type",
        "format",
        "relative_path",
        "size_bytes",
        "sha256",
    }

    missing_columns = sorted(
        required_columns - set(registry.columns)
    )

    if missing_columns:
        raise RuntimeError(
            f"{manifest_label} artifact registry is missing: "
            + ", ".join(missing_columns)
        )

    duplicated_types = registry.loc[
        registry["artifact_type"].duplicated(keep=False),
        "artifact_type",
    ].dropna()

    if not duplicated_types.empty:
        raise RuntimeError(
            f"{manifest_label} contains duplicate artifact types: "
            + ", ".join(
                sorted(duplicated_types.astype(str).unique())
            )
        )

    return registry


def verified_artifact_path(
    registry: pd.DataFrame,
    artifact_type: str,
) -> Path:
    """Resolve and verify one artifact registered by an output manifest."""
    matches = registry.loc[
        registry["artifact_type"].eq(artifact_type)
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one {artifact_type} artifact; "
            f"found {len(matches)}."
        )

    record = matches.iloc[0]
    relative_path = record["relative_path"]

    if pd.isna(relative_path):
        raise RuntimeError(
            f"{artifact_type} lacks relative_path."
        )

    recorded_path = Path(str(relative_path))

    artifact_path = (
        recorded_path
        if recorded_path.is_absolute()
        else V0_1_ROOT / recorded_path
    ).resolve(strict=False)

    require_file(
        artifact_path,
        artifact_type,
    )

    if not path_is_within(artifact_path, V0_1_ROOT):
        raise RuntimeError(
            f"{artifact_type} is outside the V0.1 output tree:\n"
            f"{artifact_path}"
        )

    if path_is_within(artifact_path, V0_0_ROOT):
        raise RuntimeError(
            f"{artifact_type} improperly enters immutable V0.0:\n"
            f"{artifact_path}"
        )

    expected_size = int(record["size_bytes"])
    observed_size = artifact_path.stat().st_size

    if observed_size != expected_size:
        raise RuntimeError(
            f"Size mismatch for {artifact_type}: "
            f"expected {expected_size:,}, "
            f"observed {observed_size:,}."
        )

    expected_hash = str(record["sha256"]).lower()
    observed_hash = sha256_file(artifact_path)

    if observed_hash != expected_hash:
        raise RuntimeError(
            f"File SHA-256 mismatch for {artifact_type}:\n"
            f"expected: {expected_hash}\n"
            f"observed: {observed_hash}\n"
            f"path: {artifact_path}"
        )

    return artifact_path


def load_manifested_json(
    registry: pd.DataFrame,
    artifact_type: str,
) -> tuple[
    Path,
    dict[str, Any],
    dict[str, Any],
    dict[str, Any],
]:
    """Load and validate one manifested JSON artifact."""
    artifact_path = verified_artifact_path(
        registry,
        artifact_type,
    )

    document = load_json_mapping(
        artifact_path,
        artifact_type,
    )

    metadata, payload = unwrap_saved_artifact(
        document,
        artifact_type,
    )

    return artifact_path, document, metadata, payload


def load_manifested_csv(
    registry: pd.DataFrame,
    artifact_type: str,
) -> tuple[Path, pd.DataFrame]:
    """Load one manifested CSV artifact after file verification."""
    artifact_path = verified_artifact_path(
        registry,
        artifact_type,
    )

    return artifact_path, pd.read_csv(artifact_path)


# ---------------------------------------------------------------------------
# Verify and load the two manifest trust roots
# ---------------------------------------------------------------------------

NOTEBOOK_00_OUTPUT_MANIFEST_PATH = require_file(
    NOTEBOOK_00_OUTPUT_MANIFEST_PATH,
    "Notebook 00 output manifest",
)

NOTEBOOK_01_OUTPUT_MANIFEST_PATH = require_file(
    NOTEBOOK_01_OUTPUT_MANIFEST_PATH,
    "Notebook 01 output manifest",
)

observed_notebook_00_manifest_hash = sha256_file(
    NOTEBOOK_00_OUTPUT_MANIFEST_PATH
)

observed_notebook_01_manifest_hash = sha256_file(
    NOTEBOOK_01_OUTPUT_MANIFEST_PATH
)

if (
    observed_notebook_00_manifest_hash
    != EXPECTED_NOTEBOOK_00_MANIFEST_SHA256
):
    raise RuntimeError(
        "Notebook 00 output-manifest file hash mismatch:\n"
        f"expected: {EXPECTED_NOTEBOOK_00_MANIFEST_SHA256}\n"
        f"observed: {observed_notebook_00_manifest_hash}"
    )

if (
    observed_notebook_01_manifest_hash
    != NOTEBOOK_01_OUTPUT_MANIFEST_SHA256
):
    raise RuntimeError(
        "Notebook 01 output-manifest file hash mismatch:\n"
        f"expected: {NOTEBOOK_01_OUTPUT_MANIFEST_SHA256}\n"
        f"observed: {observed_notebook_01_manifest_hash}"
    )

notebook_00_manifest_document = load_json_mapping(
    NOTEBOOK_00_OUTPUT_MANIFEST_PATH,
    "Notebook 00 output manifest",
)

notebook_01_manifest_document = load_json_mapping(
    NOTEBOOK_01_OUTPUT_MANIFEST_PATH,
    "Notebook 01 output manifest",
)

(
    NOTEBOOK_00_MANIFEST_METADATA,
    NOTEBOOK_00_MANIFEST_PAYLOAD,
) = unwrap_saved_artifact(
    notebook_00_manifest_document,
    "NOTEBOOK_00_OUTPUT_MANIFEST",
)

(
    NOTEBOOK_01_MANIFEST_METADATA,
    NOTEBOOK_01_MANIFEST_PAYLOAD,
) = unwrap_saved_artifact(
    notebook_01_manifest_document,
    "NOTEBOOK_01_OUTPUT_MANIFEST",
)

if (
    NOTEBOOK_01_MANIFEST_PAYLOAD.get("next_notebook")
    != NOTEBOOK_FILE
):
    raise RuntimeError(
        "Notebook 01 manifest does not hand off to "
        f"{NOTEBOOK_FILE!r}."
    )

if not bool(
    NOTEBOOK_01_MANIFEST_PAYLOAD.get(
        "notebook_02_authorized",
        False,
    )
):
    raise RuntimeError(
        "Notebook 01 manifest does not authorize Notebook 02."
    )

NOTEBOOK_00_ARTIFACT_REGISTRY = build_artifact_registry(
    NOTEBOOK_00_MANIFEST_PAYLOAD,
    "Notebook 00 output manifest",
)

NOTEBOOK_01_ARTIFACT_REGISTRY = build_artifact_registry(
    NOTEBOOK_01_MANIFEST_PAYLOAD,
    "Notebook 01 output manifest",
)


# ---------------------------------------------------------------------------
# Load Notebook 00 contracts from its artifact registry
# ---------------------------------------------------------------------------

(
    RUN_CONFIG_PATH,
    RUN_CONFIG_DOCUMENT,
    RUN_CONFIG_METADATA,
    V0_1_RUN_CONFIG,
) = load_manifested_json(
    NOTEBOOK_00_ARTIFACT_REGISTRY,
    "V0_1_RUN_CONFIG",
)

(
    MARKET_DATA_CONTRACT_PATH,
    MARKET_DATA_CONTRACT_DOCUMENT,
    MARKET_DATA_CONTRACT_METADATA,
    MARKET_DATA_CONTRACT,
) = load_manifested_json(
    NOTEBOOK_00_ARTIFACT_REGISTRY,
    "MARKET_DATA_CONTRACT",
)

(
    REJECTION_CONTRACT_PATH,
    REJECTION_CONTRACT_DOCUMENT,
    REJECTION_CONTRACT_METADATA,
    MISSING_DATA_REJECTION_CONTRACT,
) = load_manifested_json(
    NOTEBOOK_00_ARTIFACT_REGISTRY,
    "MISSING_DATA_REJECTION_CONTRACT",
)

(
    SPLIT_CONTRACT_PATH,
    SPLIT_CONTRACT_DOCUMENT,
    SPLIT_CONTRACT_METADATA,
    CHRONOLOGICAL_SPLIT_CONTRACT,
) = load_manifested_json(
    NOTEBOOK_00_ARTIFACT_REGISTRY,
    "CHRONOLOGICAL_SPLIT_CONTRACT",
)

(
    DECISION_CONTRACT_PATH,
    DECISION_CONTRACT_DOCUMENT,
    DECISION_CONTRACT_METADATA,
    DECISION_AND_TOLERANCE_CONTRACT,
) = load_manifested_json(
    NOTEBOOK_00_ARTIFACT_REGISTRY,
    "DECISION_AND_TOLERANCE_CONTRACT",
)

(
    RUN_IDENTITY_PATH,
    RUN_IDENTITY_DOCUMENT,
    RUN_IDENTITY_METADATA,
    V0_1_RUN_IDENTITY,
) = load_manifested_json(
    NOTEBOOK_00_ARTIFACT_REGISTRY,
    "V0_1_RUN_IDENTITY",
)


# ---------------------------------------------------------------------------
# Load Notebook 01 outputs from its artifact registry
# ---------------------------------------------------------------------------

(
    RAW_DATA_AUDIT_PATH,
    RAW_DATA_AUDIT_DOCUMENT,
    RAW_DATA_AUDIT_METADATA,
    V0_1_RAW_DATA_AUDIT,
) = load_manifested_json(
    NOTEBOOK_01_ARTIFACT_REGISTRY,
    "V0_1_RAW_DATA_AUDIT",
)

(
    NOTEBOOK_01_HANDOFF_PATH,
    NOTEBOOK_01_HANDOFF_DOCUMENT,
    NOTEBOOK_01_HANDOFF_METADATA,
    NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF,
) = load_manifested_json(
    NOTEBOOK_01_ARTIFACT_REGISTRY,
    "NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF",
)

(
    RAW_SOURCE_MANIFEST_PATH,
    AUTHORITATIVE_RAW_SOURCE_MANIFEST,
) = load_manifested_csv(
    NOTEBOOK_01_ARTIFACT_REGISTRY,
    "AUTHORITATIVE_RAW_SOURCE_MANIFEST",
)

(
    SOURCE_REHASH_PATH,
    NOTEBOOK_01_SOURCE_REHASH,
) = load_manifested_csv(
    NOTEBOOK_01_ARTIFACT_REGISTRY,
    "NOTEBOOK_01_SOURCE_REHASH",
)

(
    DEPTH_UPDATE_ID_AUDIT_PATH,
    DEPTH_UPDATE_ID_AUDIT,
) = load_manifested_csv(
    NOTEBOOK_01_ARTIFACT_REGISTRY,
    "DEPTH_UPDATE_ID_AUDIT",
)


# ---------------------------------------------------------------------------
# Verify payload hashes and frozen identities
# ---------------------------------------------------------------------------

observed_raw_data_audit_payload_hash = canonical_json_sha256(
    V0_1_RAW_DATA_AUDIT
)

observed_handoff_payload_hash = canonical_json_sha256(
    NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF
)

if (
    observed_raw_data_audit_payload_hash
    != EXPECTED_RAW_DATA_AUDIT_PAYLOAD_SHA256
):
    raise RuntimeError(
        "Notebook 01 raw-data-audit payload hash mismatch:\n"
        f"expected: {EXPECTED_RAW_DATA_AUDIT_PAYLOAD_SHA256}\n"
        f"observed: {observed_raw_data_audit_payload_hash}"
    )

if (
    observed_handoff_payload_hash
    != NOTEBOOK_01_HANDOFF_SHA256
):
    raise RuntimeError(
        "Notebook 01 handoff payload hash mismatch:\n"
        f"expected: {NOTEBOOK_01_HANDOFF_SHA256}\n"
        f"observed: {observed_handoff_payload_hash}"
    )

handoff_identity_requirements = {
    "source_set_hash": SOURCE_SET_SHA256,
    "v0_1_run_config_hash": RUN_CONFIG_SHA256,
    "v0_1_run_identity_hash": RUN_IDENTITY_SHA256,
    "raw_data_audit_hash": (
        EXPECTED_RAW_DATA_AUDIT_PAYLOAD_SHA256
    ),
}

for field, expected_value in handoff_identity_requirements.items():
    observed_value = NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF.get(
        field
    )

    if observed_value != expected_value:
        raise RuntimeError(
            f"Notebook 01 handoff identity mismatch for {field}:\n"
            f"expected: {expected_value}\n"
            f"observed: {observed_value}"
        )

if not bool(
    NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF.get(
        "notebook_02_authorized",
        False,
    )
):
    raise RuntimeError(
        "Notebook 01 handoff does not authorize Notebook 02."
    )


# ---------------------------------------------------------------------------
# Resolve authoritative raw sources from Notebook 01's saved manifest
# ---------------------------------------------------------------------------

required_source_columns = {
    "source_role",
    "resolved_path",
    "size_bytes",
    "sha256",
}

missing_source_columns = sorted(
    required_source_columns
    - set(AUTHORITATIVE_RAW_SOURCE_MANIFEST.columns)
)

if missing_source_columns:
    raise RuntimeError(
        "Authoritative raw-source manifest is missing columns: "
        + ", ".join(missing_source_columns)
    )


def authoritative_source_path(source_role: str) -> Path:
    """Resolve one immutable raw source by its registered source role."""
    matches = AUTHORITATIVE_RAW_SOURCE_MANIFEST.loc[
        AUTHORITATIVE_RAW_SOURCE_MANIFEST[
            "source_role"
        ].eq(source_role)
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one {source_role} source; "
            f"found {len(matches)}."
        )

    source_path = Path(
        str(matches.iloc[0]["resolved_path"])
    ).resolve(strict=False)

    require_file(source_path, source_role)

    if not path_is_within(source_path, V0_0_ROOT):
        raise RuntimeError(
            f"{source_role} is outside immutable V0.0:\n"
            f"{source_path}"
        )

    if path_is_within(source_path, V0_1_ROOT):
        raise RuntimeError(
            f"{source_role} improperly enters V0.1:\n"
            f"{source_path}"
        )

    return source_path


TRADE_SOURCE_PATH = authoritative_source_path(
    "TRADE_STREAM"
)

DEPTH_SOURCE_PATH = authoritative_source_path(
    "DEPTH_STREAM"
)

REST_SNAPSHOT_PATH = authoritative_source_path(
    "REST_SNAPSHOT"
)

COLLECTOR_METADATA_PATH = authoritative_source_path(
    "COLLECTOR_METADATA"
)

RUN_MANIFEST_PATH = authoritative_source_path(
    "RUN_MANIFEST"
)


# ---------------------------------------------------------------------------
# Re-hash every immutable raw source before reconstruction
# ---------------------------------------------------------------------------

source_rehash_rows: list[dict[str, Any]] = []

for source_record in (
    AUTHORITATIVE_RAW_SOURCE_MANIFEST
    .itertuples(index=False)
):
    source_path = Path(
        str(source_record.resolved_path)
    ).resolve(strict=False)

    require_file(
        source_path,
        str(source_record.source_role),
    )

    expected_size = int(source_record.size_bytes)
    expected_hash = str(source_record.sha256).lower()

    observed_size = source_path.stat().st_size
    observed_hash = sha256_file(source_path)

    source_passed = (
        observed_size == expected_size
        and observed_hash == expected_hash
        and path_is_within(source_path, V0_0_ROOT)
        and not path_is_within(source_path, V0_1_ROOT)
    )

    source_rehash_rows.append(
        {
            "source_role": source_record.source_role,
            "resolved_path": str(source_path),
            "expected_size_bytes": expected_size,
            "observed_size_bytes": observed_size,
            "size_matches": observed_size == expected_size,
            "expected_sha256": expected_hash,
            "observed_sha256": observed_hash,
            "hash_matches": observed_hash == expected_hash,
            "source_passed": source_passed,
        }
    )

NOTEBOOK_02_SOURCE_REHASH = pd.DataFrame(
    source_rehash_rows
)

failed_sources = NOTEBOOK_02_SOURCE_REHASH.loc[
    ~NOTEBOOK_02_SOURCE_REHASH["source_passed"]
]

if not failed_sources.empty:
    raise RuntimeError(
        "One or more immutable raw sources failed verification:\n"
        + failed_sources[
            [
                "source_role",
                "size_matches",
                "hash_matches",
                "resolved_path",
            ]
        ].to_string(index=False)
    )


# ---------------------------------------------------------------------------
# Extract the exact reconstruction contract from the saved handoff
# ---------------------------------------------------------------------------

SNAPSHOT_CONTRACT = (
    NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF.get(
        "snapshot_contract"
    )
)

DEPTH_RECONSTRUCTION_CONTRACT = (
    NOTEBOOK_01_TO_NOTEBOOK_02_HANDOFF.get(
        "depth_reconstruction_contract"
    )
)

if not isinstance(SNAPSHOT_CONTRACT, dict):
    raise RuntimeError(
        "Notebook 01 handoff lacks snapshot_contract."
    )

if not isinstance(
    DEPTH_RECONSTRUCTION_CONTRACT,
    dict,
):
    raise RuntimeError(
        "Notebook 01 handoff lacks "
        "depth_reconstruction_contract."
    )

SNAPSHOT_LAST_UPDATE_ID = int(
    SNAPSHOT_CONTRACT["snapshot_last_update_id"]
)

SNAPSHOT_SESSION_ID = str(
    SNAPSHOT_CONTRACT["snapshot_session_id"]
)

FIRST_APPLICABLE_DEPTH_POSITION = int(
    DEPTH_RECONSTRUCTION_CONTRACT[
        "first_applicable_depth_position"
    ]
)

FIRST_APPLICABLE_DEPTH_COLLECTOR_SEQUENCE = int(
    DEPTH_RECONSTRUCTION_CONTRACT[
        "first_applicable_collector_sequence"
    ]
)

FIRST_APPLICABLE_DEPTH_U = int(
    DEPTH_RECONSTRUCTION_CONTRACT[
        "first_applicable_U"
    ]
)

FIRST_APPLICABLE_DEPTH_u = int(
    DEPTH_RECONSTRUCTION_CONTRACT[
        "first_applicable_u"
    ]
)

STALE_PRE_SNAPSHOT_DEPTH_EVENT_COUNT = int(
    DEPTH_RECONSTRUCTION_CONTRACT[
        "stale_pre_snapshot_event_count"
    ]
)

TOTAL_DEPTH_EVENT_COUNT = int(
    DEPTH_RECONSTRUCTION_CONTRACT[
        "total_depth_event_count"
    ]
)

POST_SNAPSHOT_DEPTH_EVENT_COUNT = int(
    DEPTH_RECONSTRUCTION_CONTRACT[
        "post_snapshot_event_count"
    ]
)

POST_SNAPSHOT_GAP_COUNT = int(
    DEPTH_RECONSTRUCTION_CONTRACT[
        "post_snapshot_gap_count"
    ]
)

POST_SNAPSHOT_OVERLAP_COUNT = int(
    DEPTH_RECONSTRUCTION_CONTRACT[
        "post_snapshot_overlap_count"
    ]
)

ZERO_QUANTITY_ACTION = str(
    DEPTH_RECONSTRUCTION_CONTRACT[
        "zero_quantity_action"
    ]
)

PRIMARY_ORDERING_AUTHORITY = str(
    DEPTH_RECONSTRUCTION_CONTRACT[
        "ordering_authority"
    ]
)

HANDOFF_TIMEZONE = str(
    DEPTH_RECONSTRUCTION_CONTRACT["timezone"]
)

if SNAPSHOT_LAST_UPDATE_ID != 97_233_590_166:
    raise RuntimeError(
        "Unexpected snapshot lastUpdateId: "
        f"{SNAPSHOT_LAST_UPDATE_ID:,}"
    )

if (
    FIRST_APPLICABLE_DEPTH_COLLECTOR_SEQUENCE
    != FIRST_RECONSTRUCTION_COLLECTOR_SEQUENCE
):
    raise RuntimeError(
        "Unexpected first reconstruction collector sequence: "
        f"{FIRST_APPLICABLE_DEPTH_COLLECTOR_SEQUENCE:,}"
    )

if not (
    FIRST_APPLICABLE_DEPTH_U
    <= SNAPSHOT_LAST_UPDATE_ID + 1
    <= FIRST_APPLICABLE_DEPTH_u
):
    raise RuntimeError(
        "The first applicable depth event does not bridge "
        "the REST snapshot."
    )

if POST_SNAPSHOT_GAP_COUNT != 0:
    raise RuntimeError(
        f"Post-snapshot update-ID gaps: "
        f"{POST_SNAPSHOT_GAP_COUNT:,}"
    )

if POST_SNAPSHOT_OVERLAP_COUNT != 0:
    raise RuntimeError(
        f"Post-snapshot update-ID overlaps: "
        f"{POST_SNAPSHOT_OVERLAP_COUNT:,}"
    )

if ZERO_QUANTITY_ACTION != "DELETE_PRICE_LEVEL":
    raise RuntimeError(
        "Unexpected zero-quantity action: "
        f"{ZERO_QUANTITY_ACTION!r}"
    )

if PRIMARY_ORDERING_AUTHORITY != "collector_sequence":
    raise RuntimeError(
        "Unexpected ordering authority: "
        f"{PRIMARY_ORDERING_AUTHORITY!r}"
    )

if HANDOFF_TIMEZONE != CANONICAL_TIMEZONE:
    raise RuntimeError(
        "Timezone-contract mismatch: "
        f"{HANDOFF_TIMEZONE!r}"
    )


# ---------------------------------------------------------------------------
# Bootstrap result
# ---------------------------------------------------------------------------

UPSTREAM_ARTIFACTS_VERIFIED: Final[bool] = True

display(
    NOTEBOOK_02_SOURCE_REHASH[
        [
            "source_role",
            "size_matches",
            "hash_matches",
            "source_passed",
            "resolved_path",
        ]
    ]
)

{
    "status": "PASS",
    "upstream_artifacts_verified": (
        UPSTREAM_ARTIFACTS_VERIFIED
    ),
    "notebook_00_manifest_hash_verified": True,
    "notebook_01_manifest_hash_verified": True,
    "notebook_01_raw_audit_payload_verified": True,
    "notebook_01_handoff_payload_verified": True,
    "notebook_02_authorized": True,
    "immutable_raw_sources_verified": int(
        NOTEBOOK_02_SOURCE_REHASH[
            "source_passed"
        ].sum()
    ),
    "snapshot_last_update_id": (
        SNAPSHOT_LAST_UPDATE_ID
    ),
    "stale_pre_snapshot_events": (
        STALE_PRE_SNAPSHOT_DEPTH_EVENT_COUNT
    ),
    "first_applicable_depth_position": (
        FIRST_APPLICABLE_DEPTH_POSITION
    ),
    "first_applicable_collector_sequence": (
        FIRST_APPLICABLE_DEPTH_COLLECTOR_SEQUENCE
    ),
    "first_applicable_update_range": {
        "U": FIRST_APPLICABLE_DEPTH_U,
        "u": FIRST_APPLICABLE_DEPTH_u,
    },
    "post_snapshot_depth_events": (
        POST_SNAPSHOT_DEPTH_EVENT_COUNT
    ),
    "post_snapshot_gaps": POST_SNAPSHOT_GAP_COUNT,
    "post_snapshot_overlaps": (
        POST_SNAPSHOT_OVERLAP_COUNT
    ),
    "ordering_authority": (
        PRIMARY_ORDERING_AUTHORITY
    ),
    "zero_quantity_action": (
        ZERO_QUANTITY_ACTION
    ),
}

,source_role,size_matches,hash_matches,source_passed,resolved_path
0,COLLECTOR_METADATA,True,True,True,D:\Clown Project\V0.0\data\raw\metadata\BTCUSD...
1,DEPTH_STREAM,True,True,True,D:\Clown Project\V0.0\data\raw\order_book\BTCU...
2,REST_SNAPSHOT,True,True,True,D:\Clown Project\V0.0\data\raw\order_book\BTCU...
3,RUN_MANIFEST,True,True,True,D:\Clown Project\V0.0\data\raw\metadata\BTCUSD...
4,TRADE_STREAM,True,True,True,D:\Clown Project\V0.0\data\raw\trades\BTCUSDT_...


{'status': 'PASS',
 'upstream_artifacts_verified': True,
 'notebook_00_manifest_hash_verified': True,
 'notebook_01_manifest_hash_verified': True,
 'notebook_01_raw_audit_payload_verified': True,
 'notebook_01_handoff_payload_verified': True,
 'notebook_02_authorized': True,
 'immutable_raw_sources_verified': 5,
 'snapshot_last_update_id': 97233590166,
 'stale_pre_snapshot_events': 9,
 'first_applicable_depth_position': 9,
 'first_applicable_collector_sequence': 10,
 'first_applicable_update_range': {'U': 97233590167, 'u': 97233590170},
 'post_snapshot_depth_events': 35985,
 'post_snapshot_gaps': 0,
 'post_snapshot_overlaps': 0,
 'ordering_authority': 'collector_sequence',
 'zero_quantity_action': 'DELETE_PRICE_LEVEL'}

In [3]:
# ---------------------------------------------------------------------------
# Load and normalize the REST snapshot and differential-depth stream
# ---------------------------------------------------------------------------

EXPECTED_DEPTH_EVENT_COUNT: Final[int] = 35_994
EXPECTED_BID_UPDATE_COUNT: Final[int] = 332_920
EXPECTED_ASK_UPDATE_COUNT: Final[int] = 277_873
EXPECTED_LEVEL_UPDATE_COUNT: Final[int] = 610_793
EXPECTED_ZERO_QUANTITY_COUNT: Final[int] = 219_617
EXPECTED_POSITIVE_QUANTITY_COUNT: Final[int] = 391_176

EXPECTED_FIRST_OBSERVED_U: Final[int] = 97_233_590_081
EXPECTED_FIRST_OBSERVED_u: Final[int] = 97_233_590_085
EXPECTED_LAST_OBSERVED_U: Final[int] = 97_234_812_158
EXPECTED_LAST_OBSERVED_u: Final[int] = 97_234_812_218

EXPECTED_SNAPSHOT_LEVELS_PER_SIDE: Final[int] = 5_000
TIMESTAMP_TEXT_TOLERANCE_NS: Final[int] = 999


# ---------------------------------------------------------------------------
# Strict scalar parsers
# ---------------------------------------------------------------------------

def require_integer_scalar(
    value: Any,
    *,
    field_name: str,
    context: str,
) -> int:
    """Require an integer scalar without accepting booleans."""
    if isinstance(value, bool) or not isinstance(
        value,
        (int, np.integer),
    ):
        raise TypeError(
            f"{context}: field {field_name!r} must be an integer; "
            f"observed {type(value).__name__}."
        )

    return int(value)


def require_string_scalar(
    value: Any,
    *,
    field_name: str,
    context: str,
) -> str:
    """Require a non-empty string scalar."""
    if not isinstance(value, str) or not value.strip():
        raise TypeError(
            f"{context}: field {field_name!r} must be a "
            "non-empty string."
        )

    return value.strip()


def parse_epoch_timestamp(
    value: Any,
    *,
    unit: str,
    field_name: str,
    context: str,
) -> tuple[int, pd.Timestamp]:
    """
    Parse an integer Unix timestamp using an explicitly frozen unit.

    exchange_event_time_raw -> milliseconds
    local_receipt_time_ns   -> nanoseconds
    """
    integer_value = require_integer_scalar(
        value,
        field_name=field_name,
        context=context,
    )

    try:
        timestamp = pd.to_datetime(
            integer_value,
            unit=unit,
            utc=True,
        )
    except Exception as exc:
        raise ValueError(
            f"{context}: field {field_name!r} cannot be parsed "
            f"as Unix epoch {unit!r}: {integer_value!r}"
        ) from exc

    if pd.isna(timestamp):
        raise ValueError(
            f"{context}: field {field_name!r} parsed to NaT."
        )

    return integer_value, pd.Timestamp(timestamp)


def parse_utc_text_timestamp(
    value: Any,
    *,
    field_name: str,
    context: str,
) -> pd.Timestamp:
    """Parse one timezone-aware UTC timestamp string."""
    text_value = require_string_scalar(
        value,
        field_name=field_name,
        context=context,
    )

    try:
        timestamp = pd.Timestamp(text_value)
    except Exception as exc:
        raise ValueError(
            f"{context}: field {field_name!r} is not a valid "
            f"timestamp: {text_value!r}"
        ) from exc

    if timestamp.tzinfo is None:
        raise ValueError(
            f"{context}: field {field_name!r} must be "
            "timezone-aware."
        )

    return timestamp.tz_convert("UTC")


def parse_decimal_scalar(
    value: Any,
    *,
    field_name: str,
    context: str,
    allow_zero: bool,
) -> Decimal:
    """Parse one finite decimal price or quantity."""
    if isinstance(value, bool):
        raise TypeError(
            f"{context}: field {field_name!r} cannot be boolean."
        )

    try:
        decimal_value = Decimal(str(value))
    except (InvalidOperation, ValueError, TypeError) as exc:
        raise ValueError(
            f"{context}: field {field_name!r} is not a valid "
            f"decimal: {value!r}"
        ) from exc

    if not decimal_value.is_finite():
        raise ValueError(
            f"{context}: field {field_name!r} must be finite."
        )

    minimum_valid = (
        decimal_value >= Decimal("0")
        if allow_zero
        else decimal_value > Decimal("0")
    )

    if not minimum_valid:
        comparison = "non-negative" if allow_zero else "positive"

        raise ValueError(
            f"{context}: field {field_name!r} must be "
            f"{comparison}; observed {decimal_value}."
        )

    return decimal_value


# ---------------------------------------------------------------------------
# JSON and raw-message parsers
# ---------------------------------------------------------------------------

def require_mapping_value(
    value: Any,
    *,
    field_name: str,
    context: str,
) -> dict[str, Any]:
    """Require a dictionary-like JSON object."""
    if not isinstance(value, dict):
        raise TypeError(
            f"{context}: field {field_name!r} must be a JSON "
            f"object; observed {type(value).__name__}."
        )

    return value


def decode_embedded_json_object(
    value: Any,
    *,
    field_name: str,
    context: str,
) -> dict[str, Any]:
    """Decode an embedded JSON object stored as text or mapping."""
    if isinstance(value, dict):
        return value

    if not isinstance(value, str):
        raise TypeError(
            f"{context}: field {field_name!r} must be JSON text "
            f"or an object; observed {type(value).__name__}."
        )

    try:
        decoded = json.loads(value)
    except json.JSONDecodeError as exc:
        raise ValueError(
            f"{context}: field {field_name!r} contains invalid JSON."
        ) from exc

    return require_mapping_value(
        decoded,
        field_name=field_name,
        context=context,
    )


def parse_depth_levels(
    value: Any,
    *,
    side: str,
    context: str,
) -> tuple[tuple[Decimal, Decimal], ...]:
    """
    Parse one Binance differential-depth side.

    Quantity zero is retained here because it is a deletion instruction.
    """
    if not isinstance(value, list):
        raise TypeError(
            f"{context}: {side} updates must be a list; "
            f"observed {type(value).__name__}."
        )

    parsed_levels: list[tuple[Decimal, Decimal]] = []
    observed_prices: set[Decimal] = set()

    for level_index, level in enumerate(value):
        level_context = (
            f"{context}, {side} level {level_index}"
        )

        if not isinstance(level, (list, tuple)) or len(level) != 2:
            raise ValueError(
                f"{level_context}: expected [price, quantity]; "
                f"observed {level!r}."
            )

        price = parse_decimal_scalar(
            level[0],
            field_name="price",
            context=level_context,
            allow_zero=False,
        )

        quantity = parse_decimal_scalar(
            level[1],
            field_name="quantity",
            context=level_context,
            allow_zero=True,
        )

        if price in observed_prices:
            raise ValueError(
                f"{level_context}: duplicate price {price} "
                f"inside one {side} event."
            )

        observed_prices.add(price)
        parsed_levels.append((price, quantity))

    return tuple(parsed_levels)


def assert_exact_match(
    wrapper_value: Any,
    raw_value: Any,
    *,
    field_name: str,
    context: str,
) -> None:
    """Require an exact wrapper-versus-raw-message match."""
    if wrapper_value != raw_value:
        raise ValueError(
            f"{context}: wrapper/raw mismatch for "
            f"{field_name!r}: wrapper={wrapper_value!r}, "
            f"raw={raw_value!r}."
        )


# ---------------------------------------------------------------------------
# Load the immutable REST snapshot
# ---------------------------------------------------------------------------

snapshot_wrapper = load_json_mapping(
    REST_SNAPSHOT_PATH,
    "Immutable REST snapshot",
)

snapshot_context = "REST snapshot"

snapshot_raw_response = decode_embedded_json_object(
    snapshot_wrapper.get(
        "raw_response",
        snapshot_wrapper,
    ),
    field_name="raw_response",
    context=snapshot_context,
)

snapshot_last_update_id = require_integer_scalar(
    snapshot_raw_response.get("lastUpdateId"),
    field_name="lastUpdateId",
    context=snapshot_context,
)

wrapper_snapshot_update_id = snapshot_wrapper.get(
    "lastUpdateId",
    snapshot_last_update_id,
)

assert_exact_match(
    require_integer_scalar(
        wrapper_snapshot_update_id,
        field_name="lastUpdateId",
        context=snapshot_context,
    ),
    snapshot_last_update_id,
    field_name="lastUpdateId",
    context=snapshot_context,
)

if snapshot_last_update_id != SNAPSHOT_LAST_UPDATE_ID:
    raise RuntimeError(
        "REST snapshot lastUpdateId does not match the frozen "
        "handoff contract:\n"
        f"expected: {SNAPSHOT_LAST_UPDATE_ID:,}\n"
        f"observed: {snapshot_last_update_id:,}"
    )

snapshot_bid_levels = parse_depth_levels(
    snapshot_raw_response.get("bids"),
    side="bid",
    context=snapshot_context,
)

snapshot_ask_levels = parse_depth_levels(
    snapshot_raw_response.get("asks"),
    side="ask",
    context=snapshot_context,
)

if len(snapshot_bid_levels) != EXPECTED_SNAPSHOT_LEVELS_PER_SIDE:
    raise RuntimeError(
        "Unexpected REST snapshot bid-level count: "
        f"{len(snapshot_bid_levels):,}"
    )

if len(snapshot_ask_levels) != EXPECTED_SNAPSHOT_LEVELS_PER_SIDE:
    raise RuntimeError(
        "Unexpected REST snapshot ask-level count: "
        f"{len(snapshot_ask_levels):,}"
    )

if any(
    quantity <= 0
    for _, quantity in (
        snapshot_bid_levels + snapshot_ask_levels
    )
):
    raise RuntimeError(
        "REST snapshot contains a non-positive resting quantity."
    )

SNAPSHOT_BID_BOOK: dict[Decimal, Decimal] = dict(
    snapshot_bid_levels
)

SNAPSHOT_ASK_BOOK: dict[Decimal, Decimal] = dict(
    snapshot_ask_levels
)

if len(SNAPSHOT_BID_BOOK) != len(snapshot_bid_levels):
    raise RuntimeError(
        "REST snapshot contains duplicate bid prices."
    )

if len(SNAPSHOT_ASK_BOOK) != len(snapshot_ask_levels):
    raise RuntimeError(
        "REST snapshot contains duplicate ask prices."
    )

snapshot_best_bid = max(SNAPSHOT_BID_BOOK)
snapshot_best_ask = min(SNAPSHOT_ASK_BOOK)

if snapshot_best_bid >= snapshot_best_ask:
    raise RuntimeError(
        "REST snapshot is locked or crossed:\n"
        f"best_bid={snapshot_best_bid}\n"
        f"best_ask={snapshot_best_ask}"
    )


# ---------------------------------------------------------------------------
# Parse the differential-depth JSONL stream
# ---------------------------------------------------------------------------

depth_event_rows: list[dict[str, Any]] = []
depth_rejection_rows: list[dict[str, Any]] = []

with DEPTH_SOURCE_PATH.open(
    "r",
    encoding="utf-8",
) as depth_handle:
    for source_line_number, raw_line in enumerate(
        depth_handle,
        start=1,
    ):
        stripped_line = raw_line.strip()
        source_position = source_line_number - 1
        context = f"depth stream line {source_line_number}"

        if not stripped_line:
            depth_rejection_rows.append(
                {
                    "source_line_number": source_line_number,
                    "source_position": source_position,
                    "rejection_code": "BLANK_RECORD",
                    "rejection_detail": (
                        f"{context}: blank JSONL record."
                    ),
                    "raw_preview": "",
                }
            )
            continue

        try:
            wrapper = json.loads(stripped_line)

            if not isinstance(wrapper, dict):
                raise TypeError(
                    f"{context}: JSON root must be an object."
                )

            collector_sequence = require_integer_scalar(
                wrapper.get("collector_sequence"),
                field_name="collector_sequence",
                context=context,
            )

            connection_session_id = require_string_scalar(
                wrapper.get("connection_session_id"),
                field_name="connection_session_id",
                context=context,
            )

            stream = require_string_scalar(
                wrapper.get("stream"),
                field_name="stream",
                context=context,
            )

            event_type = require_string_scalar(
                wrapper.get("event_type"),
                field_name="event_type",
                context=context,
            )

            symbol = require_string_scalar(
                wrapper.get("symbol"),
                field_name="symbol",
                context=context,
            )

            exchange_event_time_ms, exchange_event_time_utc = (
                parse_epoch_timestamp(
                    wrapper.get("exchange_event_time_raw"),
                    unit="ms",
                    field_name="exchange_event_time_raw",
                    context=context,
                )
            )

            local_receipt_time_ns, local_receipt_time_utc = (
                parse_epoch_timestamp(
                    wrapper.get("local_receipt_time_ns"),
                    unit="ns",
                    field_name="local_receipt_time_ns",
                    context=context,
                )
            )

            local_receipt_text_utc = parse_utc_text_timestamp(
                wrapper.get("local_receipt_time_utc"),
                field_name="local_receipt_time_utc",
                context=context,
            )

            local_text_difference_ns = abs(
                local_receipt_text_utc.value
                - local_receipt_time_utc.value
            )

            if (
                local_text_difference_ns
                > TIMESTAMP_TEXT_TOLERANCE_NS
            ):
                raise ValueError(
                    f"{context}: local_receipt_time_ns and "
                    "local_receipt_time_utc differ by "
                    f"{local_text_difference_ns:,} ns."
                )

            first_update_id = require_integer_scalar(
                wrapper.get("first_update_id"),
                field_name="first_update_id",
                context=context,
            )

            final_update_id = require_integer_scalar(
                wrapper.get("final_update_id"),
                field_name="final_update_id",
                context=context,
            )

            if first_update_id > final_update_id:
                raise ValueError(
                    f"{context}: first_update_id exceeds "
                    "final_update_id."
                )

            raw_message = decode_embedded_json_object(
                wrapper.get("raw_message"),
                field_name="raw_message",
                context=context,
            )

            raw_stream = require_string_scalar(
                raw_message.get("stream"),
                field_name="raw_message.stream",
                context=context,
            )

            raw_data = require_mapping_value(
                raw_message.get("data"),
                field_name="raw_message.data",
                context=context,
            )

            raw_event_type = require_string_scalar(
                raw_data.get("e"),
                field_name="raw_message.data.e",
                context=context,
            )

            raw_exchange_event_time_ms = (
                require_integer_scalar(
                    raw_data.get("E"),
                    field_name="raw_message.data.E",
                    context=context,
                )
            )

            raw_symbol = require_string_scalar(
                raw_data.get("s"),
                field_name="raw_message.data.s",
                context=context,
            )

            raw_first_update_id = require_integer_scalar(
                raw_data.get("U"),
                field_name="raw_message.data.U",
                context=context,
            )

            raw_final_update_id = require_integer_scalar(
                raw_data.get("u"),
                field_name="raw_message.data.u",
                context=context,
            )

            wrapper_bid_updates = wrapper.get(
                "bid_changes_raw"
            )

            wrapper_ask_updates = wrapper.get(
                "ask_changes_raw"
            )

            raw_bid_updates = raw_data.get("b")
            raw_ask_updates = raw_data.get("a")

            assert_exact_match(
                stream,
                raw_stream,
                field_name="stream",
                context=context,
            )

            assert_exact_match(
                event_type,
                raw_event_type,
                field_name="event_type",
                context=context,
            )

            assert_exact_match(
                exchange_event_time_ms,
                raw_exchange_event_time_ms,
                field_name="exchange_event_time_raw",
                context=context,
            )

            assert_exact_match(
                symbol,
                raw_symbol,
                field_name="symbol",
                context=context,
            )

            assert_exact_match(
                first_update_id,
                raw_first_update_id,
                field_name="first_update_id",
                context=context,
            )

            assert_exact_match(
                final_update_id,
                raw_final_update_id,
                field_name="final_update_id",
                context=context,
            )

            assert_exact_match(
                wrapper_bid_updates,
                raw_bid_updates,
                field_name="bid_changes_raw",
                context=context,
            )

            assert_exact_match(
                wrapper_ask_updates,
                raw_ask_updates,
                field_name="ask_changes_raw",
                context=context,
            )

            bid_updates = parse_depth_levels(
                raw_bid_updates,
                side="bid",
                context=context,
            )

            ask_updates = parse_depth_levels(
                raw_ask_updates,
                side="ask",
                context=context,
            )

            if not bid_updates and not ask_updates:
                raise ValueError(
                    f"{context}: both update sides are empty."
                )

            bid_delete_count = sum(
                quantity == 0
                for _, quantity in bid_updates
            )

            ask_delete_count = sum(
                quantity == 0
                for _, quantity in ask_updates
            )

            depth_event_rows.append(
                {
                    "source_line_number": source_line_number,
                    "source_position": source_position,
                    "connection_session_id": (
                        connection_session_id
                    ),
                    "collector_sequence": collector_sequence,
                    "local_receipt_time_ns": (
                        local_receipt_time_ns
                    ),
                    "local_receipt_time_utc": (
                        local_receipt_time_utc
                    ),
                    "exchange_event_time_ms": (
                        exchange_event_time_ms
                    ),
                    "exchange_event_time_utc": (
                        exchange_event_time_utc
                    ),
                    "receipt_minus_exchange_ns": (
                        local_receipt_time_utc.value
                        - exchange_event_time_utc.value
                    ),
                    "stream": stream,
                    "event_type": event_type,
                    "symbol": symbol,
                    "first_update_id": first_update_id,
                    "final_update_id": final_update_id,
                    "bid_updates": bid_updates,
                    "ask_updates": ask_updates,
                    "bid_update_count": len(bid_updates),
                    "ask_update_count": len(ask_updates),
                    "level_update_count": (
                        len(bid_updates) + len(ask_updates)
                    ),
                    "bid_delete_count": bid_delete_count,
                    "ask_delete_count": ask_delete_count,
                    "delete_update_count": (
                        bid_delete_count + ask_delete_count
                    ),
                    "positive_update_count": (
                        len(bid_updates)
                        + len(ask_updates)
                        - bid_delete_count
                        - ask_delete_count
                    ),
                }
            )

        except Exception as exc:
            depth_rejection_rows.append(
                {
                    "source_line_number": source_line_number,
                    "source_position": source_position,
                    "rejection_code": (
                        type(exc).__name__.upper()
                    ),
                    "rejection_detail": str(exc),
                    "raw_preview": stripped_line[:500],
                }
            )


DEPTH_PARSE_REJECTIONS = pd.DataFrame(
    depth_rejection_rows,
    columns=[
        "source_line_number",
        "source_position",
        "rejection_code",
        "rejection_detail",
        "raw_preview",
    ],
)

if not DEPTH_PARSE_REJECTIONS.empty:
    display(DEPTH_PARSE_REJECTIONS.head(25))

    raise RuntimeError(
        "Differential-depth ingestion produced rejected "
        f"records: {len(DEPTH_PARSE_REJECTIONS):,}"
    )


DEPTH_EVENTS = pd.DataFrame(depth_event_rows)

if len(DEPTH_EVENTS) != EXPECTED_DEPTH_EVENT_COUNT:
    raise RuntimeError(
        "Differential-depth row-count mismatch:\n"
        f"expected: {EXPECTED_DEPTH_EVENT_COUNT:,}\n"
        f"observed: {len(DEPTH_EVENTS):,}"
    )


# ---------------------------------------------------------------------------
# Stream-level schema and ordering invariants
# ---------------------------------------------------------------------------

if not DEPTH_EVENTS["source_position"].equals(
    pd.Series(
        np.arange(len(DEPTH_EVENTS)),
        name="source_position",
    )
):
    raise RuntimeError(
        "Depth source positions are not complete and zero-based."
    )

if not DEPTH_EVENTS[
    "collector_sequence"
].is_monotonic_increasing:
    raise RuntimeError(
        "Depth records are not ordered by collector_sequence."
    )

if DEPTH_EVENTS["collector_sequence"].duplicated().any():
    raise RuntimeError(
        "Duplicate collector_sequence values exist in the "
        "depth stream."
    )

if not DEPTH_EVENTS[
    "local_receipt_time_ns"
].is_monotonic_increasing:
    raise RuntimeError(
        "Depth local receipt timestamps reverse in file order."
    )

expected_constant_fields = {
    "connection_session_id": SNAPSHOT_SESSION_ID,
    "event_type": "depthUpdate",
    "symbol": "BTCUSDT",
    "stream": "btcusdt@depth@100ms",
}

for field_name, expected_value in (
    expected_constant_fields.items()
):
    observed_values = DEPTH_EVENTS[
        field_name
    ].drop_duplicates()

    if (
        len(observed_values) != 1
        or observed_values.iloc[0] != expected_value
    ):
        raise RuntimeError(
            f"Unexpected values in depth field {field_name!r}:\n"
            f"{observed_values.tolist()}"
        )


# ---------------------------------------------------------------------------
# Level-count reconciliation
# ---------------------------------------------------------------------------

observed_bid_update_count = int(
    DEPTH_EVENTS["bid_update_count"].sum()
)

observed_ask_update_count = int(
    DEPTH_EVENTS["ask_update_count"].sum()
)

observed_level_update_count = int(
    DEPTH_EVENTS["level_update_count"].sum()
)

observed_delete_update_count = int(
    DEPTH_EVENTS["delete_update_count"].sum()
)

observed_positive_update_count = int(
    DEPTH_EVENTS["positive_update_count"].sum()
)

level_count_expectations = {
    "bid updates": (
        observed_bid_update_count,
        EXPECTED_BID_UPDATE_COUNT,
    ),
    "ask updates": (
        observed_ask_update_count,
        EXPECTED_ASK_UPDATE_COUNT,
    ),
    "total level updates": (
        observed_level_update_count,
        EXPECTED_LEVEL_UPDATE_COUNT,
    ),
    "zero-quantity updates": (
        observed_delete_update_count,
        EXPECTED_ZERO_QUANTITY_COUNT,
    ),
    "positive-quantity updates": (
        observed_positive_update_count,
        EXPECTED_POSITIVE_QUANTITY_COUNT,
    ),
}

for count_label, (
    observed_count,
    expected_count,
) in level_count_expectations.items():
    if observed_count != expected_count:
        raise RuntimeError(
            f"Depth {count_label} mismatch: "
            f"expected {expected_count:,}, "
            f"observed {observed_count:,}."
        )


# ---------------------------------------------------------------------------
# Observed update-range reconciliation
# ---------------------------------------------------------------------------

first_depth_event = DEPTH_EVENTS.iloc[0]
last_depth_event = DEPTH_EVENTS.iloc[-1]

if (
    int(first_depth_event["first_update_id"])
    != EXPECTED_FIRST_OBSERVED_U
    or int(first_depth_event["final_update_id"])
    != EXPECTED_FIRST_OBSERVED_u
):
    raise RuntimeError(
        "Unexpected first observed depth-update range."
    )

if (
    int(last_depth_event["first_update_id"])
    != EXPECTED_LAST_OBSERVED_U
    or int(last_depth_event["final_update_id"])
    != EXPECTED_LAST_OBSERVED_u
):
    raise RuntimeError(
        "Unexpected last observed depth-update range."
    )


# ---------------------------------------------------------------------------
# Snapshot bridge and post-snapshot continuity
# ---------------------------------------------------------------------------

STALE_DEPTH_EVENTS = DEPTH_EVENTS.loc[
    DEPTH_EVENTS["final_update_id"]
    <= SNAPSHOT_LAST_UPDATE_ID
].copy()

POST_SNAPSHOT_DEPTH_EVENTS = DEPTH_EVENTS.loc[
    DEPTH_EVENTS["final_update_id"]
    > SNAPSHOT_LAST_UPDATE_ID
].copy()

if len(STALE_DEPTH_EVENTS) != (
    STALE_PRE_SNAPSHOT_DEPTH_EVENT_COUNT
):
    raise RuntimeError(
        "Stale pre-snapshot event-count mismatch:\n"
        f"expected: "
        f"{STALE_PRE_SNAPSHOT_DEPTH_EVENT_COUNT:,}\n"
        f"observed: {len(STALE_DEPTH_EVENTS):,}"
    )

if len(POST_SNAPSHOT_DEPTH_EVENTS) != (
    POST_SNAPSHOT_DEPTH_EVENT_COUNT
):
    raise RuntimeError(
        "Post-snapshot event-count mismatch:\n"
        f"expected: {POST_SNAPSHOT_DEPTH_EVENT_COUNT:,}\n"
        f"observed: {len(POST_SNAPSHOT_DEPTH_EVENTS):,}"
    )

first_applicable_event = (
    POST_SNAPSHOT_DEPTH_EVENTS.iloc[0]
)

if (
    int(first_applicable_event["source_position"])
    != FIRST_APPLICABLE_DEPTH_POSITION
):
    raise RuntimeError(
        "Unexpected first applicable depth position."
    )

if (
    int(first_applicable_event["collector_sequence"])
    != FIRST_APPLICABLE_DEPTH_COLLECTOR_SEQUENCE
):
    raise RuntimeError(
        "Unexpected first applicable collector sequence."
    )

if (
    int(first_applicable_event["first_update_id"])
    != FIRST_APPLICABLE_DEPTH_U
    or int(first_applicable_event["final_update_id"])
    != FIRST_APPLICABLE_DEPTH_u
):
    raise RuntimeError(
        "Unexpected first applicable depth-update range."
    )

required_successor_update_id = (
    SNAPSHOT_LAST_UPDATE_ID + 1
)

if not (
    int(first_applicable_event["first_update_id"])
    <= required_successor_update_id
    <= int(first_applicable_event["final_update_id"])
):
    raise RuntimeError(
        "The first applicable depth event does not bridge "
        "the REST snapshot."
    )

previous_final_ids = (
    POST_SNAPSHOT_DEPTH_EVENTS[
        "final_update_id"
    ]
    .shift(1)
)

POST_SNAPSHOT_DEPTH_EVENTS[
    "expected_first_update_id"
] = previous_final_ids + 1

POST_SNAPSHOT_DEPTH_EVENTS[
    "continuity_status"
] = np.where(
    previous_final_ids.isna(),
    "SNAPSHOT_BRIDGE",
    np.where(
        POST_SNAPSHOT_DEPTH_EVENTS[
            "first_update_id"
        ].eq(
            POST_SNAPSHOT_DEPTH_EVENTS[
                "expected_first_update_id"
            ]
        ),
        "EXACT_SUCCESSOR",
        np.where(
            POST_SNAPSHOT_DEPTH_EVENTS[
                "first_update_id"
            ].le(
                previous_final_ids
            ),
            "OVERLAP",
            "GAP",
        ),
    ),
)

continuity_counts = (
    POST_SNAPSHOT_DEPTH_EVENTS[
        "continuity_status"
    ]
    .value_counts()
    .to_dict()
)

observed_gap_count = int(
    continuity_counts.get("GAP", 0)
)

observed_overlap_count = int(
    continuity_counts.get("OVERLAP", 0)
)

observed_exact_transition_count = int(
    continuity_counts.get("EXACT_SUCCESSOR", 0)
)

if observed_gap_count != POST_SNAPSHOT_GAP_COUNT:
    raise RuntimeError(
        "Post-snapshot gap-count mismatch."
    )

if (
    observed_overlap_count
    != POST_SNAPSHOT_OVERLAP_COUNT
):
    raise RuntimeError(
        "Post-snapshot overlap-count mismatch."
    )

expected_exact_transition_count = (
    len(POST_SNAPSHOT_DEPTH_EVENTS) - 1
)

if (
    observed_exact_transition_count
    != expected_exact_transition_count
):
    raise RuntimeError(
        "Not every post-bridge transition is an exact "
        "update-ID successor."
    )

POST_SNAPSHOT_DEPTH_EVENTS.reset_index(
    drop=True,
    inplace=True,
)


# ---------------------------------------------------------------------------
# Compact ingestion audit
# ---------------------------------------------------------------------------

DEPTH_INGESTION_AUDIT = pd.DataFrame(
    [
        {
            "metric": "snapshot_bid_levels",
            "observed": len(SNAPSHOT_BID_BOOK),
            "expected": EXPECTED_SNAPSHOT_LEVELS_PER_SIDE,
            "passed": (
                len(SNAPSHOT_BID_BOOK)
                == EXPECTED_SNAPSHOT_LEVELS_PER_SIDE
            ),
        },
        {
            "metric": "snapshot_ask_levels",
            "observed": len(SNAPSHOT_ASK_BOOK),
            "expected": EXPECTED_SNAPSHOT_LEVELS_PER_SIDE,
            "passed": (
                len(SNAPSHOT_ASK_BOOK)
                == EXPECTED_SNAPSHOT_LEVELS_PER_SIDE
            ),
        },
        {
            "metric": "depth_events",
            "observed": len(DEPTH_EVENTS),
            "expected": EXPECTED_DEPTH_EVENT_COUNT,
            "passed": (
                len(DEPTH_EVENTS)
                == EXPECTED_DEPTH_EVENT_COUNT
            ),
        },
        {
            "metric": "post_snapshot_events",
            "observed": len(
                POST_SNAPSHOT_DEPTH_EVENTS
            ),
            "expected": POST_SNAPSHOT_DEPTH_EVENT_COUNT,
            "passed": (
                len(POST_SNAPSHOT_DEPTH_EVENTS)
                == POST_SNAPSHOT_DEPTH_EVENT_COUNT
            ),
        },
        {
            "metric": "bid_level_updates",
            "observed": observed_bid_update_count,
            "expected": EXPECTED_BID_UPDATE_COUNT,
            "passed": (
                observed_bid_update_count
                == EXPECTED_BID_UPDATE_COUNT
            ),
        },
        {
            "metric": "ask_level_updates",
            "observed": observed_ask_update_count,
            "expected": EXPECTED_ASK_UPDATE_COUNT,
            "passed": (
                observed_ask_update_count
                == EXPECTED_ASK_UPDATE_COUNT
            ),
        },
        {
            "metric": "zero_quantity_deletions",
            "observed": observed_delete_update_count,
            "expected": EXPECTED_ZERO_QUANTITY_COUNT,
            "passed": (
                observed_delete_update_count
                == EXPECTED_ZERO_QUANTITY_COUNT
            ),
        },
        {
            "metric": "post_snapshot_gaps",
            "observed": observed_gap_count,
            "expected": POST_SNAPSHOT_GAP_COUNT,
            "passed": (
                observed_gap_count
                == POST_SNAPSHOT_GAP_COUNT
            ),
        },
        {
            "metric": "post_snapshot_overlaps",
            "observed": observed_overlap_count,
            "expected": POST_SNAPSHOT_OVERLAP_COUNT,
            "passed": (
                observed_overlap_count
                == POST_SNAPSHOT_OVERLAP_COUNT
            ),
        },
    ]
)

if not DEPTH_INGESTION_AUDIT["passed"].all():
    raise RuntimeError(
        "Snapshot/depth ingestion audit contains failed gates."
    )

display(DEPTH_INGESTION_AUDIT)

display(
    POST_SNAPSHOT_DEPTH_EVENTS[
        [
            "source_position",
            "collector_sequence",
            "local_receipt_time_utc",
            "exchange_event_time_utc",
            "first_update_id",
            "final_update_id",
            "bid_update_count",
            "ask_update_count",
            "delete_update_count",
            "continuity_status",
        ]
    ].head(5)
)

{
    "status": "PASS",
    "snapshot_last_update_id": SNAPSHOT_LAST_UPDATE_ID,
    "snapshot_bid_levels": len(SNAPSHOT_BID_BOOK),
    "snapshot_ask_levels": len(SNAPSHOT_ASK_BOOK),
    "snapshot_best_bid": str(snapshot_best_bid),
    "snapshot_best_ask": str(snapshot_best_ask),
    "depth_records_parsed": len(DEPTH_EVENTS),
    "depth_records_rejected": len(
        DEPTH_PARSE_REJECTIONS
    ),
    "stale_pre_snapshot_events": len(
        STALE_DEPTH_EVENTS
    ),
    "post_snapshot_events": len(
        POST_SNAPSHOT_DEPTH_EVENTS
    ),
    "first_applied_collector_sequence": int(
        POST_SNAPSHOT_DEPTH_EVENTS.iloc[0][
            "collector_sequence"
        ]
    ),
    "first_applied_update_range": {
        "U": int(
            POST_SNAPSHOT_DEPTH_EVENTS.iloc[0][
                "first_update_id"
            ]
        ),
        "u": int(
            POST_SNAPSHOT_DEPTH_EVENTS.iloc[0][
                "final_update_id"
            ]
        ),
    },
    "bid_level_updates": observed_bid_update_count,
    "ask_level_updates": observed_ask_update_count,
    "zero_quantity_deletions": (
        observed_delete_update_count
    ),
    "positive_quantity_updates": (
        observed_positive_update_count
    ),
    "exact_continuity_transitions": (
        observed_exact_transition_count
    ),
    "gap_count": observed_gap_count,
    "overlap_count": observed_overlap_count,
    "timestamp_units": {
        "exchange_event_time_raw": "ms",
        "local_receipt_time_ns": "ns",
    },
}

,metric,observed,expected,passed
0,snapshot_bid_levels,5000,5000,True
1,snapshot_ask_levels,5000,5000,True
2,depth_events,35994,35994,True
3,post_snapshot_events,35985,35985,True
4,bid_level_updates,332920,332920,True
5,ask_level_updates,277873,277873,True
6,zero_quantity_deletions,219617,219617,True
7,post_snapshot_gaps,0,0,True
8,post_snapshot_overlaps,0,0,True


,source_position,collector_sequence,local_receipt_time_utc,exchange_event_time_utc,first_update_id,final_update_id,bid_update_count,ask_update_count,delete_update_count,continuity_status
0,9,10,2026-07-10 06:37:48.432309400+00:00,2026-07-10 06:37:49.614000+00:00,97233590167,97233590170,1,0,0,SNAPSHOT_BRIDGE
1,10,11,2026-07-10 06:37:48.532310900+00:00,2026-07-10 06:37:49.714000+00:00,97233590171,97233590187,1,9,3,EXACT_SUCCESSOR
2,11,12,2026-07-10 06:37:48.632155900+00:00,2026-07-10 06:37:49.814000+00:00,97233590188,97233590208,3,7,0,EXACT_SUCCESSOR
3,12,13,2026-07-10 06:37:48.733660800+00:00,2026-07-10 06:37:49.915000+00:00,97233590209,97233590215,2,1,1,EXACT_SUCCESSOR
4,13,15,2026-07-10 06:37:48.832237300+00:00,2026-07-10 06:37:50.014000+00:00,97233590216,97233590243,9,3,6,EXACT_SUCCESSOR


{'status': 'PASS',
 'snapshot_last_update_id': 97233590166,
 'snapshot_bid_levels': 5000,
 'snapshot_ask_levels': 5000,
 'snapshot_best_bid': '63914.36000000',
 'snapshot_best_ask': '63914.37000000',
 'depth_records_parsed': 35994,
 'depth_records_rejected': 0,
 'stale_pre_snapshot_events': 9,
 'post_snapshot_events': 35985,
 'first_applied_collector_sequence': 10,
 'first_applied_update_range': {'U': 97233590167, 'u': 97233590170},
 'bid_level_updates': 332920,
 'ask_level_updates': 277873,
 'zero_quantity_deletions': 219617,
 'positive_quantity_updates': 391176,
 'exact_continuity_transitions': 35984,
 'gap_count': 0,
 'overlap_count': 0,
 'timestamp_units': {'exchange_event_time_raw': 'ms',
  'local_receipt_time_ns': 'ns'}}

In [4]:
# ---------------------------------------------------------------------------
# Independently reconstruct the visible market-by-price book
# ---------------------------------------------------------------------------

import heapq


EXPECTED_RECONSTRUCTED_STATE_COUNT: Final[int] = (
    POST_SNAPSHOT_DEPTH_EVENT_COUNT
)

RECONSTRUCTION_SESSION_ID: Final[str] = (
    f"{V0_1_RUN_ID}__book_session_0001"
)


# ---------------------------------------------------------------------------
# Heap-backed visible-book helpers
# ---------------------------------------------------------------------------

BookSide = dict[Decimal, Decimal]
GenerationMap = dict[Decimal, int]
HeapEntry = tuple[Decimal, int, Decimal]


def initialize_side_heap(
    book: BookSide,
    *,
    is_bid: bool,
) -> tuple[list[HeapEntry], GenerationMap]:
    """
    Build a heap and generation map for one visible book side.

    Generation numbers allow stale heap entries to be discarded after
    replacements, deletions, and later reinsertion of the same price.
    """
    generations: GenerationMap = {
        price: 0
        for price in book
    }

    heap: list[HeapEntry] = [
        (
            -price if is_bid else price,
            0,
            price,
        )
        for price in book
    ]

    heapq.heapify(heap)

    return heap, generations


def apply_side_updates(
    book: BookSide,
    heap: list[HeapEntry],
    generations: GenerationMap,
    updates: tuple[tuple[Decimal, Decimal], ...],
    *,
    is_bid: bool,
) -> dict[str, int]:
    """
    Apply absolute Binance MBP updates to one side.

    quantity > 0:
        Insert the price or replace its visible quantity.

    quantity == 0:
        Delete the exact price when present. Deleting an absent price is
        recorded as an idempotent no-op rather than treated as corruption.
    """
    action_counts = {
        "inserted": 0,
        "replaced": 0,
        "unchanged_replacement": 0,
        "deleted": 0,
        "absent_delete_noop": 0,
        "actual_changed_levels": 0,
    }

    for price, quantity in updates:
        previous_quantity = book.get(price)

        if quantity == 0:
            if previous_quantity is None:
                action_counts["absent_delete_noop"] += 1
                continue

            generations[price] = generations.get(price, 0) + 1
            del book[price]

            action_counts["deleted"] += 1
            action_counts["actual_changed_levels"] += 1
            continue

        if previous_quantity == quantity:
            action_counts["unchanged_replacement"] += 1
            continue

        generations[price] = generations.get(price, 0) + 1
        generation = generations[price]

        book[price] = quantity

        heapq.heappush(
            heap,
            (
                -price if is_bid else price,
                generation,
                price,
            ),
        )

        if previous_quantity is None:
            action_counts["inserted"] += 1
        else:
            action_counts["replaced"] += 1

        action_counts["actual_changed_levels"] += 1

    return action_counts


def extract_top_levels(
    book: BookSide,
    heap: list[HeapEntry],
    generations: GenerationMap,
    *,
    depth: int,
) -> tuple[tuple[Decimal, Decimal], ...]:
    """
    Return the best valid levels while lazily removing stale heap entries.
    """
    retained_entries: list[HeapEntry] = []
    top_levels: list[tuple[Decimal, Decimal]] = []

    while heap and len(top_levels) < depth:
        heap_key, generation, price = heapq.heappop(heap)

        current_quantity = book.get(price)
        current_generation = generations.get(price)

        if current_quantity is None:
            continue

        if current_generation != generation:
            continue

        retained_entries.append(
            (
                heap_key,
                generation,
                price,
            )
        )

        top_levels.append(
            (
                price,
                current_quantity,
            )
        )

    for entry in retained_entries:
        heapq.heappush(heap, entry)

    return tuple(top_levels)


def validate_top_levels(
    bid_levels: tuple[tuple[Decimal, Decimal], ...],
    ask_levels: tuple[tuple[Decimal, Decimal], ...],
    *,
    context: str,
) -> None:
    """Validate every retained top-ten reconstruction invariant."""
    if len(bid_levels) != TOP_N_LEVELS:
        raise RuntimeError(
            f"{context}: only {len(bid_levels)} bid levels are "
            f"available; {TOP_N_LEVELS} are required."
        )

    if len(ask_levels) != TOP_N_LEVELS:
        raise RuntimeError(
            f"{context}: only {len(ask_levels)} ask levels are "
            f"available; {TOP_N_LEVELS} are required."
        )

    bid_prices = [
        price
        for price, _ in bid_levels
    ]

    ask_prices = [
        price
        for price, _ in ask_levels
    ]

    bid_quantities = [
        quantity
        for _, quantity in bid_levels
    ]

    ask_quantities = [
        quantity
        for _, quantity in ask_levels
    ]

    if len(set(bid_prices)) != TOP_N_LEVELS:
        raise RuntimeError(
            f"{context}: duplicate retained bid prices."
        )

    if len(set(ask_prices)) != TOP_N_LEVELS:
        raise RuntimeError(
            f"{context}: duplicate retained ask prices."
        )

    if not all(
        left_price > right_price
        for left_price, right_price in zip(
            bid_prices,
            bid_prices[1:],
        )
    ):
        raise RuntimeError(
            f"{context}: bid prices are not strictly descending."
        )

    if not all(
        left_price < right_price
        for left_price, right_price in zip(
            ask_prices,
            ask_prices[1:],
        )
    ):
        raise RuntimeError(
            f"{context}: ask prices are not strictly ascending."
        )

    if not all(
        quantity > 0
        for quantity in bid_quantities
    ):
        raise RuntimeError(
            f"{context}: a retained bid quantity is not positive."
        )

    if not all(
        quantity > 0
        for quantity in ask_quantities
    ):
        raise RuntimeError(
            f"{context}: a retained ask quantity is not positive."
        )


def top_state_sha256(
    bid_levels: tuple[tuple[Decimal, Decimal], ...],
    ask_levels: tuple[tuple[Decimal, Decimal], ...],
) -> str:
    """Hash the exact top-ten price and quantity representation."""
    state_payload = {
        "bids": [
            [
                format(price, "f"),
                format(quantity, "f"),
            ]
            for price, quantity in bid_levels
        ],
        "asks": [
            [
                format(price, "f"),
                format(quantity, "f"),
            ]
            for price, quantity in ask_levels
        ],
    }

    encoded = json.dumps(
        state_payload,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
    ).encode("utf-8")

    return hashlib.sha256(encoded).hexdigest()


def combine_action_counts(
    bid_actions: dict[str, int],
    ask_actions: dict[str, int],
) -> dict[str, int]:
    """Combine bid- and ask-side action counts."""
    return {
        action_name: (
            bid_actions[action_name]
            + ask_actions[action_name]
        )
        for action_name in bid_actions
    }


# ---------------------------------------------------------------------------
# Initialize independent mutable book state from the frozen REST snapshot
# ---------------------------------------------------------------------------

visible_bid_book: BookSide = dict(SNAPSHOT_BID_BOOK)
visible_ask_book: BookSide = dict(SNAPSHOT_ASK_BOOK)

bid_heap, bid_generations = initialize_side_heap(
    visible_bid_book,
    is_bid=True,
)

ask_heap, ask_generations = initialize_side_heap(
    visible_ask_book,
    is_bid=False,
)

initial_bid_levels = extract_top_levels(
    visible_bid_book,
    bid_heap,
    bid_generations,
    depth=TOP_N_LEVELS,
)

initial_ask_levels = extract_top_levels(
    visible_ask_book,
    ask_heap,
    ask_generations,
    depth=TOP_N_LEVELS,
)

validate_top_levels(
    initial_bid_levels,
    initial_ask_levels,
    context="REST snapshot initialization",
)

if initial_bid_levels[0][0] >= initial_ask_levels[0][0]:
    raise RuntimeError(
        "REST snapshot initialization is locked or crossed."
    )


# ---------------------------------------------------------------------------
# Reconstruction ledgers
# ---------------------------------------------------------------------------

book_state_rows: list[dict[str, Any]] = []
update_ledger_rows: list[dict[str, Any]] = []
rejected_update_rows: list[dict[str, Any]] = []
resynchronization_rows: list[dict[str, Any]] = []

previous_final_update_id = SNAPSHOT_LAST_UPDATE_ID
previous_collector_sequence: int | None = None
previous_local_receipt_ns: int | None = None
previous_exchange_event_ms: int | None = None

previous_best_bid: Decimal | None = None
previous_best_ask: Decimal | None = None
previous_midpoint: Decimal | None = None

previous_bid_levels: (
    tuple[tuple[Decimal, Decimal], ...] | None
) = None

previous_ask_levels: (
    tuple[tuple[Decimal, Decimal], ...] | None
) = None

reconstruction_halted = False
reconstruction_started_at = perf_counter()


# ---------------------------------------------------------------------------
# Apply every authoritative post-snapshot update in collector order
# ---------------------------------------------------------------------------

for state_id, event in enumerate(
    POST_SNAPSHOT_DEPTH_EVENTS.itertuples(index=False)
):
    collector_sequence = int(event.collector_sequence)
    local_receipt_time_ns = int(event.local_receipt_time_ns)
    exchange_event_time_ms = int(event.exchange_event_time_ms)
    first_update_id = int(event.first_update_id)
    final_update_id = int(event.final_update_id)

    update_context = (
        f"collector_sequence={collector_sequence}, "
        f"U={first_update_id}, u={final_update_id}"
    )

    is_first_applied_event = state_id == 0

    if is_first_applied_event:
        required_update_id = SNAPSHOT_LAST_UPDATE_ID + 1

        continuity_valid = (
            first_update_id
            <= required_update_id
            <= final_update_id
        )

        continuity_status = (
            "SNAPSHOT_BRIDGE"
            if continuity_valid
            else "INVALID_SNAPSHOT_BRIDGE"
        )

        expected_first_update_id: int | None = (
            required_update_id
        )

    else:
        expected_first_update_id = (
            previous_final_update_id + 1
        )

        continuity_valid = (
            first_update_id
            == expected_first_update_id
        )

        continuity_status = (
            "EXACT_SUCCESSOR"
            if continuity_valid
            else (
                "OVERLAP"
                if first_update_id
                <= previous_final_update_id
                else "GAP"
            )
        )

    collector_order_valid = (
        previous_collector_sequence is None
        or collector_sequence
        > previous_collector_sequence
    )

    local_time_order_valid = (
        previous_local_receipt_ns is None
        or local_receipt_time_ns
        >= previous_local_receipt_ns
    )

    if not (
        continuity_valid
        and collector_order_valid
        and local_time_order_valid
    ):
        failure_reasons: list[str] = []

        if not continuity_valid:
            failure_reasons.append(
                f"update continuity failure: "
                f"status={continuity_status}, "
                f"expected_U={expected_first_update_id}, "
                f"observed_U={first_update_id}"
            )

        if not collector_order_valid:
            failure_reasons.append(
                "collector_sequence is not strictly increasing"
            )

        if not local_time_order_valid:
            failure_reasons.append(
                "local receipt time reverses"
            )

        failure_detail = "; ".join(failure_reasons)

        rejected_update_rows.append(
            {
                "reconstruction_session_id": (
                    RECONSTRUCTION_SESSION_ID
                ),
                "state_id_attempted": state_id,
                "source_position": int(
                    event.source_position
                ),
                "source_line_number": int(
                    event.source_line_number
                ),
                "collector_sequence": collector_sequence,
                "local_receipt_time_ns": (
                    local_receipt_time_ns
                ),
                "first_update_id": first_update_id,
                "final_update_id": final_update_id,
                "rejection_code": (
                    "RECONSTRUCTION_CONTINUITY_FAILURE"
                ),
                "rejection_detail": failure_detail,
            }
        )

        resynchronization_rows.append(
            {
                "reconstruction_session_id": (
                    RECONSTRUCTION_SESSION_ID
                ),
                "trigger_collector_sequence": (
                    collector_sequence
                ),
                "trigger_first_update_id": first_update_id,
                "trigger_final_update_id": final_update_id,
                "trigger_reason": failure_detail,
                "resynchronization_required": True,
                "replacement_snapshot_available": False,
                "resolution_status": (
                    "UNRESOLVED_NO_LATER_SNAPSHOT"
                ),
            }
        )

        reconstruction_halted = True
        break

    try:
        bid_actions = apply_side_updates(
            visible_bid_book,
            bid_heap,
            bid_generations,
            event.bid_updates,
            is_bid=True,
        )

        ask_actions = apply_side_updates(
            visible_ask_book,
            ask_heap,
            ask_generations,
            event.ask_updates,
            is_bid=False,
        )

        combined_actions = combine_action_counts(
            bid_actions,
            ask_actions,
        )

        top_bid_levels = extract_top_levels(
            visible_bid_book,
            bid_heap,
            bid_generations,
            depth=TOP_N_LEVELS,
        )

        top_ask_levels = extract_top_levels(
            visible_ask_book,
            ask_heap,
            ask_generations,
            depth=TOP_N_LEVELS,
        )

        validate_top_levels(
            top_bid_levels,
            top_ask_levels,
            context=update_context,
        )

        best_bid, best_bid_quantity = top_bid_levels[0]
        best_ask, best_ask_quantity = top_ask_levels[0]

        locked_state = best_bid == best_ask
        crossed_state = best_bid > best_ask
        incomplete_state = (
            len(top_bid_levels) < TOP_N_LEVELS
            or len(top_ask_levels) < TOP_N_LEVELS
        )

        if locked_state or crossed_state or incomplete_state:
            raise RuntimeError(
                f"{update_context}: invalid reconstructed state; "
                f"locked={locked_state}, "
                f"crossed={crossed_state}, "
                f"incomplete={incomplete_state}, "
                f"best_bid={best_bid}, "
                f"best_ask={best_ask}"
            )

        spread = best_ask - best_bid
        midpoint = (
            best_bid + best_ask
        ) / Decimal("2")

        level_one_depth = (
            best_bid_quantity + best_ask_quantity
        )

        if level_one_depth <= 0:
            raise RuntimeError(
                f"{update_context}: invalid level-one depth."
            )

        top_level_imbalance = (
            best_bid_quantity - best_ask_quantity
        ) / level_one_depth

        visible_bid_depth = sum(
            (
                quantity
                for _, quantity in top_bid_levels
            ),
            Decimal("0"),
        )

        visible_ask_depth = sum(
            (
                quantity
                for _, quantity in top_ask_levels
            ),
            Decimal("0"),
        )

        visible_total_depth = (
            visible_bid_depth + visible_ask_depth
        )

        if visible_total_depth <= 0:
            raise RuntimeError(
                f"{update_context}: invalid visible depth."
            )

        multi_level_imbalance = (
            visible_bid_depth - visible_ask_depth
        ) / visible_total_depth

        microprice = (
            best_ask * best_bid_quantity
            + best_bid * best_ask_quantity
        ) / level_one_depth

        if not (
            best_bid
            <= microprice
            <= best_ask
        ):
            raise RuntimeError(
                f"{update_context}: microprice lies outside "
                "the visible spread."
            )

        bid_depth_concentration = (
            best_bid_quantity / visible_bid_depth
        )

        ask_depth_concentration = (
            best_ask_quantity / visible_ask_depth
        )

        combined_top_level_concentration = (
            level_one_depth / visible_total_depth
        )

        collector_gap_from_previous_book_update = (
            pd.NA
            if previous_collector_sequence is None
            else (
                collector_sequence
                - previous_collector_sequence
            )
        )

        local_interval_ns = (
            pd.NA
            if previous_local_receipt_ns is None
            else (
                local_receipt_time_ns
                - previous_local_receipt_ns
            )
        )

        exchange_interval_ms = (
            pd.NA
            if previous_exchange_event_ms is None
            else (
                exchange_event_time_ms
                - previous_exchange_event_ms
            )
        )

        best_bid_change = (
            pd.NA
            if previous_best_bid is None
            else best_bid - previous_best_bid
        )

        best_ask_change = (
            pd.NA
            if previous_best_ask is None
            else best_ask - previous_best_ask
        )

        midpoint_change = (
            pd.NA
            if previous_midpoint is None
            else midpoint - previous_midpoint
        )

        top_of_book_moved = (
            previous_best_bid is None
            or best_bid != previous_best_bid
            or best_ask != previous_best_ask
        )

        top_ten_changed = (
            previous_bid_levels is None
            or previous_ask_levels is None
            or top_bid_levels != previous_bid_levels
            or top_ask_levels != previous_ask_levels
        )

        if previous_midpoint is None:
            movement_code = "INITIAL_APPLIED_STATE"
        elif midpoint > previous_midpoint:
            movement_code = "MIDPOINT_UP"
        elif midpoint < previous_midpoint:
            movement_code = "MIDPOINT_DOWN"
        elif top_of_book_moved:
            movement_code = "QUOTE_CHANGE_FLAT_MIDPOINT"
        elif top_ten_changed:
            movement_code = "DEPTH_CHANGE_ONLY"
        else:
            movement_code = "NO_VISIBLE_TOP_10_CHANGE"

        state_row: dict[str, Any] = {
            "reconstruction_session_id": (
                RECONSTRUCTION_SESSION_ID
            ),
            "book_state_id": state_id,
            "source_position": int(
                event.source_position
            ),
            "source_line_number": int(
                event.source_line_number
            ),
            "connection_session_id": (
                event.connection_session_id
            ),
            "collector_sequence": collector_sequence,
            "collector_gap_from_previous_book_update": (
                collector_gap_from_previous_book_update
            ),
            "local_receipt_time_ns": (
                local_receipt_time_ns
            ),
            "local_receipt_time_utc": (
                event.local_receipt_time_utc
            ),
            "local_interval_ns": local_interval_ns,
            "exchange_event_time_ms": (
                exchange_event_time_ms
            ),
            "exchange_event_time_utc": (
                event.exchange_event_time_utc
            ),
            "exchange_interval_ms": (
                exchange_interval_ms
            ),
            "first_update_id": first_update_id,
            "final_update_id": final_update_id,
            "update_id_span": (
                final_update_id
                - first_update_id
                + 1
            ),
            "previous_final_update_id": (
                previous_final_update_id
            ),
            "expected_first_update_id": (
                expected_first_update_id
            ),
            "continuity_status": continuity_status,
            "collector_order_valid": (
                collector_order_valid
            ),
            "local_time_order_valid": (
                local_time_order_valid
            ),
            "bid_message_level_count": int(
                event.bid_update_count
            ),
            "ask_message_level_count": int(
                event.ask_update_count
            ),
            "message_level_count": int(
                event.level_update_count
            ),
            "message_delete_count": int(
                event.delete_update_count
            ),
            "inserted_level_count": (
                combined_actions["inserted"]
            ),
            "replaced_level_count": (
                combined_actions["replaced"]
            ),
            "unchanged_replacement_count": (
                combined_actions[
                    "unchanged_replacement"
                ]
            ),
            "deleted_level_count": (
                combined_actions["deleted"]
            ),
            "absent_delete_noop_count": (
                combined_actions[
                    "absent_delete_noop"
                ]
            ),
            "actual_changed_level_count": (
                combined_actions[
                    "actual_changed_levels"
                ]
            ),
            "full_bid_level_count": len(
                visible_bid_book
            ),
            "full_ask_level_count": len(
                visible_ask_book
            ),
            "retained_bid_level_count": len(
                top_bid_levels
            ),
            "retained_ask_level_count": len(
                top_ask_levels
            ),
            "best_bid": best_bid,
            "best_ask": best_ask,
            "best_bid_quantity": (
                best_bid_quantity
            ),
            "best_ask_quantity": (
                best_ask_quantity
            ),
            "spread": spread,
            "midpoint": midpoint,
            "microprice": microprice,
            "microprice_minus_midpoint": (
                microprice - midpoint
            ),
            "top_level_imbalance": (
                top_level_imbalance
            ),
            "multi_level_imbalance": (
                multi_level_imbalance
            ),
            "visible_bid_depth_top_10": (
                visible_bid_depth
            ),
            "visible_ask_depth_top_10": (
                visible_ask_depth
            ),
            "visible_total_depth_top_10": (
                visible_total_depth
            ),
            "bid_depth_concentration_l1_top_10": (
                bid_depth_concentration
            ),
            "ask_depth_concentration_l1_top_10": (
                ask_depth_concentration
            ),
            "combined_l1_depth_concentration_top_10": (
                combined_top_level_concentration
            ),
            "best_bid_change": best_bid_change,
            "best_ask_change": best_ask_change,
            "midpoint_change": midpoint_change,
            "top_of_book_moved": (
                top_of_book_moved
            ),
            "top_10_changed": top_ten_changed,
            "movement_code": movement_code,
            "locked_state": locked_state,
            "crossed_state": crossed_state,
            "incomplete_state": incomplete_state,
            "top_10_state_sha256": (
                top_state_sha256(
                    top_bid_levels,
                    top_ask_levels,
                )
            ),
            "state_accepted": True,
        }

        for level_number, (
            price,
            quantity,
        ) in enumerate(
            top_bid_levels,
            start=1,
        ):
            state_row[
                f"bid_price_{level_number}"
            ] = price

            state_row[
                f"bid_quantity_{level_number}"
            ] = quantity

        for level_number, (
            price,
            quantity,
        ) in enumerate(
            top_ask_levels,
            start=1,
        ):
            state_row[
                f"ask_price_{level_number}"
            ] = price

            state_row[
                f"ask_quantity_{level_number}"
            ] = quantity

        book_state_rows.append(state_row)

        update_ledger_rows.append(
            {
                "reconstruction_session_id": (
                    RECONSTRUCTION_SESSION_ID
                ),
                "book_state_id": state_id,
                "source_position": int(
                    event.source_position
                ),
                "collector_sequence": (
                    collector_sequence
                ),
                "local_receipt_time_ns": (
                    local_receipt_time_ns
                ),
                "first_update_id": (
                    first_update_id
                ),
                "final_update_id": (
                    final_update_id
                ),
                "expected_first_update_id": (
                    expected_first_update_id
                ),
                "continuity_status": (
                    continuity_status
                ),
                "collector_order_valid": (
                    collector_order_valid
                ),
                "local_time_order_valid": (
                    local_time_order_valid
                ),
                "state_accepted": True,
                "resynchronization_required": False,
            }
        )

        previous_final_update_id = final_update_id
        previous_collector_sequence = collector_sequence
        previous_local_receipt_ns = (
            local_receipt_time_ns
        )
        previous_exchange_event_ms = (
            exchange_event_time_ms
        )

        previous_best_bid = best_bid
        previous_best_ask = best_ask
        previous_midpoint = midpoint

        previous_bid_levels = top_bid_levels
        previous_ask_levels = top_ask_levels

    except Exception as exc:
        rejected_update_rows.append(
            {
                "reconstruction_session_id": (
                    RECONSTRUCTION_SESSION_ID
                ),
                "state_id_attempted": state_id,
                "source_position": int(
                    event.source_position
                ),
                "source_line_number": int(
                    event.source_line_number
                ),
                "collector_sequence": (
                    collector_sequence
                ),
                "local_receipt_time_ns": (
                    local_receipt_time_ns
                ),
                "first_update_id": (
                    first_update_id
                ),
                "final_update_id": (
                    final_update_id
                ),
                "rejection_code": (
                    "INVALID_RECONSTRUCTED_STATE"
                ),
                "rejection_detail": str(exc),
            }
        )

        resynchronization_rows.append(
            {
                "reconstruction_session_id": (
                    RECONSTRUCTION_SESSION_ID
                ),
                "trigger_collector_sequence": (
                    collector_sequence
                ),
                "trigger_first_update_id": (
                    first_update_id
                ),
                "trigger_final_update_id": (
                    final_update_id
                ),
                "trigger_reason": str(exc),
                "resynchronization_required": True,
                "replacement_snapshot_available": False,
                "resolution_status": (
                    "UNRESOLVED_NO_LATER_SNAPSHOT"
                ),
            }
        )

        reconstruction_halted = True
        break


reconstruction_elapsed_seconds = (
    perf_counter() - reconstruction_started_at
)


# ---------------------------------------------------------------------------
# Materialize reconstruction outputs
# ---------------------------------------------------------------------------

RECONSTRUCTED_BOOK_STATES = pd.DataFrame(
    book_state_rows
)

UPDATE_CONTINUITY_REPORT = pd.DataFrame(
    update_ledger_rows,
    columns=[
        "reconstruction_session_id",
        "book_state_id",
        "source_position",
        "collector_sequence",
        "local_receipt_time_ns",
        "first_update_id",
        "final_update_id",
        "expected_first_update_id",
        "continuity_status",
        "collector_order_valid",
        "local_time_order_valid",
        "state_accepted",
        "resynchronization_required",
    ],
)

REJECTED_BOOK_UPDATES = pd.DataFrame(
    rejected_update_rows,
    columns=[
        "reconstruction_session_id",
        "state_id_attempted",
        "source_position",
        "source_line_number",
        "collector_sequence",
        "local_receipt_time_ns",
        "first_update_id",
        "final_update_id",
        "rejection_code",
        "rejection_detail",
    ],
)

RESYNCHRONIZATION_LEDGER = pd.DataFrame(
    resynchronization_rows,
    columns=[
        "reconstruction_session_id",
        "trigger_collector_sequence",
        "trigger_first_update_id",
        "trigger_final_update_id",
        "trigger_reason",
        "resynchronization_required",
        "replacement_snapshot_available",
        "resolution_status",
    ],
)


if reconstruction_halted:
    display(REJECTED_BOOK_UPDATES.tail(10))
    display(RESYNCHRONIZATION_LEDGER.tail(10))

    raise RuntimeError(
        "Visible-book reconstruction halted after "
        f"{len(RECONSTRUCTED_BOOK_STATES):,} accepted states."
    )


# ---------------------------------------------------------------------------
# Full reconstructed-state invariant audit
# ---------------------------------------------------------------------------

if len(RECONSTRUCTED_BOOK_STATES) != (
    EXPECTED_RECONSTRUCTED_STATE_COUNT
):
    raise RuntimeError(
        "Reconstructed-state count mismatch:\n"
        f"expected: {EXPECTED_RECONSTRUCTED_STATE_COUNT:,}\n"
        f"observed: {len(RECONSTRUCTED_BOOK_STATES):,}"
    )

if not REJECTED_BOOK_UPDATES.empty:
    raise RuntimeError(
        "At least one depth update was rejected during "
        "reconstruction."
    )

if not RESYNCHRONIZATION_LEDGER.empty:
    raise RuntimeError(
        "At least one resynchronization was required."
    )

if RECONSTRUCTED_BOOK_STATES[
    "collector_sequence"
].duplicated().any():
    raise RuntimeError(
        "Duplicate book collector sequences exist."
    )

if not RECONSTRUCTED_BOOK_STATES[
    "collector_sequence"
].is_monotonic_increasing:
    raise RuntimeError(
        "Book collector sequences are not increasing."
    )

if not RECONSTRUCTED_BOOK_STATES[
    "local_receipt_time_ns"
].is_monotonic_increasing:
    raise RuntimeError(
        "Book local receipt times reverse."
    )

if not RECONSTRUCTED_BOOK_STATES[
    "final_update_id"
].is_monotonic_increasing:
    raise RuntimeError(
        "Final update IDs are not increasing."
    )

if not (
    RECONSTRUCTED_BOOK_STATES["best_bid"]
    < RECONSTRUCTED_BOOK_STATES["best_ask"]
).all():
    raise RuntimeError(
        "A retained state is locked or crossed."
    )

if RECONSTRUCTED_BOOK_STATES[
    [
        "locked_state",
        "crossed_state",
        "incomplete_state",
    ]
].any(axis=None):
    raise RuntimeError(
        "A retained state contains a book-quality failure."
    )

for level_number in range(1, TOP_N_LEVELS):
    next_level_number = level_number + 1

    if not (
        RECONSTRUCTED_BOOK_STATES[
            f"bid_price_{level_number}"
        ]
        > RECONSTRUCTED_BOOK_STATES[
            f"bid_price_{next_level_number}"
        ]
    ).all():
        raise RuntimeError(
            "Retained bid prices fail strict descending order "
            f"between levels {level_number} and "
            f"{next_level_number}."
        )

    if not (
        RECONSTRUCTED_BOOK_STATES[
            f"ask_price_{level_number}"
        ]
        < RECONSTRUCTED_BOOK_STATES[
            f"ask_price_{next_level_number}"
        ]
    ).all():
        raise RuntimeError(
            "Retained ask prices fail strict ascending order "
            f"between levels {level_number} and "
            f"{next_level_number}."
        )

for level_number in range(1, TOP_N_LEVELS + 1):
    if not (
        RECONSTRUCTED_BOOK_STATES[
            f"bid_quantity_{level_number}"
        ] > 0
    ).all():
        raise RuntimeError(
            f"Non-positive bid quantity at level "
            f"{level_number}."
        )

    if not (
        RECONSTRUCTED_BOOK_STATES[
            f"ask_quantity_{level_number}"
        ] > 0
    ).all():
        raise RuntimeError(
            f"Non-positive ask quantity at level "
            f"{level_number}."
        )

if (
    int(
        RECONSTRUCTED_BOOK_STATES.iloc[-1][
            "final_update_id"
        ]
    )
    != EXPECTED_LAST_OBSERVED_u
):
    raise RuntimeError(
        "Final reconstructed update ID does not match the "
        "validated depth-stream endpoint."
    )

if (
    int(
        RECONSTRUCTED_BOOK_STATES.iloc[0][
            "collector_sequence"
        ]
    )
    != FIRST_APPLICABLE_DEPTH_COLLECTOR_SEQUENCE
):
    raise RuntimeError(
        "Reconstruction did not begin at the authorized "
        "collector sequence."
    )


# ---------------------------------------------------------------------------
# Session boundary and wide top-ten table
# ---------------------------------------------------------------------------

RECONSTRUCTION_SESSION_LEDGER = pd.DataFrame(
    [
        {
            "reconstruction_session_id": (
                RECONSTRUCTION_SESSION_ID
            ),
            "initialization_type": "REST_SNAPSHOT",
            "snapshot_last_update_id": (
                SNAPSHOT_LAST_UPDATE_ID
            ),
            "first_collector_sequence": int(
                RECONSTRUCTED_BOOK_STATES.iloc[0][
                    "collector_sequence"
                ]
            ),
            "last_collector_sequence": int(
                RECONSTRUCTED_BOOK_STATES.iloc[-1][
                    "collector_sequence"
                ]
            ),
            "first_applied_update_id": int(
                RECONSTRUCTED_BOOK_STATES.iloc[0][
                    "first_update_id"
                ]
            ),
            "last_applied_update_id": int(
                RECONSTRUCTED_BOOK_STATES.iloc[-1][
                    "final_update_id"
                ]
            ),
            "accepted_state_count": len(
                RECONSTRUCTED_BOOK_STATES
            ),
            "rejected_update_count": len(
                REJECTED_BOOK_UPDATES
            ),
            "resynchronization_count": len(
                RESYNCHRONIZATION_LEDGER
            ),
            "session_status": "CLOSED_CONTINUOUS",
        }
    ]
)

top_level_columns = [
    column_name
    for level_number in range(
        1,
        TOP_N_LEVELS + 1,
    )
    for column_name in (
        f"bid_price_{level_number}",
        f"bid_quantity_{level_number}",
        f"ask_price_{level_number}",
        f"ask_quantity_{level_number}",
    )
]

TOP_10_VISIBLE_BOOK_WIDE = (
    RECONSTRUCTED_BOOK_STATES[
        [
            "reconstruction_session_id",
            "book_state_id",
            "collector_sequence",
            "local_receipt_time_ns",
            "local_receipt_time_utc",
            "exchange_event_time_ms",
            "exchange_event_time_utc",
            "first_update_id",
            "final_update_id",
            "top_10_state_sha256",
            *top_level_columns,
        ]
    ]
    .copy()
)


# ---------------------------------------------------------------------------
# Compact reconstruction audit
# ---------------------------------------------------------------------------

RECONSTRUCTION_CORE_AUDIT = pd.DataFrame(
    [
        {
            "metric": "accepted_book_states",
            "observed": len(
                RECONSTRUCTED_BOOK_STATES
            ),
            "expected": (
                EXPECTED_RECONSTRUCTED_STATE_COUNT
            ),
            "passed": (
                len(RECONSTRUCTED_BOOK_STATES)
                == EXPECTED_RECONSTRUCTED_STATE_COUNT
            ),
        },
        {
            "metric": "rejected_updates",
            "observed": len(
                REJECTED_BOOK_UPDATES
            ),
            "expected": 0,
            "passed": (
                REJECTED_BOOK_UPDATES.empty
            ),
        },
        {
            "metric": "resynchronizations",
            "observed": len(
                RESYNCHRONIZATION_LEDGER
            ),
            "expected": 0,
            "passed": (
                RESYNCHRONIZATION_LEDGER.empty
            ),
        },
        {
            "metric": "locked_states",
            "observed": int(
                RECONSTRUCTED_BOOK_STATES[
                    "locked_state"
                ].sum()
            ),
            "expected": 0,
            "passed": not RECONSTRUCTED_BOOK_STATES[
                "locked_state"
            ].any(),
        },
        {
            "metric": "crossed_states",
            "observed": int(
                RECONSTRUCTED_BOOK_STATES[
                    "crossed_state"
                ].sum()
            ),
            "expected": 0,
            "passed": not RECONSTRUCTED_BOOK_STATES[
                "crossed_state"
            ].any(),
        },
        {
            "metric": "incomplete_states",
            "observed": int(
                RECONSTRUCTED_BOOK_STATES[
                    "incomplete_state"
                ].sum()
            ),
            "expected": 0,
            "passed": not RECONSTRUCTED_BOOK_STATES[
                "incomplete_state"
            ].any(),
        },
        {
            "metric": "final_update_id",
            "observed": int(
                RECONSTRUCTED_BOOK_STATES.iloc[-1][
                    "final_update_id"
                ]
            ),
            "expected": EXPECTED_LAST_OBSERVED_u,
            "passed": (
                int(
                    RECONSTRUCTED_BOOK_STATES.iloc[-1][
                        "final_update_id"
                    ]
                )
                == EXPECTED_LAST_OBSERVED_u
            ),
        },
    ]
)

if not RECONSTRUCTION_CORE_AUDIT[
    "passed"
].all():
    raise RuntimeError(
        "The core reconstruction audit contains failed gates."
    )

display(RECONSTRUCTION_CORE_AUDIT)

display(
    RECONSTRUCTED_BOOK_STATES[
        [
            "book_state_id",
            "collector_sequence",
            "local_receipt_time_utc",
            "first_update_id",
            "final_update_id",
            "best_bid",
            "best_ask",
            "spread",
            "midpoint",
            "microprice",
            "top_level_imbalance",
            "multi_level_imbalance",
            "actual_changed_level_count",
            "movement_code",
        ]
    ].head(5)
)

{
    "status": "PASS",
    "reconstruction_session_id": (
        RECONSTRUCTION_SESSION_ID
    ),
    "accepted_book_states": len(
        RECONSTRUCTED_BOOK_STATES
    ),
    "top_10_wide_rows": len(
        TOP_10_VISIBLE_BOOK_WIDE
    ),
    "rejected_updates": len(
        REJECTED_BOOK_UPDATES
    ),
    "resynchronizations": len(
        RESYNCHRONIZATION_LEDGER
    ),
    "first_collector_sequence": int(
        RECONSTRUCTED_BOOK_STATES.iloc[0][
            "collector_sequence"
        ]
    ),
    "last_collector_sequence": int(
        RECONSTRUCTED_BOOK_STATES.iloc[-1][
            "collector_sequence"
        ]
    ),
    "first_update_range": {
        "U": int(
            RECONSTRUCTED_BOOK_STATES.iloc[0][
                "first_update_id"
            ]
        ),
        "u": int(
            RECONSTRUCTED_BOOK_STATES.iloc[0][
                "final_update_id"
            ]
        ),
    },
    "last_final_update_id": int(
        RECONSTRUCTED_BOOK_STATES.iloc[-1][
            "final_update_id"
        ]
    ),
    "locked_states": int(
        RECONSTRUCTED_BOOK_STATES[
            "locked_state"
        ].sum()
    ),
    "crossed_states": int(
        RECONSTRUCTED_BOOK_STATES[
            "crossed_state"
        ].sum()
    ),
    "incomplete_states": int(
        RECONSTRUCTED_BOOK_STATES[
            "incomplete_state"
        ].sum()
    ),
    "elapsed_seconds": round(
        reconstruction_elapsed_seconds,
        3,
    ),
}

,metric,observed,expected,passed
0,accepted_book_states,35985,35985,True
1,rejected_updates,0,0,True
2,resynchronizations,0,0,True
3,locked_states,0,0,True
4,crossed_states,0,0,True
5,incomplete_states,0,0,True
6,final_update_id,97234812218,97234812218,True


,book_state_id,collector_sequence,local_receipt_time_utc,first_update_id,final_update_id,best_bid,best_ask,spread,midpoint,microprice,top_level_imbalance,multi_level_imbalance,actual_changed_level_count,movement_code
0,0,10,2026-07-10 06:37:48.432309400+00:00,97233590167,97233590170,63914.36000000,63914.37000000,0.01000000,63914.36500000,63914.36599282807171928280717,0.1985656143438565614343856561,0.03686820994728397891359156544,1,INITIAL_APPLIED_STATE
1,1,11,2026-07-10 06:37:48.532310900+00:00,97233590171,97233590187,63914.36000000,63914.37000000,0.01000000,63914.36500000,63914.36599293354629495184163,0.1985867092589903683267992837,0.03686630874845977820806196092,9,DEPTH_CHANGE_ONLY
2,2,12,2026-07-10 06:37:48.632155900+00:00,97233590188,97233590208,63914.36000000,63914.37000000,0.01000000,63914.36500000,63914.36599293354629495184163,0.1985867092589903683267992837,0.03686630874845977820806196092,6,NO_VISIBLE_TOP_10_CHANGE
3,3,13,2026-07-10 06:37:48.733660800+00:00,97233590209,97233590215,63914.36000000,63914.37000000,0.01000000,63914.36500000,63914.36599254562176291204802,0.1985091243525824096035626119,0.03678859850513081939075645662,1,DEPTH_CHANGE_ONLY
4,4,15,2026-07-10 06:37:48.832237300+00:00,97233590216,97233590243,63914.36000000,63914.37000000,0.01000000,63914.36500000,63914.36599257199183271139900,0.1985143983665422798000422446,0.03679240093154603634496543377,6,DEPTH_CHANGE_ONLY


{'status': 'PASS',
 'reconstruction_session_id': 'v0_1_20260714T090616Z_e82325081a81__book_session_0001',
 'accepted_book_states': 35985,
 'top_10_wide_rows': 35985,
 'rejected_updates': 0,
 'resynchronizations': 0,
 'first_collector_sequence': 10,
 'last_collector_sequence': 103677,
 'first_update_range': {'U': 97233590167, 'u': 97233590170},
 'last_final_update_id': 97234812218,
 'locked_states': 0,
 'crossed_states': 0,
 'incomplete_states': 0,
 'elapsed_seconds': 4.223}

In [6]:
# ---------------------------------------------------------------------------
# Apply and audit the frozen chronological partition contract
# ---------------------------------------------------------------------------

from collections.abc import Mapping, Sequence


# ---------------------------------------------------------------------------
# Exact frozen engineering-partition specification
# ---------------------------------------------------------------------------

FROZEN_PARTITION_SPECS: Final[tuple[dict[str, Any], ...]] = (
    {
        "partition_order": 1,
        "partition": "DEVELOPMENT",
        "collector_sequence_start": 1,
        "collector_sequence_end_exclusive": 51_840,
        "row_count": 51_839,
        "trade_row_count": 33_820,
        "depth_row_count": 18_019,
    },
    {
        "partition_order": 2,
        "partition": "CALIBRATION",
        "collector_sequence_start": 51_840,
        "collector_sequence_end_exclusive": 72_576,
        "row_count": 20_736,
        "trade_row_count": 13_532,
        "depth_row_count": 7_204,
    },
    {
        "partition_order": 3,
        "partition": "VALIDATION",
        "collector_sequence_start": 72_576,
        "collector_sequence_end_exclusive": 88_127,
        "row_count": 15_551,
        "trade_row_count": 10_198,
        "depth_row_count": 5_353,
    },
    {
        "partition_order": 4,
        "partition": "ENGINEERING_HOLDOUT",
        "collector_sequence_start": 88_127,
        "collector_sequence_end_exclusive": 103_678,
        "row_count": 15_551,
        "trade_row_count": 10_133,
        "depth_row_count": 5_418,
    },
)

FROZEN_PARTITION_ORDER: Final[tuple[str, ...]] = tuple(
    specification["partition"]
    for specification in FROZEN_PARTITION_SPECS
)


# ---------------------------------------------------------------------------
# Cell-local validation helpers
# ---------------------------------------------------------------------------

def require_mapping(
    value: Any,
    *,
    label: str,
) -> Mapping[str, Any]:
    """Require a mapping without depending on helpers from another cell."""
    if not isinstance(value, Mapping):
        raise TypeError(
            f"{label} must be a mapping; "
            f"observed {type(value).__name__}."
        )

    return value


def require_sequence(
    value: Any,
    *,
    label: str,
) -> Sequence[Any]:
    """Require a non-string sequence."""
    if not isinstance(value, Sequence) or isinstance(
        value,
        (str, bytes, bytearray),
    ):
        raise TypeError(
            f"{label} must be a non-string sequence; "
            f"observed {type(value).__name__}."
        )

    return value


def require_exact_contract_value(
    contract: Mapping[str, Any],
    *,
    field_name: str,
    expected_value: Any,
    contract_label: str,
) -> None:
    """Require one exact top-level contract value."""
    if field_name not in contract:
        raise KeyError(
            f"{contract_label} lacks required field "
            f"{field_name!r}."
        )

    observed_value = contract[field_name]

    if observed_value != expected_value:
        raise RuntimeError(
            f"{contract_label} field {field_name!r} mismatch:\n"
            f"expected: {expected_value!r}\n"
            f"observed: {observed_value!r}"
        )


def require_integer_field(
    record: Mapping[str, Any],
    *,
    field_name: str,
    record_label: str,
) -> int:
    """Require an integer-like field without accepting booleans."""
    if field_name not in record:
        raise KeyError(
            f"{record_label} lacks required field "
            f"{field_name!r}."
        )

    value = record[field_name]

    if isinstance(value, bool):
        raise TypeError(
            f"{record_label}.{field_name} cannot be boolean."
        )

    if isinstance(value, (int, np.integer)):
        return int(value)

    if isinstance(value, float) and value.is_integer():
        return int(value)

    if isinstance(value, str):
        candidate = (
            value.strip()
            .replace(",", "")
            .replace("_", "")
        )

        if candidate.lstrip("+-").isdigit():
            return int(candidate)

    raise TypeError(
        f"{record_label}.{field_name} must be integer-like; "
        f"observed {value!r}."
    )


def assign_partition_labels(
    collector_sequences: pd.Series,
    boundary_table: pd.DataFrame,
) -> pd.Categorical:
    """
    Assign half-open collector-sequence partitions.

    Each sequence must be covered exactly once.
    """
    numeric_sequences = pd.to_numeric(
        collector_sequences,
        errors="raise",
    ).astype("int64")

    sequence_array = numeric_sequences.to_numpy(
        copy=False
    )

    labels = np.full(
        len(sequence_array),
        None,
        dtype=object,
    )

    coverage_count = np.zeros(
        len(sequence_array),
        dtype=np.int8,
    )

    for boundary in boundary_table.itertuples(
        index=False
    ):
        partition_name = str(boundary.partition)
        sequence_start = int(
            boundary.collector_sequence_start
        )
        sequence_end = int(
            boundary.collector_sequence_end_exclusive
        )

        mask = (
            (sequence_array >= sequence_start)
            & (sequence_array < sequence_end)
        )

        coverage_count[mask] += 1
        labels[mask] = partition_name

    multiply_covered = coverage_count > 1

    if multiply_covered.any():
        offending_sequences = sequence_array[
            multiply_covered
        ][:20]

        raise RuntimeError(
            "At least one collector sequence belongs to more "
            "than one frozen partition:\n"
            f"{offending_sequences.tolist()}"
        )

    uncovered = coverage_count == 0

    if uncovered.any():
        offending_sequences = sequence_array[
            uncovered
        ][:20]

        raise RuntimeError(
            "At least one collector sequence is outside the "
            "frozen partition contract:\n"
            f"{offending_sequences.tolist()}"
        )

    return pd.Categorical(
        labels,
        categories=list(FROZEN_PARTITION_ORDER),
        ordered=True,
    )


def add_frozen_partition(
    frame: pd.DataFrame,
    *,
    frame_label: str,
) -> pd.DataFrame:
    """Return a copy with one audited ordered partition column."""
    if "collector_sequence" not in frame.columns:
        raise KeyError(
            f"{frame_label} lacks collector_sequence."
        )

    partitioned = frame.copy()

    partitioned["partition"] = assign_partition_labels(
        partitioned["collector_sequence"],
        FROZEN_PARTITION_BOUNDARIES,
    )

    if partitioned["partition"].isna().any():
        raise RuntimeError(
            f"{frame_label} contains unassigned partitions."
        )

    return partitioned


# ---------------------------------------------------------------------------
# Validate the saved split-contract structure directly
# ---------------------------------------------------------------------------

split_contract = require_mapping(
    CHRONOLOGICAL_SPLIT_CONTRACT,
    label="Chronological split contract",
)

require_exact_contract_value(
    split_contract,
    field_name="operating_mode",
    expected_value=OPERATING_MODE,
    contract_label="Chronological split contract",
)

require_exact_contract_value(
    split_contract,
    field_name="split_method",
    expected_value="INTRA_SESSION_ENGINEERING",
    contract_label="Chronological split contract",
)

require_exact_contract_value(
    split_contract,
    field_name="primary_boundary_authority",
    expected_value="collector_sequence",
    contract_label="Chronological split contract",
)

require_exact_contract_value(
    split_contract,
    field_name="interval_convention",
    expected_value="[start, end)",
    contract_label="Chronological split contract",
)

require_exact_contract_value(
    split_contract,
    field_name="canonical_timezone",
    expected_value=CANONICAL_TIMEZONE,
    contract_label="Chronological split contract",
)

require_exact_contract_value(
    split_contract,
    field_name="partition_order",
    expected_value=list(FROZEN_PARTITION_ORDER),
    contract_label="Chronological split contract",
)

require_exact_contract_value(
    split_contract,
    field_name="partitions_are_independent",
    expected_value=False,
    contract_label="Chronological split contract",
)

require_exact_contract_value(
    split_contract,
    field_name="claim_bearing_holdout_available",
    expected_value=False,
    contract_label="Chronological split contract",
)

require_exact_contract_value(
    split_contract,
    field_name="final_partition_model_selection_access",
    expected_value=False,
    contract_label="Chronological split contract",
)


boundary_records = require_sequence(
    split_contract.get("boundary_table"),
    label="Chronological split contract boundary_table",
)

if len(boundary_records) != len(
    FROZEN_PARTITION_SPECS
):
    raise RuntimeError(
        "Frozen boundary-table row-count mismatch:\n"
        f"expected: {len(FROZEN_PARTITION_SPECS)}\n"
        f"observed: {len(boundary_records)}"
    )

normalized_boundary_rows: list[dict[str, Any]] = []

for boundary_index, raw_boundary in enumerate(
    boundary_records,
    start=1,
):
    boundary = require_mapping(
        raw_boundary,
        label=(
            "Chronological split boundary "
            f"{boundary_index}"
        ),
    )

    required_boundary_fields = {
        "partition_order",
        "partition",
        "boundary_authority",
        "collector_sequence_start",
        "collector_sequence_end_exclusive",
        "collector_sequence_last_included",
        "receipt_time_start_ns",
        "receipt_time_end_ns_exclusive",
        "row_count",
        "trade_row_count",
        "depth_row_count",
        "session_count",
        "independent_partition",
        "claim_authority",
        "interval_convention",
    }

    missing_fields = sorted(
        required_boundary_fields - set(boundary)
    )

    if missing_fields:
        raise RuntimeError(
            f"Boundary {boundary_index} is missing fields: "
            + ", ".join(missing_fields)
        )

    normalized_row = dict(boundary)

    for integer_field in (
        "partition_order",
        "collector_sequence_start",
        "collector_sequence_end_exclusive",
        "collector_sequence_last_included",
        "receipt_time_start_ns",
        "receipt_time_end_ns_exclusive",
        "row_count",
        "trade_row_count",
        "depth_row_count",
        "session_count",
    ):
        normalized_row[integer_field] = (
            require_integer_field(
                boundary,
                field_name=integer_field,
                record_label=(
                    f"boundary[{boundary_index}]"
                ),
            )
        )

    normalized_boundary_rows.append(
        normalized_row
    )


FROZEN_PARTITION_BOUNDARIES = (
    pd.DataFrame(normalized_boundary_rows)
    .sort_values(
        "partition_order",
        kind="stable",
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------------
# Reconcile every exact frozen boundary
# ---------------------------------------------------------------------------

for specification in FROZEN_PARTITION_SPECS:
    partition_name = specification["partition"]

    matches = FROZEN_PARTITION_BOUNDARIES.loc[
        FROZEN_PARTITION_BOUNDARIES[
            "partition"
        ].eq(partition_name)
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one frozen boundary for "
            f"{partition_name}; found {len(matches)}."
        )

    observed_boundary = matches.iloc[0]

    for field_name in (
        "partition_order",
        "collector_sequence_start",
        "collector_sequence_end_exclusive",
        "row_count",
        "trade_row_count",
        "depth_row_count",
    ):
        expected_value = int(
            specification[field_name]
        )

        observed_value = int(
            observed_boundary[field_name]
        )

        if observed_value != expected_value:
            raise RuntimeError(
                f"{partition_name} boundary mismatch for "
                f"{field_name}:\n"
                f"expected: {expected_value:,}\n"
                f"observed: {observed_value:,}"
            )

    if (
        observed_boundary["boundary_authority"]
        != "INTRA_SESSION_COLLECTOR_SEQUENCE"
    ):
        raise RuntimeError(
            f"{partition_name} has unexpected boundary "
            "authority."
        )

    if (
        observed_boundary["interval_convention"]
        != "[start, end)"
    ):
        raise RuntimeError(
            f"{partition_name} has an unexpected interval "
            "convention."
        )

    if bool(
        observed_boundary["independent_partition"]
    ):
        raise RuntimeError(
            f"{partition_name} is incorrectly marked as an "
            "independent partition."
        )

    if bool(observed_boundary["claim_authority"]):
        raise RuntimeError(
            f"{partition_name} is incorrectly marked as "
            "claim-bearing."
        )


# ---------------------------------------------------------------------------
# Boundary-table integrity
# ---------------------------------------------------------------------------

for row_index in range(
    len(FROZEN_PARTITION_BOUNDARIES)
):
    row = FROZEN_PARTITION_BOUNDARIES.iloc[
        row_index
    ]

    sequence_start = int(
        row["collector_sequence_start"]
    )

    sequence_end = int(
        row["collector_sequence_end_exclusive"]
    )

    last_included = int(
        row["collector_sequence_last_included"]
    )

    row_count = int(row["row_count"])

    if sequence_start >= sequence_end:
        raise RuntimeError(
            f"Invalid frozen interval for "
            f"{row['partition']}."
        )

    if last_included != sequence_end - 1:
        raise RuntimeError(
            f"Last-included sequence mismatch for "
            f"{row['partition']}."
        )

    if row_count != sequence_end - sequence_start:
        raise RuntimeError(
            f"Row-count versus interval-length mismatch for "
            f"{row['partition']}."
        )

    if row_index > 0:
        previous_end = int(
            FROZEN_PARTITION_BOUNDARIES.iloc[
                row_index - 1
            ]["collector_sequence_end_exclusive"]
        )

        if sequence_start != previous_end:
            raise RuntimeError(
                "Frozen partition boundaries are not "
                "contiguous."
            )


if (
    int(
        FROZEN_PARTITION_BOUNDARIES.iloc[0][
            "collector_sequence_start"
        ]
    )
    != 1
):
    raise RuntimeError(
        "Frozen partition coverage does not begin at "
        "collector sequence 1."
    )

if (
    int(
        FROZEN_PARTITION_BOUNDARIES.iloc[-1][
            "collector_sequence_end_exclusive"
        ]
    )
    != 103_678
):
    raise RuntimeError(
        "Frozen partition coverage does not end at "
        "collector sequence 103,677."
    )

if int(
    FROZEN_PARTITION_BOUNDARIES["row_count"].sum()
) != 103_677:
    raise RuntimeError(
        "Frozen partition total-record count mismatch."
    )

if int(
    FROZEN_PARTITION_BOUNDARIES[
        "trade_row_count"
    ].sum()
) != 67_683:
    raise RuntimeError(
        "Frozen partition trade-record count mismatch."
    )

if int(
    FROZEN_PARTITION_BOUNDARIES[
        "depth_row_count"
    ].sum()
) != 35_994:
    raise RuntimeError(
        "Frozen partition depth-record count mismatch."
    )


# ---------------------------------------------------------------------------
# Verify exact state-to-depth-event identity before partitioning
# ---------------------------------------------------------------------------

state_sequences = (
    RECONSTRUCTED_BOOK_STATES[
        "collector_sequence"
    ]
    .astype("int64")
    .to_numpy()
)

post_snapshot_sequences = (
    POST_SNAPSHOT_DEPTH_EVENTS[
        "collector_sequence"
    ]
    .astype("int64")
    .to_numpy()
)

if not np.array_equal(
    state_sequences,
    post_snapshot_sequences,
):
    raise RuntimeError(
        "Reconstructed state sequences do not exactly match "
        "the validated post-snapshot depth-event sequences."
    )


# ---------------------------------------------------------------------------
# Assign the frozen partition to every reconstruction output
# ---------------------------------------------------------------------------

RECONSTRUCTED_BOOK_STATES = add_frozen_partition(
    RECONSTRUCTED_BOOK_STATES,
    frame_label="Reconstructed book states",
)

TOP_10_VISIBLE_BOOK_WIDE = add_frozen_partition(
    TOP_10_VISIBLE_BOOK_WIDE,
    frame_label="Top-10 visible-book table",
)

UPDATE_CONTINUITY_REPORT = add_frozen_partition(
    UPDATE_CONTINUITY_REPORT,
    frame_label="Update-continuity report",
)

partitioned_stale_events = add_frozen_partition(
    STALE_DEPTH_EVENTS,
    frame_label="Stale pre-snapshot depth events",
)

partitioned_post_snapshot_events = (
    add_frozen_partition(
        POST_SNAPSHOT_DEPTH_EVENTS,
        frame_label="Post-snapshot depth events",
    )
)


# ---------------------------------------------------------------------------
# Cross-table partition identity checks
# ---------------------------------------------------------------------------

state_partition_labels = (
    RECONSTRUCTED_BOOK_STATES["partition"]
    .astype("string")
    .to_numpy()
)

top_10_partition_labels = (
    TOP_10_VISIBLE_BOOK_WIDE["partition"]
    .astype("string")
    .to_numpy()
)

continuity_partition_labels = (
    UPDATE_CONTINUITY_REPORT["partition"]
    .astype("string")
    .to_numpy()
)

post_snapshot_partition_labels = (
    partitioned_post_snapshot_events["partition"]
    .astype("string")
    .to_numpy()
)

if not np.array_equal(
    state_partition_labels,
    top_10_partition_labels,
):
    raise RuntimeError(
        "Partition assignments differ between the full-state "
        "and top-10 tables."
    )

if not np.array_equal(
    state_partition_labels,
    continuity_partition_labels,
):
    raise RuntimeError(
        "Partition assignments differ between reconstructed "
        "states and the continuity report."
    )

if not np.array_equal(
    state_partition_labels,
    post_snapshot_partition_labels,
):
    raise RuntimeError(
        "Partition assignments differ between reconstructed "
        "states and post-snapshot depth events."
    )


# ---------------------------------------------------------------------------
# Verify chronological partition progression
# ---------------------------------------------------------------------------

partition_codes = (
    RECONSTRUCTED_BOOK_STATES[
        "partition"
    ]
    .cat.codes
    .to_numpy()
)

partition_code_differences = np.diff(
    partition_codes
)

partition_reversal_count = int(
    np.count_nonzero(
        partition_code_differences < 0
    )
)

partition_skip_count = int(
    np.count_nonzero(
        partition_code_differences > 1
    )
)

partition_transition_count = int(
    np.count_nonzero(
        partition_code_differences > 0
    )
)

if partition_reversal_count != 0:
    raise RuntimeError(
        "Reconstructed states reverse across chronological "
        "partitions."
    )

if partition_skip_count != 0:
    raise RuntimeError(
        "Reconstructed states skip at least one frozen "
        "partition."
    )

if partition_transition_count != (
    len(FROZEN_PARTITION_ORDER) - 1
):
    raise RuntimeError(
        "Unexpected reconstructed partition-transition count:\n"
        f"expected: {len(FROZEN_PARTITION_ORDER) - 1}\n"
        f"observed: {partition_transition_count}"
    )


# ---------------------------------------------------------------------------
# Reconcile frozen raw depth counts, stale events, and retained states
# ---------------------------------------------------------------------------

stale_counts = (
    partitioned_stale_events.groupby(
        "partition",
        observed=False,
    )
    .size()
    .reindex(
        FROZEN_PARTITION_ORDER,
        fill_value=0,
    )
    .astype("int64")
)

post_snapshot_counts = (
    partitioned_post_snapshot_events.groupby(
        "partition",
        observed=False,
    )
    .size()
    .reindex(
        FROZEN_PARTITION_ORDER,
        fill_value=0,
    )
    .astype("int64")
)

reconstructed_counts = (
    RECONSTRUCTED_BOOK_STATES.groupby(
        "partition",
        observed=False,
    )
    .size()
    .reindex(
        FROZEN_PARTITION_ORDER,
        fill_value=0,
    )
    .astype("int64")
)

partition_summary_rows: list[dict[str, Any]] = []

for boundary in FROZEN_PARTITION_BOUNDARIES.itertuples(
    index=False
):
    partition_name = str(boundary.partition)

    raw_depth_count = int(
        boundary.depth_row_count
    )

    stale_count = int(
        stale_counts.loc[partition_name]
    )

    expected_reconstructed_count = (
        raw_depth_count - stale_count
    )

    post_snapshot_count = int(
        post_snapshot_counts.loc[
            partition_name
        ]
    )

    observed_reconstructed_count = int(
        reconstructed_counts.loc[
            partition_name
        ]
    )

    partition_states = (
        RECONSTRUCTED_BOOK_STATES.loc[
            RECONSTRUCTED_BOOK_STATES[
                "partition"
            ].eq(partition_name)
        ]
    )

    if partition_states.empty:
        first_state_sequence: int | None = None
        last_state_sequence: int | None = None
    else:
        first_state_sequence = int(
            partition_states[
                "collector_sequence"
            ].iloc[0]
        )

        last_state_sequence = int(
            partition_states[
                "collector_sequence"
            ].iloc[-1]
        )

    count_passed = (
        post_snapshot_count
        == expected_reconstructed_count
        == observed_reconstructed_count
    )

    interval_passed = bool(
        first_state_sequence is not None
        and last_state_sequence is not None
        and first_state_sequence
        >= int(boundary.collector_sequence_start)
        and last_state_sequence
        < int(
            boundary.collector_sequence_end_exclusive
        )
    )

    partition_summary_rows.append(
        {
            "partition_order": int(
                boundary.partition_order
            ),
            "partition": partition_name,
            "collector_sequence_start": int(
                boundary.collector_sequence_start
            ),
            "collector_sequence_end_exclusive": int(
                boundary.collector_sequence_end_exclusive
            ),
            "frozen_raw_depth_count": (
                raw_depth_count
            ),
            "stale_pre_snapshot_count": (
                stale_count
            ),
            "expected_reconstructed_count": (
                expected_reconstructed_count
            ),
            "post_snapshot_event_count": (
                post_snapshot_count
            ),
            "observed_reconstructed_count": (
                observed_reconstructed_count
            ),
            "first_reconstructed_sequence": (
                first_state_sequence
            ),
            "last_reconstructed_sequence": (
                last_state_sequence
            ),
            "count_passed": count_passed,
            "interval_passed": interval_passed,
            "partition_passed": (
                count_passed and interval_passed
            ),
        }
    )


RECONSTRUCTION_PARTITION_SUMMARY = (
    pd.DataFrame(partition_summary_rows)
    .sort_values(
        "partition_order",
        kind="stable",
    )
    .reset_index(drop=True)
)

if not RECONSTRUCTION_PARTITION_SUMMARY[
    "partition_passed"
].all():
    raise RuntimeError(
        "Reconstructed-state partition reconciliation "
        "contains failed gates:\n"
        + RECONSTRUCTION_PARTITION_SUMMARY.loc[
            ~RECONSTRUCTION_PARTITION_SUMMARY[
                "partition_passed"
            ]
        ].to_string(index=False)
    )


# ---------------------------------------------------------------------------
# Audit book-state continuity across partition boundaries
# ---------------------------------------------------------------------------

boundary_carry_rows: list[dict[str, Any]] = []

for partition_name in FROZEN_PARTITION_ORDER[1:]:
    partition_positions = np.flatnonzero(
        RECONSTRUCTED_BOOK_STATES[
            "partition"
        ]
        .astype("string")
        .eq(partition_name)
        .to_numpy()
    )

    if len(partition_positions) == 0:
        raise RuntimeError(
            f"No reconstructed states exist in "
            f"{partition_name}."
        )

    first_position = int(partition_positions[0])

    if first_position == 0:
        raise RuntimeError(
            f"{partition_name} unexpectedly begins at the "
            "first reconstructed state."
        )

    previous_state = (
        RECONSTRUCTED_BOOK_STATES.iloc[
            first_position - 1
        ]
    )

    first_partition_state = (
        RECONSTRUCTED_BOOK_STATES.iloc[
            first_position
        ]
    )

    previous_final_update_id = int(
        previous_state["final_update_id"]
    )

    first_update_id = int(
        first_partition_state[
            "first_update_id"
        ]
    )

    update_ids_contiguous = (
        first_update_id
        == previous_final_update_id + 1
    )

    state_ids_contiguous = (
        int(
            first_partition_state[
                "book_state_id"
            ]
        )
        == int(previous_state["book_state_id"]) + 1
    )

    boundary_carry_rows.append(
        {
            "partition": partition_name,
            "previous_partition": str(
                previous_state["partition"]
            ),
            "previous_book_state_id": int(
                previous_state["book_state_id"]
            ),
            "first_book_state_id": int(
                first_partition_state[
                    "book_state_id"
                ]
            ),
            "previous_collector_sequence": int(
                previous_state[
                    "collector_sequence"
                ]
            ),
            "first_collector_sequence": int(
                first_partition_state[
                    "collector_sequence"
                ]
            ),
            "previous_final_update_id": (
                previous_final_update_id
            ),
            "first_update_id": first_update_id,
            "state_ids_contiguous": (
                state_ids_contiguous
            ),
            "update_ids_contiguous": (
                update_ids_contiguous
            ),
            "book_history_carried": (
                state_ids_contiguous
                and update_ids_contiguous
            ),
        }
    )


PARTITION_BOUNDARY_CARRY_AUDIT = pd.DataFrame(
    boundary_carry_rows
)

if not PARTITION_BOUNDARY_CARRY_AUDIT[
    "book_history_carried"
].all():
    raise RuntimeError(
        "Book history was not carried continuously across "
        "every frozen partition boundary."
    )


# ---------------------------------------------------------------------------
# Final partition-assignment audit
# ---------------------------------------------------------------------------

PARTITION_ASSIGNMENT_AUDIT = pd.DataFrame(
    [
        {
            "metric": "frozen_partition_count",
            "observed": len(
                FROZEN_PARTITION_BOUNDARIES
            ),
            "expected": 4,
            "passed": (
                len(FROZEN_PARTITION_BOUNDARIES)
                == 4
            ),
        },
        {
            "metric": "reconstructed_state_count",
            "observed": len(
                RECONSTRUCTED_BOOK_STATES
            ),
            "expected": (
                POST_SNAPSHOT_DEPTH_EVENT_COUNT
            ),
            "passed": (
                len(RECONSTRUCTED_BOOK_STATES)
                == POST_SNAPSHOT_DEPTH_EVENT_COUNT
            ),
        },
        {
            "metric": "assigned_state_count",
            "observed": int(
                RECONSTRUCTED_BOOK_STATES[
                    "partition"
                ].notna().sum()
            ),
            "expected": len(
                RECONSTRUCTED_BOOK_STATES
            ),
            "passed": (
                RECONSTRUCTED_BOOK_STATES[
                    "partition"
                ].notna().all()
            ),
        },
        {
            "metric": "partition_reversals",
            "observed": partition_reversal_count,
            "expected": 0,
            "passed": (
                partition_reversal_count == 0
            ),
        },
        {
            "metric": "partition_skips",
            "observed": partition_skip_count,
            "expected": 0,
            "passed": (
                partition_skip_count == 0
            ),
        },
        {
            "metric": "partition_transitions",
            "observed": (
                partition_transition_count
            ),
            "expected": 3,
            "passed": (
                partition_transition_count == 3
            ),
        },
        {
            "metric": "boundary_carry_failures",
            "observed": int(
                (
                    ~PARTITION_BOUNDARY_CARRY_AUDIT[
                        "book_history_carried"
                    ]
                ).sum()
            ),
            "expected": 0,
            "passed": (
                PARTITION_BOUNDARY_CARRY_AUDIT[
                    "book_history_carried"
                ].all()
            ),
        },
    ]
)

if not PARTITION_ASSIGNMENT_AUDIT[
    "passed"
].all():
    raise RuntimeError(
        "Frozen partition-assignment audit contains "
        "failed gates."
    )


display(RECONSTRUCTION_PARTITION_SUMMARY)

display(PARTITION_BOUNDARY_CARRY_AUDIT)

display(PARTITION_ASSIGNMENT_AUDIT)

{
    "status": "PASS",
    "split_method": split_contract["split_method"],
    "boundary_authority": split_contract[
        "primary_boundary_authority"
    ],
    "interval_convention": split_contract[
        "interval_convention"
    ],
    "partition_order": list(
        FROZEN_PARTITION_ORDER
    ),
    "reconstructed_partition_counts": {
        partition_name: int(
            reconstructed_counts.loc[
                partition_name
            ]
        )
        for partition_name in (
            FROZEN_PARTITION_ORDER
        )
    },
    "stale_pre_snapshot_partition_counts": {
        partition_name: int(
            stale_counts.loc[
                partition_name
            ]
        )
        for partition_name in (
            FROZEN_PARTITION_ORDER
        )
    },
    "partition_reversals": (
        partition_reversal_count
    ),
    "partition_skips": partition_skip_count,
    "partition_transitions": (
        partition_transition_count
    ),
    "boundary_carry_failures": int(
        (
            ~PARTITION_BOUNDARY_CARRY_AUDIT[
                "book_history_carried"
            ]
        ).sum()
    ),
    "partitions_are_independent": False,
    "claim_bearing_holdout_available": False,
}

,partition_order,partition,collector_sequence_start,collector_sequence_end_exclusive,frozen_raw_depth_count,stale_pre_snapshot_count,expected_reconstructed_count,post_snapshot_event_count,observed_reconstructed_count,first_reconstructed_sequence,last_reconstructed_sequence,count_passed,interval_passed,partition_passed
0,1,DEVELOPMENT,1,51840,18019,9,18010,18010,18010,10,51839,True,True,True
1,2,CALIBRATION,51840,72576,7204,0,7204,7204,7204,51840,72575,True,True,True
2,3,VALIDATION,72576,88127,5353,0,5353,5353,5353,72577,88126,True,True,True
3,4,ENGINEERING_HOLDOUT,88127,103678,5418,0,5418,5418,5418,88127,103677,True,True,True


,partition,previous_partition,previous_book_state_id,first_book_state_id,previous_collector_sequence,first_collector_sequence,previous_final_update_id,first_update_id,state_ids_contiguous,update_ids_contiguous,book_history_carried
0,CALIBRATION,DEVELOPMENT,18009,18010,51839,51840,97234203793,97234203794,True,True,True
1,VALIDATION,CALIBRATION,25213,25214,72575,72577,97234465981,97234465982,True,True,True
2,ENGINEERING_HOLDOUT,VALIDATION,30566,30567,88126,88127,97234647326,97234647327,True,True,True


,metric,observed,expected,passed
0,frozen_partition_count,4,4,True
1,reconstructed_state_count,35985,35985,True
2,assigned_state_count,35985,35985,True
3,partition_reversals,0,0,True
4,partition_skips,0,0,True
5,partition_transitions,3,3,True
6,boundary_carry_failures,0,0,True


{'status': 'PASS',
 'split_method': 'INTRA_SESSION_ENGINEERING',
 'boundary_authority': 'collector_sequence',
 'interval_convention': '[start, end)',
 'partition_order': ['DEVELOPMENT',
  'CALIBRATION',
  'VALIDATION',
  'ENGINEERING_HOLDOUT'],
 'reconstructed_partition_counts': {'DEVELOPMENT': 18010,
  'CALIBRATION': 7204,
  'VALIDATION': 5353,
  'ENGINEERING_HOLDOUT': 5418},
 'stale_pre_snapshot_partition_counts': {'DEVELOPMENT': 9,
  'CALIBRATION': 0,
  'VALIDATION': 0,
  'ENGINEERING_HOLDOUT': 0},
 'partition_reversals': 0,
 'partition_skips': 0,
 'partition_transitions': 3,
 'boundary_carry_failures': 0,
 'partitions_are_independent': False,
 'claim_bearing_holdout_available': False}

In [7]:
# ---------------------------------------------------------------------------
# Book-quality, spread, depth-completeness, and interval diagnostics
# ---------------------------------------------------------------------------

import re


# ---------------------------------------------------------------------------
# Required in-memory inputs
# ---------------------------------------------------------------------------

required_objects = (
    "RECONSTRUCTED_BOOK_STATES",
    "TOP_10_VISIBLE_BOOK_WIDE",
    "UPDATE_CONTINUITY_REPORT",
    "RECONSTRUCTION_SESSION_LEDGER",
    "DEPTH_EVENTS",
    "RECONSTRUCTION_PARTITION_SUMMARY",
    "FROZEN_PARTITION_ORDER",
    "TOP_N_LEVELS",
)

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "This cell requires missing upstream objects: "
        + ", ".join(missing_objects)
    )


def require_columns(
    frame: pd.DataFrame,
    required_columns: Sequence[str],
    *,
    frame_label: str,
) -> None:
    """Require every named column in one DataFrame."""
    missing_columns = sorted(
        set(required_columns) - set(frame.columns)
    )

    if missing_columns:
        raise KeyError(
            f"{frame_label} is missing required columns: "
            + ", ".join(missing_columns)
        )


def decimal_value(
    value: Any,
    *,
    field_name: str,
) -> Decimal:
    """Return one finite Decimal without binary-float authority."""
    if isinstance(value, Decimal):
        result = value
    else:
        try:
            result = Decimal(str(value))
        except (
            InvalidOperation,
            TypeError,
            ValueError,
        ) as exc:
            raise ValueError(
                f"{field_name} cannot be represented as Decimal: "
                f"{value!r}"
            ) from exc

    if not result.is_finite():
        raise ValueError(
            f"{field_name} must be finite; observed {result!r}."
        )

    return result


def exact_increment_multiple(
    value: Any,
    increment: Decimal,
    *,
    field_name: str,
) -> bool:
    """Return True when value is an exact integer multiple."""
    numeric_value = decimal_value(
        value,
        field_name=field_name,
    )

    quotient = numeric_value / increment

    return quotient == quotient.to_integral_value()


def decimal_increment_count(
    value: Any,
    increment: Decimal,
    *,
    field_name: str,
) -> int:
    """Convert an exact Decimal multiple into its integer count."""
    numeric_value = decimal_value(
        value,
        field_name=field_name,
    )

    quotient = numeric_value / increment
    integral_quotient = quotient.to_integral_value()

    if quotient != integral_quotient:
        raise ValueError(
            f"{field_name}={numeric_value} is not an exact "
            f"multiple of {increment}."
        )

    return int(integral_quotient)


def finite_float_series(
    series: pd.Series,
    *,
    field_name: str,
) -> pd.Series:
    """Convert a complete numeric series for summary calculations."""
    converted = pd.to_numeric(
        series,
        errors="coerce",
    ).astype("float64")

    invalid_mask = ~np.isfinite(
        converted.to_numpy()
    )

    if invalid_mask.any():
        invalid_count = int(
            invalid_mask.sum()
        )

        raise RuntimeError(
            f"{field_name} contains {invalid_count:,} "
            "non-finite values."
        )

    return converted


def iter_partition_scopes(
    frame: pd.DataFrame,
):
    """Yield the full sample followed by each frozen partition."""
    yield "ALL", frame

    for partition_name in FROZEN_PARTITION_ORDER:
        partition_frame = frame.loc[
            frame["partition"].eq(
                partition_name
            )
        ]

        if partition_frame.empty:
            raise RuntimeError(
                f"No reconstructed states exist in "
                f"{partition_name}."
            )

        yield partition_name, partition_frame


# ---------------------------------------------------------------------------
# Required reconstructed-state schema
# ---------------------------------------------------------------------------

bid_price_columns = [
    f"bid_price_{level_number}"
    for level_number in range(
        1,
        TOP_N_LEVELS + 1,
    )
]

ask_price_columns = [
    f"ask_price_{level_number}"
    for level_number in range(
        1,
        TOP_N_LEVELS + 1,
    )
]

bid_quantity_columns = [
    f"bid_quantity_{level_number}"
    for level_number in range(
        1,
        TOP_N_LEVELS + 1,
    )
]

ask_quantity_columns = [
    f"ask_quantity_{level_number}"
    for level_number in range(
        1,
        TOP_N_LEVELS + 1,
    )
]

required_state_columns = [
    "book_state_id",
    "collector_sequence",
    "local_receipt_time_ns",
    "local_interval_ns",
    "first_update_id",
    "final_update_id",
    "continuity_status",
    "partition",
    "state_accepted",
    "retained_bid_level_count",
    "retained_ask_level_count",
    "full_bid_level_count",
    "full_ask_level_count",
    "best_bid",
    "best_ask",
    "spread",
    "midpoint",
    "microprice",
    "top_level_imbalance",
    "multi_level_imbalance",
    "visible_bid_depth_top_10",
    "visible_ask_depth_top_10",
    "visible_total_depth_top_10",
    "top_of_book_moved",
    "top_10_changed",
    "movement_code",
    "locked_state",
    "crossed_state",
    "incomplete_state",
    "top_10_state_sha256",
    *bid_price_columns,
    *ask_price_columns,
    *bid_quantity_columns,
    *ask_quantity_columns,
]

require_columns(
    RECONSTRUCTED_BOOK_STATES,
    required_state_columns,
    frame_label="Reconstructed book states",
)


# ---------------------------------------------------------------------------
# Independently infer and validate the observed price grid
# ---------------------------------------------------------------------------

positive_price_gap_candidates: list[Decimal] = []

for spread_value in (
    RECONSTRUCTED_BOOK_STATES[
        "spread"
    ].drop_duplicates()
):
    numeric_spread = decimal_value(
        spread_value,
        field_name="spread",
    )

    if numeric_spread > 0:
        positive_price_gap_candidates.append(
            numeric_spread
        )

for level_number in range(
    1,
    TOP_N_LEVELS,
):
    next_level_number = level_number + 1

    bid_gaps = (
        RECONSTRUCTED_BOOK_STATES[
            f"bid_price_{level_number}"
        ]
        - RECONSTRUCTED_BOOK_STATES[
            f"bid_price_{next_level_number}"
        ]
    )

    ask_gaps = (
        RECONSTRUCTED_BOOK_STATES[
            f"ask_price_{next_level_number}"
        ]
        - RECONSTRUCTED_BOOK_STATES[
            f"ask_price_{level_number}"
        ]
    )

    positive_bid_gaps = [
        decimal_value(
            value,
            field_name=(
                f"bid gap {level_number}-"
                f"{next_level_number}"
            ),
        )
        for value in bid_gaps.drop_duplicates()
        if decimal_value(
            value,
            field_name="bid gap",
        ) > 0
    ]

    positive_ask_gaps = [
        decimal_value(
            value,
            field_name=(
                f"ask gap {level_number}-"
                f"{next_level_number}"
            ),
        )
        for value in ask_gaps.drop_duplicates()
        if decimal_value(
            value,
            field_name="ask gap",
        ) > 0
    ]

    positive_price_gap_candidates.extend(
        positive_bid_gaps
    )

    positive_price_gap_candidates.extend(
        positive_ask_gaps
    )

if not positive_price_gap_candidates:
    raise RuntimeError(
        "No positive price increment could be inferred."
    )

OBSERVED_PRICE_GRID = min(
    positive_price_gap_candidates
)

if OBSERVED_PRICE_GRID <= 0:
    raise RuntimeError(
        "The inferred price grid is not positive."
    )


unique_retained_prices: set[Decimal] = set()

for price_column in (
    bid_price_columns + ask_price_columns
):
    unique_retained_prices.update(
        decimal_value(
            value,
            field_name=price_column,
        )
        for value in (
            RECONSTRUCTED_BOOK_STATES[
                price_column
            ].drop_duplicates()
        )
    )

off_grid_prices = sorted(
    price
    for price in unique_retained_prices
    if not exact_increment_multiple(
        price,
        OBSERVED_PRICE_GRID,
        field_name="retained price",
    )
)

unique_spreads = {
    decimal_value(
        value,
        field_name="spread",
    )
    for value in (
        RECONSTRUCTED_BOOK_STATES[
            "spread"
        ].drop_duplicates()
    )
}

off_grid_spreads = sorted(
    spread_value
    for spread_value in unique_spreads
    if not exact_increment_multiple(
        spread_value,
        OBSERVED_PRICE_GRID,
        field_name="spread",
    )
)

if off_grid_prices:
    raise RuntimeError(
        "Retained prices exist outside the inferred price grid. "
        f"First examples: {off_grid_prices[:10]}"
    )

if off_grid_spreads:
    raise RuntimeError(
        "Retained spreads exist outside the inferred price grid. "
        f"First examples: {off_grid_spreads[:10]}"
    )


spread_tick_values = [
    decimal_increment_count(
        spread_value,
        OBSERVED_PRICE_GRID,
        field_name="spread",
    )
    for spread_value in (
        RECONSTRUCTED_BOOK_STATES[
            "spread"
        ]
    )
]

RECONSTRUCTED_BOOK_STATES[
    "spread_ticks"
] = pd.Series(
    spread_tick_values,
    index=RECONSTRUCTED_BOOK_STATES.index,
    dtype="Int64",
)

if (
    RECONSTRUCTED_BOOK_STATES[
        "spread_ticks"
    ] < 1
).any():
    raise RuntimeError(
        "At least one reconstructed spread is below one "
        "observed price-grid increment."
    )


midpoint_float = finite_float_series(
    RECONSTRUCTED_BOOK_STATES[
        "midpoint"
    ],
    field_name="midpoint",
)

spread_float = finite_float_series(
    RECONSTRUCTED_BOOK_STATES[
        "spread"
    ],
    field_name="spread",
)

RECONSTRUCTED_BOOK_STATES[
    "spread_basis_points"
] = (
    spread_float
    / midpoint_float
    * 10_000.0
)


# ---------------------------------------------------------------------------
# Derive the nominal depth cadence and flag stale intervals
# ---------------------------------------------------------------------------

stream_values = (
    DEPTH_EVENTS["stream"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .tolist()
)

if len(stream_values) != 1:
    raise RuntimeError(
        "A unique differential-depth stream cadence could not "
        f"be established: {stream_values}"
    )

DEPTH_STREAM_NAME = stream_values[0]

cadence_match = re.search(
    r"@(\d+)ms$",
    DEPTH_STREAM_NAME,
    flags=re.IGNORECASE,
)

if cadence_match is None:
    raise RuntimeError(
        "The differential-depth stream name does not expose "
        f"an explicit millisecond cadence: {DEPTH_STREAM_NAME!r}"
    )

NOMINAL_DEPTH_INTERVAL_MS = int(
    cadence_match.group(1)
)

if NOMINAL_DEPTH_INTERVAL_MS <= 0:
    raise RuntimeError(
        "The parsed depth-stream cadence is not positive."
    )

NOMINAL_DEPTH_INTERVAL_NS = (
    NOMINAL_DEPTH_INTERVAL_MS
    * 1_000_000
)

# A state is operationally stale only after the interval exceeds both:
# - ten scheduled depth intervals, and
# - one full second.
#
# This is a diagnostic classification, not a rejection rule.
STALE_INTERVAL_MULTIPLIER = 10

STALE_INTERVAL_THRESHOLD_NS = max(
    NOMINAL_DEPTH_INTERVAL_NS
    * STALE_INTERVAL_MULTIPLIER,
    1_000_000_000,
)

local_interval_ns = pd.to_numeric(
    RECONSTRUCTED_BOOK_STATES[
        "local_interval_ns"
    ],
    errors="coerce",
).astype("Float64")

negative_interval_mask = (
    local_interval_ns < 0
).fillna(False)

if negative_interval_mask.any():
    raise RuntimeError(
        "At least one reconstructed local interval is negative."
    )

RECONSTRUCTED_BOOK_STATES[
    "interval_multiple_of_nominal"
] = (
    local_interval_ns
    / float(NOMINAL_DEPTH_INTERVAL_NS)
)

RECONSTRUCTED_BOOK_STATES[
    "stale_interval"
] = (
    local_interval_ns
    .gt(STALE_INTERVAL_THRESHOLD_NS)
    .fillna(False)
    .astype(bool)
)

RECONSTRUCTED_BOOK_STATES[
    "stale_interval_threshold_ns"
] = STALE_INTERVAL_THRESHOLD_NS


# ---------------------------------------------------------------------------
# Propagate state annotations to aligned reconstruction outputs
# ---------------------------------------------------------------------------

for target_label, target_frame in (
    (
        "Top-10 visible-book table",
        TOP_10_VISIBLE_BOOK_WIDE,
    ),
    (
        "Update-continuity report",
        UPDATE_CONTINUITY_REPORT,
    ),
):
    require_columns(
        target_frame,
        [
            "book_state_id",
            "collector_sequence",
        ],
        frame_label=target_label,
    )

    if len(target_frame) != len(
        RECONSTRUCTED_BOOK_STATES
    ):
        raise RuntimeError(
            f"{target_label} row count does not match the "
            "reconstructed state table."
        )

    if not np.array_equal(
        target_frame[
            "book_state_id"
        ].astype("int64").to_numpy(),
        RECONSTRUCTED_BOOK_STATES[
            "book_state_id"
        ].astype("int64").to_numpy(),
    ):
        raise RuntimeError(
            f"{target_label} book-state identity does not match "
            "the reconstructed state table."
        )

    if not np.array_equal(
        target_frame[
            "collector_sequence"
        ].astype("int64").to_numpy(),
        RECONSTRUCTED_BOOK_STATES[
            "collector_sequence"
        ].astype("int64").to_numpy(),
    ):
        raise RuntimeError(
            f"{target_label} collector ordering does not match "
            "the reconstructed state table."
        )

    target_frame["spread_ticks"] = (
        RECONSTRUCTED_BOOK_STATES[
            "spread_ticks"
        ].to_numpy()
    )

    target_frame["spread_basis_points"] = (
        RECONSTRUCTED_BOOK_STATES[
            "spread_basis_points"
        ].to_numpy()
    )

    target_frame[
        "interval_multiple_of_nominal"
    ] = (
        RECONSTRUCTED_BOOK_STATES[
            "interval_multiple_of_nominal"
        ].to_numpy()
    )

    target_frame["stale_interval"] = (
        RECONSTRUCTED_BOOK_STATES[
            "stale_interval"
        ].to_numpy()
    )


RECONSTRUCTION_SESSION_LEDGER[
    "depth_stream"
] = DEPTH_STREAM_NAME

RECONSTRUCTION_SESSION_LEDGER[
    "nominal_depth_interval_ms"
] = NOMINAL_DEPTH_INTERVAL_MS

RECONSTRUCTION_SESSION_LEDGER[
    "stale_interval_threshold_ns"
] = STALE_INTERVAL_THRESHOLD_NS

RECONSTRUCTION_SESSION_LEDGER[
    "stale_interval_count"
] = int(
    RECONSTRUCTED_BOOK_STATES[
        "stale_interval"
    ].sum()
)


# ---------------------------------------------------------------------------
# Independently recompute structural failure counts
# ---------------------------------------------------------------------------

bid_order_failure_mask = pd.Series(
    False,
    index=RECONSTRUCTED_BOOK_STATES.index,
)

ask_order_failure_mask = pd.Series(
    False,
    index=RECONSTRUCTED_BOOK_STATES.index,
)

for level_number in range(
    1,
    TOP_N_LEVELS,
):
    next_level_number = level_number + 1

    bid_order_failure_mask |= ~(
        RECONSTRUCTED_BOOK_STATES[
            f"bid_price_{level_number}"
        ]
        > RECONSTRUCTED_BOOK_STATES[
            f"bid_price_{next_level_number}"
        ]
    )

    ask_order_failure_mask |= ~(
        RECONSTRUCTED_BOOK_STATES[
            f"ask_price_{level_number}"
        ]
        < RECONSTRUCTED_BOOK_STATES[
            f"ask_price_{next_level_number}"
        ]
    )


bid_quantity_failure_mask = pd.Series(
    False,
    index=RECONSTRUCTED_BOOK_STATES.index,
)

ask_quantity_failure_mask = pd.Series(
    False,
    index=RECONSTRUCTED_BOOK_STATES.index,
)

for quantity_column in bid_quantity_columns:
    bid_quantity_failure_mask |= ~(
        RECONSTRUCTED_BOOK_STATES[
            quantity_column
        ] > 0
    )

for quantity_column in ask_quantity_columns:
    ask_quantity_failure_mask |= ~(
        RECONSTRUCTED_BOOK_STATES[
            quantity_column
        ] > 0
    )


bid_price_matrix = (
    RECONSTRUCTED_BOOK_STATES[
        bid_price_columns
    ].to_numpy(dtype=object)
)

ask_price_matrix = (
    RECONSTRUCTED_BOOK_STATES[
        ask_price_columns
    ].to_numpy(dtype=object)
)

duplicate_bid_price_mask = np.fromiter(
    (
        len(set(row)) != TOP_N_LEVELS
        for row in bid_price_matrix
    ),
    dtype=bool,
    count=len(
        RECONSTRUCTED_BOOK_STATES
    ),
)

duplicate_ask_price_mask = np.fromiter(
    (
        len(set(row)) != TOP_N_LEVELS
        for row in ask_price_matrix
    ),
    dtype=bool,
    count=len(
        RECONSTRUCTED_BOOK_STATES
    ),
)


top_level_imbalance_float = finite_float_series(
    RECONSTRUCTED_BOOK_STATES[
        "top_level_imbalance"
    ],
    field_name="top_level_imbalance",
)

multi_level_imbalance_float = finite_float_series(
    RECONSTRUCTED_BOOK_STATES[
        "multi_level_imbalance"
    ],
    field_name="multi_level_imbalance",
)

microprice_float = finite_float_series(
    RECONSTRUCTED_BOOK_STATES[
        "microprice"
    ],
    field_name="microprice",
)

best_bid_float = finite_float_series(
    RECONSTRUCTED_BOOK_STATES[
        "best_bid"
    ],
    field_name="best_bid",
)

best_ask_float = finite_float_series(
    RECONSTRUCTED_BOOK_STATES[
        "best_ask"
    ],
    field_name="best_ask",
)

bid_depth_float = finite_float_series(
    RECONSTRUCTED_BOOK_STATES[
        "visible_bid_depth_top_10"
    ],
    field_name="visible_bid_depth_top_10",
)

ask_depth_float = finite_float_series(
    RECONSTRUCTED_BOOK_STATES[
        "visible_ask_depth_top_10"
    ],
    field_name="visible_ask_depth_top_10",
)

total_depth_float = finite_float_series(
    RECONSTRUCTED_BOOK_STATES[
        "visible_total_depth_top_10"
    ],
    field_name="visible_total_depth_top_10",
)


imbalance_tolerance = 1e-12

top_imbalance_out_of_bounds = (
    top_level_imbalance_float.abs()
    > 1.0 + imbalance_tolerance
)

multi_imbalance_out_of_bounds = (
    multi_level_imbalance_float.abs()
    > 1.0 + imbalance_tolerance
)

microprice_outside_spread = (
    (microprice_float < best_bid_float)
    | (microprice_float > best_ask_float)
)

nonpositive_visible_depth = (
    (bid_depth_float <= 0)
    | (ask_depth_float <= 0)
    | (total_depth_float <= 0)
)

retained_depth_count_failure = (
    RECONSTRUCTED_BOOK_STATES[
        "retained_bid_level_count"
    ].ne(TOP_N_LEVELS)
    | RECONSTRUCTED_BOOK_STATES[
        "retained_ask_level_count"
    ].ne(TOP_N_LEVELS)
)

full_depth_insufficient = (
    RECONSTRUCTED_BOOK_STATES[
        "full_bid_level_count"
    ].lt(TOP_N_LEVELS)
    | RECONSTRUCTED_BOOK_STATES[
        "full_ask_level_count"
    ].lt(TOP_N_LEVELS)
)

invalid_continuity_status = ~(
    RECONSTRUCTED_BOOK_STATES[
        "continuity_status"
    ].isin(
        [
            "SNAPSHOT_BRIDGE",
            "EXACT_SUCCESSOR",
        ]
    )
)


# ---------------------------------------------------------------------------
# Authoritative book-quality gates
# ---------------------------------------------------------------------------

BOOK_QUALITY_GATES = pd.DataFrame(
    [
        {
            "gate": "accepted_state_count",
            "observed": int(
                RECONSTRUCTED_BOOK_STATES[
                    "state_accepted"
                ].sum()
            ),
            "expected": len(
                RECONSTRUCTED_BOOK_STATES
            ),
            "passed": bool(
                RECONSTRUCTED_BOOK_STATES[
                    "state_accepted"
                ].all()
            ),
            "critical": True,
        },
        {
            "gate": "locked_state_count",
            "observed": int(
                RECONSTRUCTED_BOOK_STATES[
                    "locked_state"
                ].sum()
            ),
            "expected": 0,
            "passed": not bool(
                RECONSTRUCTED_BOOK_STATES[
                    "locked_state"
                ].any()
            ),
            "critical": True,
        },
        {
            "gate": "crossed_state_count",
            "observed": int(
                RECONSTRUCTED_BOOK_STATES[
                    "crossed_state"
                ].sum()
            ),
            "expected": 0,
            "passed": not bool(
                RECONSTRUCTED_BOOK_STATES[
                    "crossed_state"
                ].any()
            ),
            "critical": True,
        },
        {
            "gate": "incomplete_state_count",
            "observed": int(
                RECONSTRUCTED_BOOK_STATES[
                    "incomplete_state"
                ].sum()
            ),
            "expected": 0,
            "passed": not bool(
                RECONSTRUCTED_BOOK_STATES[
                    "incomplete_state"
                ].any()
            ),
            "critical": True,
        },
        {
            "gate": "bid_order_failure_count",
            "observed": int(
                bid_order_failure_mask.sum()
            ),
            "expected": 0,
            "passed": not bool(
                bid_order_failure_mask.any()
            ),
            "critical": True,
        },
        {
            "gate": "ask_order_failure_count",
            "observed": int(
                ask_order_failure_mask.sum()
            ),
            "expected": 0,
            "passed": not bool(
                ask_order_failure_mask.any()
            ),
            "critical": True,
        },
        {
            "gate": "duplicate_bid_price_state_count",
            "observed": int(
                duplicate_bid_price_mask.sum()
            ),
            "expected": 0,
            "passed": not bool(
                duplicate_bid_price_mask.any()
            ),
            "critical": True,
        },
        {
            "gate": "duplicate_ask_price_state_count",
            "observed": int(
                duplicate_ask_price_mask.sum()
            ),
            "expected": 0,
            "passed": not bool(
                duplicate_ask_price_mask.any()
            ),
            "critical": True,
        },
        {
            "gate": "nonpositive_bid_quantity_state_count",
            "observed": int(
                bid_quantity_failure_mask.sum()
            ),
            "expected": 0,
            "passed": not bool(
                bid_quantity_failure_mask.any()
            ),
            "critical": True,
        },
        {
            "gate": "nonpositive_ask_quantity_state_count",
            "observed": int(
                ask_quantity_failure_mask.sum()
            ),
            "expected": 0,
            "passed": not bool(
                ask_quantity_failure_mask.any()
            ),
            "critical": True,
        },
        {
            "gate": "retained_depth_count_failure",
            "observed": int(
                retained_depth_count_failure.sum()
            ),
            "expected": 0,
            "passed": not bool(
                retained_depth_count_failure.any()
            ),
            "critical": True,
        },
        {
            "gate": "full_depth_insufficient_count",
            "observed": int(
                full_depth_insufficient.sum()
            ),
            "expected": 0,
            "passed": not bool(
                full_depth_insufficient.any()
            ),
            "critical": True,
        },
        {
            "gate": "nonpositive_visible_depth_count",
            "observed": int(
                nonpositive_visible_depth.sum()
            ),
            "expected": 0,
            "passed": not bool(
                nonpositive_visible_depth.any()
            ),
            "critical": True,
        },
        {
            "gate": "microprice_outside_spread_count",
            "observed": int(
                microprice_outside_spread.sum()
            ),
            "expected": 0,
            "passed": not bool(
                microprice_outside_spread.any()
            ),
            "critical": True,
        },
        {
            "gate": "top_imbalance_out_of_bounds_count",
            "observed": int(
                top_imbalance_out_of_bounds.sum()
            ),
            "expected": 0,
            "passed": not bool(
                top_imbalance_out_of_bounds.any()
            ),
            "critical": True,
        },
        {
            "gate": "multi_imbalance_out_of_bounds_count",
            "observed": int(
                multi_imbalance_out_of_bounds.sum()
            ),
            "expected": 0,
            "passed": not bool(
                multi_imbalance_out_of_bounds.any()
            ),
            "critical": True,
        },
        {
            "gate": "off_grid_unique_price_count",
            "observed": len(
                off_grid_prices
            ),
            "expected": 0,
            "passed": not off_grid_prices,
            "critical": True,
        },
        {
            "gate": "off_grid_spread_count",
            "observed": len(
                off_grid_spreads
            ),
            "expected": 0,
            "passed": not off_grid_spreads,
            "critical": True,
        },
        {
            "gate": "invalid_continuity_status_count",
            "observed": int(
                invalid_continuity_status.sum()
            ),
            "expected": 0,
            "passed": not bool(
                invalid_continuity_status.any()
            ),
            "critical": True,
        },
        {
            "gate": "negative_local_interval_count",
            "observed": int(
                negative_interval_mask.sum()
            ),
            "expected": 0,
            "passed": not bool(
                negative_interval_mask.any()
            ),
            "critical": True,
        },
        {
            "gate": "partition_reconciliation_failures",
            "observed": int(
                (
                    ~RECONSTRUCTION_PARTITION_SUMMARY[
                        "partition_passed"
                    ]
                ).sum()
            ),
            "expected": 0,
            "passed": bool(
                RECONSTRUCTION_PARTITION_SUMMARY[
                    "partition_passed"
                ].all()
            ),
            "critical": True,
        },
        {
            "gate": "stale_interval_count_recorded",
            "observed": int(
                RECONSTRUCTED_BOOK_STATES[
                    "stale_interval"
                ].sum()
            ),
            "expected": "DIAGNOSTIC",
            "passed": True,
            "critical": False,
        },
    ]
)

failed_critical_book_gates = (
    BOOK_QUALITY_GATES.loc[
        BOOK_QUALITY_GATES[
            "critical"
        ]
        & ~BOOK_QUALITY_GATES[
            "passed"
        ]
    ]
)

if not failed_critical_book_gates.empty:
    raise RuntimeError(
        "Critical reconstructed-book quality gates failed:\n"
        + failed_critical_book_gates.to_string(
            index=False
        )
    )


# ---------------------------------------------------------------------------
# Summary working table
# ---------------------------------------------------------------------------

BOOK_QUALITY_WORKING = pd.DataFrame(
    {
        "book_state_id": (
            RECONSTRUCTED_BOOK_STATES[
                "book_state_id"
            ].astype("int64")
        ),
        "partition": (
            RECONSTRUCTED_BOOK_STATES[
                "partition"
            ]
        ),
        "local_interval_ns": (
            local_interval_ns
        ),
        "spread_ticks": (
            RECONSTRUCTED_BOOK_STATES[
                "spread_ticks"
            ].astype("Int64")
        ),
        "spread_basis_points": (
            RECONSTRUCTED_BOOK_STATES[
                "spread_basis_points"
            ].astype("float64")
        ),
        "bid_depth_top_10": (
            bid_depth_float
        ),
        "ask_depth_top_10": (
            ask_depth_float
        ),
        "total_depth_top_10": (
            total_depth_float
        ),
        "top_level_imbalance": (
            top_level_imbalance_float
        ),
        "multi_level_imbalance": (
            multi_level_imbalance_float
        ),
        "retained_bid_level_count": (
            RECONSTRUCTED_BOOK_STATES[
                "retained_bid_level_count"
            ].astype("int64")
        ),
        "retained_ask_level_count": (
            RECONSTRUCTED_BOOK_STATES[
                "retained_ask_level_count"
            ].astype("int64")
        ),
        "full_bid_level_count": (
            RECONSTRUCTED_BOOK_STATES[
                "full_bid_level_count"
            ].astype("int64")
        ),
        "full_ask_level_count": (
            RECONSTRUCTED_BOOK_STATES[
                "full_ask_level_count"
            ].astype("int64")
        ),
        "top_of_book_moved": (
            RECONSTRUCTED_BOOK_STATES[
                "top_of_book_moved"
            ].astype(bool)
        ),
        "top_10_changed": (
            RECONSTRUCTED_BOOK_STATES[
                "top_10_changed"
            ].astype(bool)
        ),
        "movement_code": (
            RECONSTRUCTED_BOOK_STATES[
                "movement_code"
            ].astype(str)
        ),
        "locked_state": (
            RECONSTRUCTED_BOOK_STATES[
                "locked_state"
            ].astype(bool)
        ),
        "crossed_state": (
            RECONSTRUCTED_BOOK_STATES[
                "crossed_state"
            ].astype(bool)
        ),
        "incomplete_state": (
            RECONSTRUCTED_BOOK_STATES[
                "incomplete_state"
            ].astype(bool)
        ),
        "stale_interval": (
            RECONSTRUCTED_BOOK_STATES[
                "stale_interval"
            ].astype(bool)
        ),
    }
)


# ---------------------------------------------------------------------------
# Book-quality summary by frozen partition
# ---------------------------------------------------------------------------

book_quality_summary_rows: list[dict[str, Any]] = []

for scope_name, scope_frame in iter_partition_scopes(
    BOOK_QUALITY_WORKING
):
    state_count = len(scope_frame)

    book_quality_summary_rows.append(
        {
            "scope": scope_name,
            "state_count": state_count,
            "locked_state_count": int(
                scope_frame[
                    "locked_state"
                ].sum()
            ),
            "crossed_state_count": int(
                scope_frame[
                    "crossed_state"
                ].sum()
            ),
            "incomplete_state_count": int(
                scope_frame[
                    "incomplete_state"
                ].sum()
            ),
            "stale_interval_count": int(
                scope_frame[
                    "stale_interval"
                ].sum()
            ),
            "stale_interval_fraction": float(
                scope_frame[
                    "stale_interval"
                ].mean()
            ),
            "top_of_book_move_count": int(
                scope_frame[
                    "top_of_book_moved"
                ].sum()
            ),
            "top_of_book_move_fraction": float(
                scope_frame[
                    "top_of_book_moved"
                ].mean()
            ),
            "top_10_change_count": int(
                scope_frame[
                    "top_10_changed"
                ].sum()
            ),
            "top_10_change_fraction": float(
                scope_frame[
                    "top_10_changed"
                ].mean()
            ),
            "no_visible_top_10_change_count": int(
                (
                    ~scope_frame[
                        "top_10_changed"
                    ]
                ).sum()
            ),
            "minimum_spread_ticks": int(
                scope_frame[
                    "spread_ticks"
                ].min()
            ),
            "median_spread_ticks": float(
                scope_frame[
                    "spread_ticks"
                ].median()
            ),
            "maximum_spread_ticks": int(
                scope_frame[
                    "spread_ticks"
                ].max()
            ),
            "one_tick_spread_fraction": float(
                (
                    scope_frame[
                        "spread_ticks"
                    ] == 1
                ).mean()
            ),
            "wide_spread_fraction": float(
                (
                    scope_frame[
                        "spread_ticks"
                    ] > 1
                ).mean()
            ),
            "minimum_spread_bps": float(
                scope_frame[
                    "spread_basis_points"
                ].min()
            ),
            "median_spread_bps": float(
                scope_frame[
                    "spread_basis_points"
                ].median()
            ),
            "maximum_spread_bps": float(
                scope_frame[
                    "spread_basis_points"
                ].max()
            ),
        }
    )

BOOK_QUALITY_SUMMARY = pd.DataFrame(
    book_quality_summary_rows
)


# ---------------------------------------------------------------------------
# Spread distribution
# ---------------------------------------------------------------------------

spread_distribution_rows: list[dict[str, Any]] = []

for scope_name, scope_frame in iter_partition_scopes(
    BOOK_QUALITY_WORKING
):
    spread_counts = (
        scope_frame[
            "spread_ticks"
        ]
        .value_counts()
        .sort_index()
    )

    for spread_ticks, state_count in (
        spread_counts.items()
    ):
        spread_distribution_rows.append(
            {
                "scope": scope_name,
                "spread_ticks": int(
                    spread_ticks
                ),
                "spread_price": str(
                    OBSERVED_PRICE_GRID
                    * int(spread_ticks)
                ),
                "state_count": int(
                    state_count
                ),
                "state_fraction": float(
                    state_count
                    / len(scope_frame)
                ),
            }
        )

SPREAD_DISTRIBUTION = pd.DataFrame(
    spread_distribution_rows
)


# ---------------------------------------------------------------------------
# Depth-completeness summary
# ---------------------------------------------------------------------------

depth_completeness_rows: list[dict[str, Any]] = []

for scope_name, scope_frame in iter_partition_scopes(
    BOOK_QUALITY_WORKING
):
    depth_completeness_rows.append(
        {
            "scope": scope_name,
            "state_count": len(
                scope_frame
            ),
            "minimum_retained_bid_levels": int(
                scope_frame[
                    "retained_bid_level_count"
                ].min()
            ),
            "maximum_retained_bid_levels": int(
                scope_frame[
                    "retained_bid_level_count"
                ].max()
            ),
            "minimum_retained_ask_levels": int(
                scope_frame[
                    "retained_ask_level_count"
                ].min()
            ),
            "maximum_retained_ask_levels": int(
                scope_frame[
                    "retained_ask_level_count"
                ].max()
            ),
            "minimum_full_bid_levels": int(
                scope_frame[
                    "full_bid_level_count"
                ].min()
            ),
            "maximum_full_bid_levels": int(
                scope_frame[
                    "full_bid_level_count"
                ].max()
            ),
            "minimum_full_ask_levels": int(
                scope_frame[
                    "full_ask_level_count"
                ].min()
            ),
            "maximum_full_ask_levels": int(
                scope_frame[
                    "full_ask_level_count"
                ].max()
            ),
            "complete_top_10_state_count": int(
                (
                    scope_frame[
                        "retained_bid_level_count"
                    ].eq(TOP_N_LEVELS)
                    & scope_frame[
                        "retained_ask_level_count"
                    ].eq(TOP_N_LEVELS)
                ).sum()
            ),
            "complete_top_10_fraction": float(
                (
                    scope_frame[
                        "retained_bid_level_count"
                    ].eq(TOP_N_LEVELS)
                    & scope_frame[
                        "retained_ask_level_count"
                    ].eq(TOP_N_LEVELS)
                ).mean()
            ),
            "minimum_bid_depth_top_10": float(
                scope_frame[
                    "bid_depth_top_10"
                ].min()
            ),
            "median_bid_depth_top_10": float(
                scope_frame[
                    "bid_depth_top_10"
                ].median()
            ),
            "mean_bid_depth_top_10": float(
                scope_frame[
                    "bid_depth_top_10"
                ].mean()
            ),
            "maximum_bid_depth_top_10": float(
                scope_frame[
                    "bid_depth_top_10"
                ].max()
            ),
            "minimum_ask_depth_top_10": float(
                scope_frame[
                    "ask_depth_top_10"
                ].min()
            ),
            "median_ask_depth_top_10": float(
                scope_frame[
                    "ask_depth_top_10"
                ].median()
            ),
            "mean_ask_depth_top_10": float(
                scope_frame[
                    "ask_depth_top_10"
                ].mean()
            ),
            "maximum_ask_depth_top_10": float(
                scope_frame[
                    "ask_depth_top_10"
                ].max()
            ),
            "minimum_total_depth_top_10": float(
                scope_frame[
                    "total_depth_top_10"
                ].min()
            ),
            "median_total_depth_top_10": float(
                scope_frame[
                    "total_depth_top_10"
                ].median()
            ),
            "mean_total_depth_top_10": float(
                scope_frame[
                    "total_depth_top_10"
                ].mean()
            ),
            "maximum_total_depth_top_10": float(
                scope_frame[
                    "total_depth_top_10"
                ].max()
            ),
        }
    )

DEPTH_COMPLETENESS_SUMMARY = pd.DataFrame(
    depth_completeness_rows
)


# ---------------------------------------------------------------------------
# Local book-update interval distribution
# ---------------------------------------------------------------------------

INTERVAL_QUANTILES = (
    0.00,
    0.01,
    0.05,
    0.25,
    0.50,
    0.75,
    0.95,
    0.99,
    1.00,
)

interval_distribution_rows: list[dict[str, Any]] = []

for scope_name, scope_frame in iter_partition_scopes(
    BOOK_QUALITY_WORKING
):
    valid_intervals = (
        scope_frame[
            "local_interval_ns"
        ]
        .dropna()
        .astype("float64")
    )

    if valid_intervals.empty:
        raise RuntimeError(
            f"No valid local intervals exist in {scope_name}."
        )

    interval_quantiles = valid_intervals.quantile(
        INTERVAL_QUANTILES
    )

    for quantile, interval_value_ns in (
        interval_quantiles.items()
    ):
        interval_distribution_rows.append(
            {
                "scope": scope_name,
                "quantile": float(
                    quantile
                ),
                "interval_ns": float(
                    interval_value_ns
                ),
                "interval_ms": float(
                    interval_value_ns
                    / 1_000_000.0
                ),
                "multiple_of_nominal": float(
                    interval_value_ns
                    / NOMINAL_DEPTH_INTERVAL_NS
                ),
            }
        )

BOOK_UPDATE_INTERVAL_DISTRIBUTION = (
    pd.DataFrame(
        interval_distribution_rows
    )
)


# ---------------------------------------------------------------------------
# Visible-book movement distribution
# ---------------------------------------------------------------------------

movement_distribution_rows: list[
    dict[str, Any]
] = []

for scope_name, scope_frame in iter_partition_scopes(
    BOOK_QUALITY_WORKING
):
    movement_counts = (
        scope_frame[
            "movement_code"
        ]
        .value_counts(dropna=False)
        .sort_index()
    )

    for movement_code, state_count in (
        movement_counts.items()
    ):
        movement_distribution_rows.append(
            {
                "scope": scope_name,
                "movement_code": str(
                    movement_code
                ),
                "state_count": int(
                    state_count
                ),
                "state_fraction": float(
                    state_count
                    / len(scope_frame)
                ),
            }
        )

BOOK_MOVEMENT_DISTRIBUTION = pd.DataFrame(
    movement_distribution_rows
)


# ---------------------------------------------------------------------------
# Final diagnostics integrity
# ---------------------------------------------------------------------------

if int(
    BOOK_QUALITY_SUMMARY.loc[
        BOOK_QUALITY_SUMMARY[
            "scope"
        ].eq("ALL"),
        "state_count",
    ].iloc[0]
) != len(RECONSTRUCTED_BOOK_STATES):
    raise RuntimeError(
        "Book-quality summary row count does not reconcile."
    )

if not (
    DEPTH_COMPLETENESS_SUMMARY[
        "complete_top_10_fraction"
    ].eq(1.0)
).all():
    raise RuntimeError(
        "At least one scope does not retain complete top-10 "
        "depth on both sides."
    )

overall_spread_count = int(
    SPREAD_DISTRIBUTION.loc[
        SPREAD_DISTRIBUTION[
            "scope"
        ].eq("ALL"),
        "state_count",
    ].sum()
)

if overall_spread_count != len(
    RECONSTRUCTED_BOOK_STATES
):
    raise RuntimeError(
        "Spread-distribution counts do not reconcile with "
        "the reconstructed state table."
    )

overall_movement_count = int(
    BOOK_MOVEMENT_DISTRIBUTION.loc[
        BOOK_MOVEMENT_DISTRIBUTION[
            "scope"
        ].eq("ALL"),
        "state_count",
    ].sum()
)

if overall_movement_count != len(
    RECONSTRUCTED_BOOK_STATES
):
    raise RuntimeError(
        "Movement-distribution counts do not reconcile with "
        "the reconstructed state table."
    )


# ---------------------------------------------------------------------------
# Compact output
# ---------------------------------------------------------------------------

display(BOOK_QUALITY_GATES)

display(BOOK_QUALITY_SUMMARY)

display(SPREAD_DISTRIBUTION)

display(DEPTH_COMPLETENESS_SUMMARY)

display(BOOK_UPDATE_INTERVAL_DISTRIBUTION)

display(BOOK_MOVEMENT_DISTRIBUTION)

{
    "status": "PASS",
    "reconstructed_state_count": len(
        RECONSTRUCTED_BOOK_STATES
    ),
    "critical_book_quality_failures": len(
        failed_critical_book_gates
    ),
    "observed_price_grid": str(
        OBSERVED_PRICE_GRID
    ),
    "unique_retained_prices": len(
        unique_retained_prices
    ),
    "off_grid_unique_prices": len(
        off_grid_prices
    ),
    "depth_stream": DEPTH_STREAM_NAME,
    "nominal_depth_interval_ms": (
        NOMINAL_DEPTH_INTERVAL_MS
    ),
    "stale_interval_threshold_ms": (
        STALE_INTERVAL_THRESHOLD_NS
        / 1_000_000
    ),
    "stale_interval_count": int(
        RECONSTRUCTED_BOOK_STATES[
            "stale_interval"
        ].sum()
    ),
    "minimum_spread_ticks": int(
        RECONSTRUCTED_BOOK_STATES[
            "spread_ticks"
        ].min()
    ),
    "maximum_spread_ticks": int(
        RECONSTRUCTED_BOOK_STATES[
            "spread_ticks"
        ].max()
    ),
    "complete_top_10_states": int(
        (
            RECONSTRUCTED_BOOK_STATES[
                "retained_bid_level_count"
            ].eq(TOP_N_LEVELS)
            & RECONSTRUCTED_BOOK_STATES[
                "retained_ask_level_count"
            ].eq(TOP_N_LEVELS)
        ).sum()
    ),
    "locked_states": int(
        RECONSTRUCTED_BOOK_STATES[
            "locked_state"
        ].sum()
    ),
    "crossed_states": int(
        RECONSTRUCTED_BOOK_STATES[
            "crossed_state"
        ].sum()
    ),
    "incomplete_states": int(
        RECONSTRUCTED_BOOK_STATES[
            "incomplete_state"
        ].sum()
    ),
}

,gate,observed,expected,passed,critical
0,accepted_state_count,35985,35985,True,True
1,locked_state_count,0,0,True,True
2,crossed_state_count,0,0,True,True
3,incomplete_state_count,0,0,True,True
4,bid_order_failure_count,0,0,True,True
5,ask_order_failure_count,0,0,True,True
6,duplicate_bid_price_state_count,0,0,True,True
7,duplicate_ask_price_state_count,0,0,True,True
8,nonpositive_bid_quantity_state_count,0,0,True,True
9,nonpositive_ask_quantity_state_count,0,0,True,True


,scope,state_count,locked_state_count,crossed_state_count,incomplete_state_count,stale_interval_count,stale_interval_fraction,top_of_book_move_count,top_of_book_move_fraction,top_10_change_count,top_10_change_fraction,no_visible_top_10_change_count,minimum_spread_ticks,median_spread_ticks,maximum_spread_ticks,one_tick_spread_fraction,wide_spread_fraction,minimum_spread_bps,median_spread_bps,maximum_spread_bps
0,ALL,35985,0,0,0,0,0.0000000000,526,0.0146172016,25317,0.7035431430,10668,1,1.0000000000,339,0.9999722106,0.0000277894,0.0015622096,0.0015649451,0.5307847731
1,DEVELOPMENT,18010,0,0,0,0,0.0000000000,269,0.0149361466,12245,0.6799000555,5765,1,1.0000000000,339,0.9999444753,0.0000555247,0.0015642964,0.0015655613,0.5307847731
2,CALIBRATION,7204,0,0,0,0,0.0000000000,107,0.0148528595,5523,0.7666574125,1681,1,1.0000000000,1,1.0000000000,0.0000000000,0.0015642110,0.0015654777,0.0015664159
3,VALIDATION,5353,0,0,0,0,0.0000000000,75,0.0140108350,3545,0.6622454698,1808,1,1.0000000000,1,1.0000000000,0.0000000000,0.0015622096,0.0015630802,0.0015643328
4,ENGINEERING_HOLDOUT,5418,0,0,0,0,0.0000000000,75,0.0138427464,4004,0.7390180879,1414,1,1.0000000000,1,1.0000000000,0.0000000000,0.0015633306,0.0015638482,0.0015647458


,scope,spread_ticks,spread_price,state_count,state_fraction
0,ALL,1,0.01000000,35984,0.9999722106
1,ALL,339,3.39000000,1,0.0000277894
2,DEVELOPMENT,1,0.01000000,18009,0.9999444753
3,DEVELOPMENT,339,3.39000000,1,0.0000555247
4,CALIBRATION,1,0.01000000,7204,1.0000000000
5,VALIDATION,1,0.01000000,5353,1.0000000000
6,ENGINEERING_HOLDOUT,1,0.01000000,5418,1.0000000000


,scope,state_count,minimum_retained_bid_levels,maximum_retained_bid_levels,minimum_retained_ask_levels,maximum_retained_ask_levels,minimum_full_bid_levels,maximum_full_bid_levels,minimum_full_ask_levels,maximum_full_ask_levels,complete_top_10_state_count,complete_top_10_fraction,minimum_bid_depth_top_10,median_bid_depth_top_10,mean_bid_depth_top_10,maximum_bid_depth_top_10,minimum_ask_depth_top_10,median_ask_depth_top_10,mean_ask_depth_top_10,maximum_ask_depth_top_10,minimum_total_depth_top_10,median_total_depth_top_10,mean_total_depth_top_10,maximum_total_depth_top_10
0,ALL,35985,10,10,10,10,4960,5671,4939,5693,35985,1.0000000000,0.0045200000,2.2859900000,2.6249313342,19.5283600000,0.0029400000,2.8249100000,3.6501316190,20.6737700000,2.0240800000,5.8319100000,6.2750629532,21.4097400000
1,DEVELOPMENT,18010,10,10,10,10,4960,5326,4939,5620,18010,1.0000000000,0.0045200000,2.2118900000,2.6573366796,19.5283600000,0.0029400000,2.8950100000,3.6661568906,19.1277000000,2.0240800000,5.8744100000,6.3234935702,19.7453200000
2,CALIBRATION,7204,10,10,10,10,5219,5484,5384,5604,7204,1.0000000000,0.0081600000,2.3962350000,2.7507947460,11.5316500000,0.0057300000,2.3864200000,2.9057542088,20.6667400000,2.5569700000,5.2775000000,5.6565489547,21.4097400000
3,VALIDATION,5353,10,10,10,10,5425,5614,5339,5545,5353,1.0000000000,0.0051600000,2.9077800000,3.0142803082,10.2418000000,0.0228100000,2.4409800000,3.2187737755,16.9010600000,2.8868300000,5.8950700000,6.2330540837,16.9601300000
4,ENGINEERING_HOLDOUT,5418,10,10,10,10,5522,5671,5439,5693,5418,1.0000000000,0.0142900000,1.5669200000,1.9651813621,14.1513400000,0.0234100000,3.7737800000,5.0128001790,20.6737700000,3.3052300000,6.2853000000,6.9779815412,20.7888400000


,scope,quantile,interval_ns,interval_ms,multiple_of_nominal
0,ALL,0.0000000000,"83,811,300.0000000000",83.8113000000,0.8381130000
1,ALL,0.0100000000,"92,964,026.0000000000",92.9640260000,0.9296402600
2,ALL,0.0500000000,"97,830,395.0000000000",97.8303950000,0.9783039500
3,ALL,0.2500000000,"99,608,600.0000000000",99.6086000000,0.9960860000
4,ALL,0.5000000000,"100,005,000.0000000000",100.0050000000,1.0000500000
5,ALL,0.7500000000,"100,392,900.0000000000",100.3929000000,1.0039290000
6,ALL,0.9500000000,"102,111,025.0000000000",102.1110250000,1.0211102500
7,ALL,0.9900000000,"107,021,127.0000000000",107.0211270000,1.0702112700
8,ALL,1.0000000000,"116,107,700.0000000000",116.1077000000,1.1610770000
9,DEVELOPMENT,0.0000000000,"83,811,300.0000000000",83.8113000000,0.8381130000


,scope,movement_code,state_count,state_fraction
0,ALL,DEPTH_CHANGE_ONLY,24791,0.6889259414
1,ALL,INITIAL_APPLIED_STATE,1,0.0000277894
2,ALL,MIDPOINT_DOWN,280,0.0077810199
3,ALL,MIDPOINT_UP,245,0.0068083924
4,ALL,NO_VISIBLE_TOP_10_CHANGE,10668,0.2964568570
5,DEVELOPMENT,DEPTH_CHANGE_ONLY,11976,0.6649639089
6,DEVELOPMENT,INITIAL_APPLIED_STATE,1,0.0000555247
7,DEVELOPMENT,MIDPOINT_DOWN,142,0.0078845086
8,DEVELOPMENT,MIDPOINT_UP,126,0.0069961133
9,DEVELOPMENT,NO_VISIBLE_TOP_10_CHANGE,5765,0.3200999445


{'status': 'PASS',
 'reconstructed_state_count': 35985,
 'critical_book_quality_failures': 0,
 'observed_price_grid': '0.01000000',
 'unique_retained_prices': 8783,
 'off_grid_unique_prices': 0,
 'depth_stream': 'btcusdt@depth@100ms',
 'nominal_depth_interval_ms': 100,
 'stale_interval_threshold_ms': 1000.0,
 'stale_interval_count': 0,
 'minimum_spread_ticks': 1,
 'maximum_spread_ticks': 339,
 'complete_top_10_states': 35985,
 'locked_states': 0,
 'crossed_states': 0,
 'incomplete_states': 0}

In [9]:
# ---------------------------------------------------------------------------
# Book-quality, spread, depth-completeness, and interval diagnostics
# ---------------------------------------------------------------------------

import re
from collections.abc import Iterator, Sequence
from decimal import Decimal, InvalidOperation
from typing import Any


# ---------------------------------------------------------------------------
# Required upstream objects
# ---------------------------------------------------------------------------

REQUIRED_UPSTREAM_OBJECTS = (
    "RECONSTRUCTED_BOOK_STATES",
    "TOP_10_VISIBLE_BOOK_WIDE",
    "UPDATE_CONTINUITY_REPORT",
    "RECONSTRUCTION_SESSION_LEDGER",
    "DEPTH_EVENTS",
    "RECONSTRUCTION_PARTITION_SUMMARY",
    "FROZEN_PARTITION_ORDER",
    "TOP_N_LEVELS",
)

missing_upstream_objects = [
    object_name
    for object_name in REQUIRED_UPSTREAM_OBJECTS
    if object_name not in globals()
]

if missing_upstream_objects:
    raise NameError(
        "Missing required upstream objects: "
        + ", ".join(missing_upstream_objects)
    )


# ---------------------------------------------------------------------------
# Cell-local helpers
# ---------------------------------------------------------------------------

def require_columns(
    frame: pd.DataFrame,
    required_columns: Sequence[str],
    *,
    frame_label: str,
) -> None:
    """Require every named column in a DataFrame."""
    missing_columns = sorted(
        set(required_columns) - set(frame.columns)
    )

    if missing_columns:
        raise KeyError(
            f"{frame_label} is missing required columns: "
            + ", ".join(missing_columns)
        )


def to_decimal(
    value: Any,
    *,
    field_name: str,
) -> Decimal:
    """Convert one finite numeric value to Decimal."""
    if isinstance(value, Decimal):
        result = value
    else:
        try:
            result = Decimal(str(value))
        except (
            InvalidOperation,
            TypeError,
            ValueError,
        ) as exc:
            raise ValueError(
                f"{field_name} cannot be represented as Decimal: "
                f"{value!r}"
            ) from exc

    if not result.is_finite():
        raise ValueError(
            f"{field_name} must be finite; observed {result!r}."
        )

    return result


def is_exact_increment_multiple(
    value: Any,
    increment: Decimal,
    *,
    field_name: str,
) -> bool:
    """Return True when value is an exact integer multiple."""
    numeric_value = to_decimal(
        value,
        field_name=field_name,
    )

    quotient = numeric_value / increment

    return quotient == quotient.to_integral_value()


def increment_count(
    value: Any,
    increment: Decimal,
    *,
    field_name: str,
) -> int:
    """Convert an exact increment multiple to an integer count."""
    numeric_value = to_decimal(
        value,
        field_name=field_name,
    )

    quotient = numeric_value / increment
    integral_quotient = quotient.to_integral_value()

    if quotient != integral_quotient:
        raise ValueError(
            f"{field_name}={numeric_value} is not an exact "
            f"multiple of {increment}."
        )

    return int(integral_quotient)


def finite_float_series(
    series: pd.Series,
    *,
    field_name: str,
) -> pd.Series:
    """Convert a complete numeric series to finite float64."""
    converted = pd.to_numeric(
        series,
        errors="coerce",
    ).astype("float64")

    invalid_mask = ~np.isfinite(
        converted.to_numpy()
    )

    if invalid_mask.any():
        raise RuntimeError(
            f"{field_name} contains "
            f"{int(invalid_mask.sum()):,} non-finite values."
        )

    return converted


def iter_partition_scopes(
    frame: pd.DataFrame,
) -> Iterator[tuple[str, pd.DataFrame]]:
    """Yield the full sample and each frozen partition."""
    yield "ALL", frame

    for partition_name in FROZEN_PARTITION_ORDER:
        partition_frame = frame.loc[
            frame["partition"].eq(partition_name)
        ]

        if partition_frame.empty:
            raise RuntimeError(
                f"No reconstructed states exist in "
                f"{partition_name}."
            )

        yield partition_name, partition_frame


# ---------------------------------------------------------------------------
# Required reconstructed-state schema
# ---------------------------------------------------------------------------

bid_price_columns = [
    f"bid_price_{level_number}"
    for level_number in range(
        1,
        TOP_N_LEVELS + 1,
    )
]

ask_price_columns = [
    f"ask_price_{level_number}"
    for level_number in range(
        1,
        TOP_N_LEVELS + 1,
    )
]

bid_quantity_columns = [
    f"bid_quantity_{level_number}"
    for level_number in range(
        1,
        TOP_N_LEVELS + 1,
    )
]

ask_quantity_columns = [
    f"ask_quantity_{level_number}"
    for level_number in range(
        1,
        TOP_N_LEVELS + 1,
    )
]

required_state_columns = [
    "book_state_id",
    "collector_sequence",
    "local_receipt_time_ns",
    "local_interval_ns",
    "first_update_id",
    "final_update_id",
    "continuity_status",
    "partition",
    "state_accepted",
    "retained_bid_level_count",
    "retained_ask_level_count",
    "full_bid_level_count",
    "full_ask_level_count",
    "best_bid",
    "best_ask",
    "spread",
    "midpoint",
    "microprice",
    "top_level_imbalance",
    "multi_level_imbalance",
    "visible_bid_depth_top_10",
    "visible_ask_depth_top_10",
    "visible_total_depth_top_10",
    "top_of_book_moved",
    "top_10_changed",
    "movement_code",
    "locked_state",
    "crossed_state",
    "incomplete_state",
    "top_10_state_sha256",
    *bid_price_columns,
    *ask_price_columns,
    *bid_quantity_columns,
    *ask_quantity_columns,
]

require_columns(
    RECONSTRUCTED_BOOK_STATES,
    required_state_columns,
    frame_label="Reconstructed book states",
)


# ---------------------------------------------------------------------------
# Infer the observed price grid
# ---------------------------------------------------------------------------

positive_price_gaps: list[Decimal] = []

for spread_value in (
    RECONSTRUCTED_BOOK_STATES["spread"]
    .dropna()
    .drop_duplicates()
):
    decimal_spread = to_decimal(
        spread_value,
        field_name="spread",
    )

    if decimal_spread > 0:
        positive_price_gaps.append(decimal_spread)


for level_number in range(
    1,
    TOP_N_LEVELS,
):
    next_level_number = level_number + 1

    bid_gaps = (
        RECONSTRUCTED_BOOK_STATES[
            f"bid_price_{level_number}"
        ]
        - RECONSTRUCTED_BOOK_STATES[
            f"bid_price_{next_level_number}"
        ]
    )

    ask_gaps = (
        RECONSTRUCTED_BOOK_STATES[
            f"ask_price_{next_level_number}"
        ]
        - RECONSTRUCTED_BOOK_STATES[
            f"ask_price_{level_number}"
        ]
    )

    for gap_value in bid_gaps.dropna().drop_duplicates():
        decimal_gap = to_decimal(
            gap_value,
            field_name=(
                f"bid_gap_{level_number}_"
                f"{next_level_number}"
            ),
        )

        if decimal_gap > 0:
            positive_price_gaps.append(decimal_gap)

    for gap_value in ask_gaps.dropna().drop_duplicates():
        decimal_gap = to_decimal(
            gap_value,
            field_name=(
                f"ask_gap_{level_number}_"
                f"{next_level_number}"
            ),
        )

        if decimal_gap > 0:
            positive_price_gaps.append(decimal_gap)


if not positive_price_gaps:
    raise RuntimeError(
        "No positive price increment could be inferred."
    )

OBSERVED_PRICE_GRID = min(positive_price_gaps)

if OBSERVED_PRICE_GRID <= 0:
    raise RuntimeError(
        "The inferred price grid is not positive."
    )


# ---------------------------------------------------------------------------
# Validate retained prices and spreads against the inferred grid
# ---------------------------------------------------------------------------

unique_retained_prices: set[Decimal] = set()

for price_column in (
    bid_price_columns + ask_price_columns
):
    for price_value in (
        RECONSTRUCTED_BOOK_STATES[
            price_column
        ]
        .dropna()
        .drop_duplicates()
    ):
        unique_retained_prices.add(
            to_decimal(
                price_value,
                field_name=price_column,
            )
        )


off_grid_prices = sorted(
    price
    for price in unique_retained_prices
    if not is_exact_increment_multiple(
        price,
        OBSERVED_PRICE_GRID,
        field_name="retained_price",
    )
)

unique_spreads = {
    to_decimal(
        spread_value,
        field_name="spread",
    )
    for spread_value in (
        RECONSTRUCTED_BOOK_STATES[
            "spread"
        ]
        .dropna()
        .drop_duplicates()
    )
}

off_grid_spreads = sorted(
    spread_value
    for spread_value in unique_spreads
    if not is_exact_increment_multiple(
        spread_value,
        OBSERVED_PRICE_GRID,
        field_name="spread",
    )
)

if off_grid_prices:
    raise RuntimeError(
        "Retained prices exist outside the inferred price grid. "
        f"First examples: {off_grid_prices[:10]}"
    )

if off_grid_spreads:
    raise RuntimeError(
        "Retained spreads exist outside the inferred price grid. "
        f"First examples: {off_grid_spreads[:10]}"
    )


# ---------------------------------------------------------------------------
# Add spread diagnostics
# ---------------------------------------------------------------------------

RECONSTRUCTED_BOOK_STATES = (
    RECONSTRUCTED_BOOK_STATES.copy()
)

RECONSTRUCTED_BOOK_STATES[
    "spread_ticks"
] = pd.Series(
    [
        increment_count(
            spread_value,
            OBSERVED_PRICE_GRID,
            field_name="spread",
        )
        for spread_value in (
            RECONSTRUCTED_BOOK_STATES[
                "spread"
            ]
        )
    ],
    index=RECONSTRUCTED_BOOK_STATES.index,
    dtype="Int64",
)

if (
    RECONSTRUCTED_BOOK_STATES[
        "spread_ticks"
    ] < 1
).any():
    raise RuntimeError(
        "At least one reconstructed spread is below one "
        "observed price-grid increment."
    )


midpoint_float = finite_float_series(
    RECONSTRUCTED_BOOK_STATES["midpoint"],
    field_name="midpoint",
)

spread_float = finite_float_series(
    RECONSTRUCTED_BOOK_STATES["spread"],
    field_name="spread",
)

RECONSTRUCTED_BOOK_STATES[
    "spread_basis_points"
] = (
    spread_float
    / midpoint_float
    * 10_000.0
)


# ---------------------------------------------------------------------------
# Parse the nominal depth-stream cadence
# ---------------------------------------------------------------------------

require_columns(
    DEPTH_EVENTS,
    ["stream"],
    frame_label="Depth events",
)

stream_values = (
    DEPTH_EVENTS["stream"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .tolist()
)

if len(stream_values) != 1:
    raise RuntimeError(
        "A unique depth-stream cadence could not be "
        f"established: {stream_values}"
    )

DEPTH_STREAM_NAME = stream_values[0]

cadence_match = re.search(
    r"@(\d+)ms$",
    DEPTH_STREAM_NAME,
    flags=re.IGNORECASE,
)

if cadence_match is None:
    raise RuntimeError(
        "The differential-depth stream name does not expose "
        f"an explicit millisecond cadence: "
        f"{DEPTH_STREAM_NAME!r}"
    )

NOMINAL_DEPTH_INTERVAL_MS = int(
    cadence_match.group(1)
)

if NOMINAL_DEPTH_INTERVAL_MS <= 0:
    raise RuntimeError(
        "The parsed depth-stream cadence is not positive."
    )

NOMINAL_DEPTH_INTERVAL_NS = (
    NOMINAL_DEPTH_INTERVAL_MS * 1_000_000
)

STALE_INTERVAL_MULTIPLIER = 10

STALE_INTERVAL_THRESHOLD_NS = max(
    NOMINAL_DEPTH_INTERVAL_NS
    * STALE_INTERVAL_MULTIPLIER,
    1_000_000_000,
)


# ---------------------------------------------------------------------------
# Derive interval diagnostics
# ---------------------------------------------------------------------------

local_interval_ns = pd.to_numeric(
    RECONSTRUCTED_BOOK_STATES[
        "local_interval_ns"
    ],
    errors="coerce",
).astype("Float64")

negative_interval_mask = (
    local_interval_ns.lt(0)
    .fillna(False)
)

if negative_interval_mask.any():
    raise RuntimeError(
        "At least one reconstructed local interval is negative."
    )

RECONSTRUCTED_BOOK_STATES[
    "interval_multiple_of_nominal"
] = (
    local_interval_ns
    / float(NOMINAL_DEPTH_INTERVAL_NS)
)

RECONSTRUCTED_BOOK_STATES[
    "stale_interval"
] = (
    local_interval_ns
    .gt(STALE_INTERVAL_THRESHOLD_NS)
    .fillna(False)
    .astype(bool)
)

RECONSTRUCTED_BOOK_STATES[
    "stale_interval_threshold_ns"
] = STALE_INTERVAL_THRESHOLD_NS


# ---------------------------------------------------------------------------
# Propagate diagnostics to aligned outputs
# ---------------------------------------------------------------------------

TOP_10_VISIBLE_BOOK_WIDE = (
    TOP_10_VISIBLE_BOOK_WIDE.copy()
)

UPDATE_CONTINUITY_REPORT = (
    UPDATE_CONTINUITY_REPORT.copy()
)

for target_label, target_frame in (
    (
        "Top-10 visible-book table",
        TOP_10_VISIBLE_BOOK_WIDE,
    ),
    (
        "Update-continuity report",
        UPDATE_CONTINUITY_REPORT,
    ),
):
    require_columns(
        target_frame,
        [
            "book_state_id",
            "collector_sequence",
        ],
        frame_label=target_label,
    )

    if len(target_frame) != len(
        RECONSTRUCTED_BOOK_STATES
    ):
        raise RuntimeError(
            f"{target_label} row count does not match the "
            "reconstructed-state table."
        )

    if not np.array_equal(
        target_frame[
            "book_state_id"
        ].astype("int64").to_numpy(),
        RECONSTRUCTED_BOOK_STATES[
            "book_state_id"
        ].astype("int64").to_numpy(),
    ):
        raise RuntimeError(
            f"{target_label} book-state identity does not "
            "match the reconstructed-state table."
        )

    if not np.array_equal(
        target_frame[
            "collector_sequence"
        ].astype("int64").to_numpy(),
        RECONSTRUCTED_BOOK_STATES[
            "collector_sequence"
        ].astype("int64").to_numpy(),
    ):
        raise RuntimeError(
            f"{target_label} collector ordering does not "
            "match the reconstructed-state table."
        )

    target_frame["spread_ticks"] = (
        RECONSTRUCTED_BOOK_STATES[
            "spread_ticks"
        ].to_numpy()
    )

    target_frame["spread_basis_points"] = (
        RECONSTRUCTED_BOOK_STATES[
            "spread_basis_points"
        ].to_numpy()
    )

    target_frame[
        "interval_multiple_of_nominal"
    ] = (
        RECONSTRUCTED_BOOK_STATES[
            "interval_multiple_of_nominal"
        ].to_numpy()
    )

    target_frame["stale_interval"] = (
        RECONSTRUCTED_BOOK_STATES[
            "stale_interval"
        ].to_numpy()
    )


RECONSTRUCTION_SESSION_LEDGER = (
    RECONSTRUCTION_SESSION_LEDGER.copy()
)

RECONSTRUCTION_SESSION_LEDGER[
    "depth_stream"
] = DEPTH_STREAM_NAME

RECONSTRUCTION_SESSION_LEDGER[
    "nominal_depth_interval_ms"
] = NOMINAL_DEPTH_INTERVAL_MS

RECONSTRUCTION_SESSION_LEDGER[
    "stale_interval_threshold_ns"
] = STALE_INTERVAL_THRESHOLD_NS

RECONSTRUCTION_SESSION_LEDGER[
    "stale_interval_count"
] = int(
    RECONSTRUCTED_BOOK_STATES[
        "stale_interval"
    ].sum()
)


# ---------------------------------------------------------------------------
# Independently recompute structural failure masks
# ---------------------------------------------------------------------------

bid_order_failure_mask = pd.Series(
    False,
    index=RECONSTRUCTED_BOOK_STATES.index,
)

ask_order_failure_mask = pd.Series(
    False,
    index=RECONSTRUCTED_BOOK_STATES.index,
)

for level_number in range(
    1,
    TOP_N_LEVELS,
):
    next_level_number = level_number + 1

    bid_order_failure_mask |= ~(
        RECONSTRUCTED_BOOK_STATES[
            f"bid_price_{level_number}"
        ]
        > RECONSTRUCTED_BOOK_STATES[
            f"bid_price_{next_level_number}"
        ]
    )

    ask_order_failure_mask |= ~(
        RECONSTRUCTED_BOOK_STATES[
            f"ask_price_{level_number}"
        ]
        < RECONSTRUCTED_BOOK_STATES[
            f"ask_price_{next_level_number}"
        ]
    )


bid_quantity_failure_mask = pd.Series(
    False,
    index=RECONSTRUCTED_BOOK_STATES.index,
)

ask_quantity_failure_mask = pd.Series(
    False,
    index=RECONSTRUCTED_BOOK_STATES.index,
)

for quantity_column in bid_quantity_columns:
    bid_quantity_failure_mask |= ~(
        RECONSTRUCTED_BOOK_STATES[
            quantity_column
        ] > 0
    )

for quantity_column in ask_quantity_columns:
    ask_quantity_failure_mask |= ~(
        RECONSTRUCTED_BOOK_STATES[
            quantity_column
        ] > 0
    )


duplicate_bid_price_mask = (
    RECONSTRUCTED_BOOK_STATES[
        bid_price_columns
    ]
    .apply(
        lambda row: row.nunique(
            dropna=False
        ) != TOP_N_LEVELS,
        axis=1,
    )
)

duplicate_ask_price_mask = (
    RECONSTRUCTED_BOOK_STATES[
        ask_price_columns
    ]
    .apply(
        lambda row: row.nunique(
            dropna=False
        ) != TOP_N_LEVELS,
        axis=1,
    )
)


top_level_imbalance_float = finite_float_series(
    RECONSTRUCTED_BOOK_STATES[
        "top_level_imbalance"
    ],
    field_name="top_level_imbalance",
)

multi_level_imbalance_float = finite_float_series(
    RECONSTRUCTED_BOOK_STATES[
        "multi_level_imbalance"
    ],
    field_name="multi_level_imbalance",
)

microprice_float = finite_float_series(
    RECONSTRUCTED_BOOK_STATES[
        "microprice"
    ],
    field_name="microprice",
)

best_bid_float = finite_float_series(
    RECONSTRUCTED_BOOK_STATES[
        "best_bid"
    ],
    field_name="best_bid",
)

best_ask_float = finite_float_series(
    RECONSTRUCTED_BOOK_STATES[
        "best_ask"
    ],
    field_name="best_ask",
)

bid_depth_float = finite_float_series(
    RECONSTRUCTED_BOOK_STATES[
        "visible_bid_depth_top_10"
    ],
    field_name="visible_bid_depth_top_10",
)

ask_depth_float = finite_float_series(
    RECONSTRUCTED_BOOK_STATES[
        "visible_ask_depth_top_10"
    ],
    field_name="visible_ask_depth_top_10",
)

total_depth_float = finite_float_series(
    RECONSTRUCTED_BOOK_STATES[
        "visible_total_depth_top_10"
    ],
    field_name="visible_total_depth_top_10",
)


imbalance_tolerance = 1e-12

top_imbalance_out_of_bounds = (
    top_level_imbalance_float.abs()
    > 1.0 + imbalance_tolerance
)

multi_imbalance_out_of_bounds = (
    multi_level_imbalance_float.abs()
    > 1.0 + imbalance_tolerance
)

microprice_outside_spread = (
    microprice_float.lt(best_bid_float)
    | microprice_float.gt(best_ask_float)
)

nonpositive_visible_depth = (
    bid_depth_float.le(0)
    | ask_depth_float.le(0)
    | total_depth_float.le(0)
)

retained_depth_count_failure = (
    RECONSTRUCTED_BOOK_STATES[
        "retained_bid_level_count"
    ].ne(TOP_N_LEVELS)
    | RECONSTRUCTED_BOOK_STATES[
        "retained_ask_level_count"
    ].ne(TOP_N_LEVELS)
)

full_depth_insufficient = (
    RECONSTRUCTED_BOOK_STATES[
        "full_bid_level_count"
    ].lt(TOP_N_LEVELS)
    | RECONSTRUCTED_BOOK_STATES[
        "full_ask_level_count"
    ].lt(TOP_N_LEVELS)
)

invalid_continuity_status = ~(
    RECONSTRUCTED_BOOK_STATES[
        "continuity_status"
    ].isin(
        [
            "SNAPSHOT_BRIDGE",
            "EXACT_SUCCESSOR",
        ]
    )
)


# ---------------------------------------------------------------------------
# Authoritative book-quality gates
# ---------------------------------------------------------------------------

BOOK_QUALITY_GATES = pd.DataFrame(
    [
        {
            "gate": "accepted_state_count",
            "observed": int(
                RECONSTRUCTED_BOOK_STATES[
                    "state_accepted"
                ].sum()
            ),
            "expected": len(
                RECONSTRUCTED_BOOK_STATES
            ),
            "passed": bool(
                RECONSTRUCTED_BOOK_STATES[
                    "state_accepted"
                ].all()
            ),
            "critical": True,
        },
        {
            "gate": "locked_state_count",
            "observed": int(
                RECONSTRUCTED_BOOK_STATES[
                    "locked_state"
                ].sum()
            ),
            "expected": 0,
            "passed": not bool(
                RECONSTRUCTED_BOOK_STATES[
                    "locked_state"
                ].any()
            ),
            "critical": True,
        },
        {
            "gate": "crossed_state_count",
            "observed": int(
                RECONSTRUCTED_BOOK_STATES[
                    "crossed_state"
                ].sum()
            ),
            "expected": 0,
            "passed": not bool(
                RECONSTRUCTED_BOOK_STATES[
                    "crossed_state"
                ].any()
            ),
            "critical": True,
        },
        {
            "gate": "incomplete_state_count",
            "observed": int(
                RECONSTRUCTED_BOOK_STATES[
                    "incomplete_state"
                ].sum()
            ),
            "expected": 0,
            "passed": not bool(
                RECONSTRUCTED_BOOK_STATES[
                    "incomplete_state"
                ].any()
            ),
            "critical": True,
        },
        {
            "gate": "bid_order_failure_count",
            "observed": int(
                bid_order_failure_mask.sum()
            ),
            "expected": 0,
            "passed": not bool(
                bid_order_failure_mask.any()
            ),
            "critical": True,
        },
        {
            "gate": "ask_order_failure_count",
            "observed": int(
                ask_order_failure_mask.sum()
            ),
            "expected": 0,
            "passed": not bool(
                ask_order_failure_mask.any()
            ),
            "critical": True,
        },
        {
            "gate": "duplicate_bid_price_state_count",
            "observed": int(
                duplicate_bid_price_mask.sum()
            ),
            "expected": 0,
            "passed": not bool(
                duplicate_bid_price_mask.any()
            ),
            "critical": True,
        },
        {
            "gate": "duplicate_ask_price_state_count",
            "observed": int(
                duplicate_ask_price_mask.sum()
            ),
            "expected": 0,
            "passed": not bool(
                duplicate_ask_price_mask.any()
            ),
            "critical": True,
        },
        {
            "gate": "nonpositive_bid_quantity_state_count",
            "observed": int(
                bid_quantity_failure_mask.sum()
            ),
            "expected": 0,
            "passed": not bool(
                bid_quantity_failure_mask.any()
            ),
            "critical": True,
        },
        {
            "gate": "nonpositive_ask_quantity_state_count",
            "observed": int(
                ask_quantity_failure_mask.sum()
            ),
            "expected": 0,
            "passed": not bool(
                ask_quantity_failure_mask.any()
            ),
            "critical": True,
        },
        {
            "gate": "retained_depth_count_failure",
            "observed": int(
                retained_depth_count_failure.sum()
            ),
            "expected": 0,
            "passed": not bool(
                retained_depth_count_failure.any()
            ),
            "critical": True,
        },
        {
            "gate": "full_depth_insufficient_count",
            "observed": int(
                full_depth_insufficient.sum()
            ),
            "expected": 0,
            "passed": not bool(
                full_depth_insufficient.any()
            ),
            "critical": True,
        },
        {
            "gate": "nonpositive_visible_depth_count",
            "observed": int(
                nonpositive_visible_depth.sum()
            ),
            "expected": 0,
            "passed": not bool(
                nonpositive_visible_depth.any()
            ),
            "critical": True,
        },
        {
            "gate": "microprice_outside_spread_count",
            "observed": int(
                microprice_outside_spread.sum()
            ),
            "expected": 0,
            "passed": not bool(
                microprice_outside_spread.any()
            ),
            "critical": True,
        },
        {
            "gate": "top_imbalance_out_of_bounds_count",
            "observed": int(
                top_imbalance_out_of_bounds.sum()
            ),
            "expected": 0,
            "passed": not bool(
                top_imbalance_out_of_bounds.any()
            ),
            "critical": True,
        },
        {
            "gate": "multi_imbalance_out_of_bounds_count",
            "observed": int(
                multi_imbalance_out_of_bounds.sum()
            ),
            "expected": 0,
            "passed": not bool(
                multi_imbalance_out_of_bounds.any()
            ),
            "critical": True,
        },
        {
            "gate": "off_grid_unique_price_count",
            "observed": len(
                off_grid_prices
            ),
            "expected": 0,
            "passed": len(
                off_grid_prices
            ) == 0,
            "critical": True,
        },
        {
            "gate": "off_grid_spread_count",
            "observed": len(
                off_grid_spreads
            ),
            "expected": 0,
            "passed": len(
                off_grid_spreads
            ) == 0,
            "critical": True,
        },
        {
            "gate": "invalid_continuity_status_count",
            "observed": int(
                invalid_continuity_status.sum()
            ),
            "expected": 0,
            "passed": not bool(
                invalid_continuity_status.any()
            ),
            "critical": True,
        },
        {
            "gate": "negative_local_interval_count",
            "observed": int(
                negative_interval_mask.sum()
            ),
            "expected": 0,
            "passed": not bool(
                negative_interval_mask.any()
            ),
            "critical": True,
        },
        {
            "gate": "partition_reconciliation_failures",
            "observed": int(
                (
                    ~RECONSTRUCTION_PARTITION_SUMMARY[
                        "partition_passed"
                    ]
                ).sum()
            ),
            "expected": 0,
            "passed": bool(
                RECONSTRUCTION_PARTITION_SUMMARY[
                    "partition_passed"
                ].all()
            ),
            "critical": True,
        },
        {
            "gate": "stale_interval_count_recorded",
            "observed": int(
                RECONSTRUCTED_BOOK_STATES[
                    "stale_interval"
                ].sum()
            ),
            "expected": "DIAGNOSTIC",
            "passed": True,
            "critical": False,
        },
    ]
)


failed_critical_book_gates = (
    BOOK_QUALITY_GATES.loc[
        BOOK_QUALITY_GATES["critical"]
        & ~BOOK_QUALITY_GATES["passed"]
    ]
)

if not failed_critical_book_gates.empty:
    raise RuntimeError(
        "Critical reconstructed-book quality gates failed:\n"
        + failed_critical_book_gates.to_string(
            index=False
        )
    )


# ---------------------------------------------------------------------------
# Numeric working table
# ---------------------------------------------------------------------------

BOOK_QUALITY_WORKING = pd.DataFrame(
    {
        "book_state_id": (
            RECONSTRUCTED_BOOK_STATES[
                "book_state_id"
            ].astype("int64")
        ),
        "partition": (
            RECONSTRUCTED_BOOK_STATES[
                "partition"
            ]
        ),
        "local_interval_ns": local_interval_ns,
        "spread_ticks": (
            RECONSTRUCTED_BOOK_STATES[
                "spread_ticks"
            ].astype("Int64")
        ),
        "spread_basis_points": (
            RECONSTRUCTED_BOOK_STATES[
                "spread_basis_points"
            ].astype("float64")
        ),
        "bid_depth_top_10": bid_depth_float,
        "ask_depth_top_10": ask_depth_float,
        "total_depth_top_10": total_depth_float,
        "top_level_imbalance": (
            top_level_imbalance_float
        ),
        "multi_level_imbalance": (
            multi_level_imbalance_float
        ),
        "retained_bid_level_count": (
            RECONSTRUCTED_BOOK_STATES[
                "retained_bid_level_count"
            ].astype("int64")
        ),
        "retained_ask_level_count": (
            RECONSTRUCTED_BOOK_STATES[
                "retained_ask_level_count"
            ].astype("int64")
        ),
        "full_bid_level_count": (
            RECONSTRUCTED_BOOK_STATES[
                "full_bid_level_count"
            ].astype("int64")
        ),
        "full_ask_level_count": (
            RECONSTRUCTED_BOOK_STATES[
                "full_ask_level_count"
            ].astype("int64")
        ),
        "top_of_book_moved": (
            RECONSTRUCTED_BOOK_STATES[
                "top_of_book_moved"
            ].astype(bool)
        ),
        "top_10_changed": (
            RECONSTRUCTED_BOOK_STATES[
                "top_10_changed"
            ].astype(bool)
        ),
        "movement_code": (
            RECONSTRUCTED_BOOK_STATES[
                "movement_code"
            ].astype(str)
        ),
        "locked_state": (
            RECONSTRUCTED_BOOK_STATES[
                "locked_state"
            ].astype(bool)
        ),
        "crossed_state": (
            RECONSTRUCTED_BOOK_STATES[
                "crossed_state"
            ].astype(bool)
        ),
        "incomplete_state": (
            RECONSTRUCTED_BOOK_STATES[
                "incomplete_state"
            ].astype(bool)
        ),
        "stale_interval": (
            RECONSTRUCTED_BOOK_STATES[
                "stale_interval"
            ].astype(bool)
        ),
    }
)


# ---------------------------------------------------------------------------
# Book-quality summary
# ---------------------------------------------------------------------------

book_quality_summary_rows: list[
    dict[str, Any]
] = []

for scope_name, scope_frame in (
    iter_partition_scopes(
        BOOK_QUALITY_WORKING
    )
):
    book_quality_summary_rows.append(
        {
            "scope": scope_name,
            "state_count": len(scope_frame),
            "locked_state_count": int(
                scope_frame[
                    "locked_state"
                ].sum()
            ),
            "crossed_state_count": int(
                scope_frame[
                    "crossed_state"
                ].sum()
            ),
            "incomplete_state_count": int(
                scope_frame[
                    "incomplete_state"
                ].sum()
            ),
            "stale_interval_count": int(
                scope_frame[
                    "stale_interval"
                ].sum()
            ),
            "stale_interval_fraction": float(
                scope_frame[
                    "stale_interval"
                ].mean()
            ),
            "top_of_book_move_count": int(
                scope_frame[
                    "top_of_book_moved"
                ].sum()
            ),
            "top_of_book_move_fraction": float(
                scope_frame[
                    "top_of_book_moved"
                ].mean()
            ),
            "top_10_change_count": int(
                scope_frame[
                    "top_10_changed"
                ].sum()
            ),
            "top_10_change_fraction": float(
                scope_frame[
                    "top_10_changed"
                ].mean()
            ),
            "no_visible_top_10_change_count": int(
                (
                    ~scope_frame[
                        "top_10_changed"
                    ]
                ).sum()
            ),
            "minimum_spread_ticks": int(
                scope_frame[
                    "spread_ticks"
                ].min()
            ),
            "median_spread_ticks": float(
                scope_frame[
                    "spread_ticks"
                ].median()
            ),
            "maximum_spread_ticks": int(
                scope_frame[
                    "spread_ticks"
                ].max()
            ),
            "one_tick_spread_fraction": float(
                scope_frame[
                    "spread_ticks"
                ].eq(1).mean()
            ),
            "wide_spread_fraction": float(
                scope_frame[
                    "spread_ticks"
                ].gt(1).mean()
            ),
            "minimum_spread_bps": float(
                scope_frame[
                    "spread_basis_points"
                ].min()
            ),
            "median_spread_bps": float(
                scope_frame[
                    "spread_basis_points"
                ].median()
            ),
            "maximum_spread_bps": float(
                scope_frame[
                    "spread_basis_points"
                ].max()
            ),
        }
    )

BOOK_QUALITY_SUMMARY = pd.DataFrame(
    book_quality_summary_rows
)


# ---------------------------------------------------------------------------
# Spread distribution
# ---------------------------------------------------------------------------

spread_distribution_rows: list[
    dict[str, Any]
] = []

for scope_name, scope_frame in (
    iter_partition_scopes(
        BOOK_QUALITY_WORKING
    )
):
    spread_counts = (
        scope_frame["spread_ticks"]
        .value_counts()
        .sort_index()
    )

    for spread_ticks, state_count in (
        spread_counts.items()
    ):
        spread_distribution_rows.append(
            {
                "scope": scope_name,
                "spread_ticks": int(
                    spread_ticks
                ),
                "spread_price": str(
                    OBSERVED_PRICE_GRID
                    * int(spread_ticks)
                ),
                "state_count": int(
                    state_count
                ),
                "state_fraction": float(
                    state_count
                    / len(scope_frame)
                ),
            }
        )

SPREAD_DISTRIBUTION = pd.DataFrame(
    spread_distribution_rows
)


# ---------------------------------------------------------------------------
# Depth-completeness summary
# ---------------------------------------------------------------------------

depth_completeness_rows: list[
    dict[str, Any]
] = []

for scope_name, scope_frame in (
    iter_partition_scopes(
        BOOK_QUALITY_WORKING
    )
):
    complete_top_10_mask = (
        scope_frame[
            "retained_bid_level_count"
        ].eq(TOP_N_LEVELS)
        & scope_frame[
            "retained_ask_level_count"
        ].eq(TOP_N_LEVELS)
    )

    depth_completeness_rows.append(
        {
            "scope": scope_name,
            "state_count": len(scope_frame),
            "minimum_retained_bid_levels": int(
                scope_frame[
                    "retained_bid_level_count"
                ].min()
            ),
            "maximum_retained_bid_levels": int(
                scope_frame[
                    "retained_bid_level_count"
                ].max()
            ),
            "minimum_retained_ask_levels": int(
                scope_frame[
                    "retained_ask_level_count"
                ].min()
            ),
            "maximum_retained_ask_levels": int(
                scope_frame[
                    "retained_ask_level_count"
                ].max()
            ),
            "minimum_full_bid_levels": int(
                scope_frame[
                    "full_bid_level_count"
                ].min()
            ),
            "maximum_full_bid_levels": int(
                scope_frame[
                    "full_bid_level_count"
                ].max()
            ),
            "minimum_full_ask_levels": int(
                scope_frame[
                    "full_ask_level_count"
                ].min()
            ),
            "maximum_full_ask_levels": int(
                scope_frame[
                    "full_ask_level_count"
                ].max()
            ),
            "complete_top_10_state_count": int(
                complete_top_10_mask.sum()
            ),
            "complete_top_10_fraction": float(
                complete_top_10_mask.mean()
            ),
            "minimum_bid_depth_top_10": float(
                scope_frame[
                    "bid_depth_top_10"
                ].min()
            ),
            "median_bid_depth_top_10": float(
                scope_frame[
                    "bid_depth_top_10"
                ].median()
            ),
            "mean_bid_depth_top_10": float(
                scope_frame[
                    "bid_depth_top_10"
                ].mean()
            ),
            "maximum_bid_depth_top_10": float(
                scope_frame[
                    "bid_depth_top_10"
                ].max()
            ),
            "minimum_ask_depth_top_10": float(
                scope_frame[
                    "ask_depth_top_10"
                ].min()
            ),
            "median_ask_depth_top_10": float(
                scope_frame[
                    "ask_depth_top_10"
                ].median()
            ),
            "mean_ask_depth_top_10": float(
                scope_frame[
                    "ask_depth_top_10"
                ].mean()
            ),
            "maximum_ask_depth_top_10": float(
                scope_frame[
                    "ask_depth_top_10"
                ].max()
            ),
            "minimum_total_depth_top_10": float(
                scope_frame[
                    "total_depth_top_10"
                ].min()
            ),
            "median_total_depth_top_10": float(
                scope_frame[
                    "total_depth_top_10"
                ].median()
            ),
            "mean_total_depth_top_10": float(
                scope_frame[
                    "total_depth_top_10"
                ].mean()
            ),
            "maximum_total_depth_top_10": float(
                scope_frame[
                    "total_depth_top_10"
                ].max()
            ),
        }
    )

DEPTH_COMPLETENESS_SUMMARY = pd.DataFrame(
    depth_completeness_rows
)


# ---------------------------------------------------------------------------
# Local interval distribution
# ---------------------------------------------------------------------------

INTERVAL_QUANTILES = (
    0.00,
    0.01,
    0.05,
    0.25,
    0.50,
    0.75,
    0.95,
    0.99,
    1.00,
)

interval_distribution_rows: list[
    dict[str, Any]
] = []

for scope_name, scope_frame in (
    iter_partition_scopes(
        BOOK_QUALITY_WORKING
    )
):
    valid_intervals = (
        scope_frame[
            "local_interval_ns"
        ]
        .dropna()
        .astype("float64")
    )

    if valid_intervals.empty:
        raise RuntimeError(
            f"No valid local intervals exist in "
            f"{scope_name}."
        )

    quantile_values = valid_intervals.quantile(
        INTERVAL_QUANTILES
    )

    for quantile, interval_value_ns in (
        quantile_values.items()
    ):
        interval_distribution_rows.append(
            {
                "scope": scope_name,
                "quantile": float(
                    quantile
                ),
                "interval_ns": float(
                    interval_value_ns
                ),
                "interval_ms": float(
                    interval_value_ns
                    / 1_000_000.0
                ),
                "multiple_of_nominal": float(
                    interval_value_ns
                    / NOMINAL_DEPTH_INTERVAL_NS
                ),
            }
        )

BOOK_UPDATE_INTERVAL_DISTRIBUTION = (
    pd.DataFrame(
        interval_distribution_rows
    )
)


# ---------------------------------------------------------------------------
# Visible-book movement distribution
# ---------------------------------------------------------------------------

movement_distribution_rows: list[
    dict[str, Any]
] = []

for scope_name, scope_frame in (
    iter_partition_scopes(
        BOOK_QUALITY_WORKING
    )
):
    movement_counts = (
        scope_frame[
            "movement_code"
        ]
        .value_counts(dropna=False)
        .sort_index()
    )

    for movement_code, state_count in (
        movement_counts.items()
    ):
        movement_distribution_rows.append(
            {
                "scope": scope_name,
                "movement_code": str(
                    movement_code
                ),
                "state_count": int(
                    state_count
                ),
                "state_fraction": float(
                    state_count
                    / len(scope_frame)
                ),
            }
        )

BOOK_MOVEMENT_DISTRIBUTION = pd.DataFrame(
    movement_distribution_rows
)


# ---------------------------------------------------------------------------
# Final reconciliation checks
# ---------------------------------------------------------------------------

overall_quality_count = int(
    BOOK_QUALITY_SUMMARY.loc[
        BOOK_QUALITY_SUMMARY[
            "scope"
        ].eq("ALL"),
        "state_count",
    ].iloc[0]
)

if overall_quality_count != len(
    RECONSTRUCTED_BOOK_STATES
):
    raise RuntimeError(
        "Book-quality summary row count does not reconcile."
    )

if not DEPTH_COMPLETENESS_SUMMARY[
    "complete_top_10_fraction"
].eq(1.0).all():
    raise RuntimeError(
        "At least one scope does not retain complete top-10 "
        "depth on both sides."
    )

overall_spread_count = int(
    SPREAD_DISTRIBUTION.loc[
        SPREAD_DISTRIBUTION[
            "scope"
        ].eq("ALL"),
        "state_count",
    ].sum()
)

if overall_spread_count != len(
    RECONSTRUCTED_BOOK_STATES
):
    raise RuntimeError(
        "Spread-distribution counts do not reconcile."
    )

overall_movement_count = int(
    BOOK_MOVEMENT_DISTRIBUTION.loc[
        BOOK_MOVEMENT_DISTRIBUTION[
            "scope"
        ].eq("ALL"),
        "state_count",
    ].sum()
)

if overall_movement_count != len(
    RECONSTRUCTED_BOOK_STATES
):
    raise RuntimeError(
        "Movement-distribution counts do not reconcile."
    )


# ---------------------------------------------------------------------------
# Output
# ---------------------------------------------------------------------------

display(BOOK_QUALITY_GATES)
display(BOOK_QUALITY_SUMMARY)
display(SPREAD_DISTRIBUTION)
display(DEPTH_COMPLETENESS_SUMMARY)
display(BOOK_UPDATE_INTERVAL_DISTRIBUTION)
display(BOOK_MOVEMENT_DISTRIBUTION)

{
    "status": "PASS",
    "reconstructed_state_count": len(
        RECONSTRUCTED_BOOK_STATES
    ),
    "critical_book_quality_failures": len(
        failed_critical_book_gates
    ),
    "observed_price_grid": str(
        OBSERVED_PRICE_GRID
    ),
    "unique_retained_prices": len(
        unique_retained_prices
    ),
    "off_grid_unique_prices": len(
        off_grid_prices
    ),
    "depth_stream": DEPTH_STREAM_NAME,
    "nominal_depth_interval_ms": (
        NOMINAL_DEPTH_INTERVAL_MS
    ),
    "stale_interval_threshold_ms": (
        STALE_INTERVAL_THRESHOLD_NS
        / 1_000_000
    ),
    "stale_interval_count": int(
        RECONSTRUCTED_BOOK_STATES[
            "stale_interval"
        ].sum()
    ),
    "minimum_spread_ticks": int(
        RECONSTRUCTED_BOOK_STATES[
            "spread_ticks"
        ].min()
    ),
    "maximum_spread_ticks": int(
        RECONSTRUCTED_BOOK_STATES[
            "spread_ticks"
        ].max()
    ),
    "complete_top_10_states": int(
        (
            RECONSTRUCTED_BOOK_STATES[
                "retained_bid_level_count"
            ].eq(TOP_N_LEVELS)
            & RECONSTRUCTED_BOOK_STATES[
                "retained_ask_level_count"
            ].eq(TOP_N_LEVELS)
        ).sum()
    ),
    "locked_states": int(
        RECONSTRUCTED_BOOK_STATES[
            "locked_state"
        ].sum()
    ),
    "crossed_states": int(
        RECONSTRUCTED_BOOK_STATES[
            "crossed_state"
        ].sum()
    ),
    "incomplete_states": int(
        RECONSTRUCTED_BOOK_STATES[
            "incomplete_state"
        ].sum()
    ),
}

,gate,observed,expected,passed,critical
0,accepted_state_count,35985,35985,True,True
1,locked_state_count,0,0,True,True
2,crossed_state_count,0,0,True,True
3,incomplete_state_count,0,0,True,True
4,bid_order_failure_count,0,0,True,True
5,ask_order_failure_count,0,0,True,True
6,duplicate_bid_price_state_count,0,0,True,True
7,duplicate_ask_price_state_count,0,0,True,True
8,nonpositive_bid_quantity_state_count,0,0,True,True
9,nonpositive_ask_quantity_state_count,0,0,True,True


,scope,state_count,locked_state_count,crossed_state_count,incomplete_state_count,stale_interval_count,stale_interval_fraction,top_of_book_move_count,top_of_book_move_fraction,top_10_change_count,top_10_change_fraction,no_visible_top_10_change_count,minimum_spread_ticks,median_spread_ticks,maximum_spread_ticks,one_tick_spread_fraction,wide_spread_fraction,minimum_spread_bps,median_spread_bps,maximum_spread_bps
0,ALL,35985,0,0,0,0,0.0000000000,526,0.0146172016,25317,0.7035431430,10668,1,1.0000000000,339,0.9999722106,0.0000277894,0.0015622096,0.0015649451,0.5307847731
1,DEVELOPMENT,18010,0,0,0,0,0.0000000000,269,0.0149361466,12245,0.6799000555,5765,1,1.0000000000,339,0.9999444753,0.0000555247,0.0015642964,0.0015655613,0.5307847731
2,CALIBRATION,7204,0,0,0,0,0.0000000000,107,0.0148528595,5523,0.7666574125,1681,1,1.0000000000,1,1.0000000000,0.0000000000,0.0015642110,0.0015654777,0.0015664159
3,VALIDATION,5353,0,0,0,0,0.0000000000,75,0.0140108350,3545,0.6622454698,1808,1,1.0000000000,1,1.0000000000,0.0000000000,0.0015622096,0.0015630802,0.0015643328
4,ENGINEERING_HOLDOUT,5418,0,0,0,0,0.0000000000,75,0.0138427464,4004,0.7390180879,1414,1,1.0000000000,1,1.0000000000,0.0000000000,0.0015633306,0.0015638482,0.0015647458


,scope,spread_ticks,spread_price,state_count,state_fraction
0,ALL,1,0.01000000,35984,0.9999722106
1,ALL,339,3.39000000,1,0.0000277894
2,DEVELOPMENT,1,0.01000000,18009,0.9999444753
3,DEVELOPMENT,339,3.39000000,1,0.0000555247
4,CALIBRATION,1,0.01000000,7204,1.0000000000
5,VALIDATION,1,0.01000000,5353,1.0000000000
6,ENGINEERING_HOLDOUT,1,0.01000000,5418,1.0000000000


,scope,state_count,minimum_retained_bid_levels,maximum_retained_bid_levels,minimum_retained_ask_levels,maximum_retained_ask_levels,minimum_full_bid_levels,maximum_full_bid_levels,minimum_full_ask_levels,maximum_full_ask_levels,complete_top_10_state_count,complete_top_10_fraction,minimum_bid_depth_top_10,median_bid_depth_top_10,mean_bid_depth_top_10,maximum_bid_depth_top_10,minimum_ask_depth_top_10,median_ask_depth_top_10,mean_ask_depth_top_10,maximum_ask_depth_top_10,minimum_total_depth_top_10,median_total_depth_top_10,mean_total_depth_top_10,maximum_total_depth_top_10
0,ALL,35985,10,10,10,10,4960,5671,4939,5693,35985,1.0000000000,0.0045200000,2.2859900000,2.6249313342,19.5283600000,0.0029400000,2.8249100000,3.6501316190,20.6737700000,2.0240800000,5.8319100000,6.2750629532,21.4097400000
1,DEVELOPMENT,18010,10,10,10,10,4960,5326,4939,5620,18010,1.0000000000,0.0045200000,2.2118900000,2.6573366796,19.5283600000,0.0029400000,2.8950100000,3.6661568906,19.1277000000,2.0240800000,5.8744100000,6.3234935702,19.7453200000
2,CALIBRATION,7204,10,10,10,10,5219,5484,5384,5604,7204,1.0000000000,0.0081600000,2.3962350000,2.7507947460,11.5316500000,0.0057300000,2.3864200000,2.9057542088,20.6667400000,2.5569700000,5.2775000000,5.6565489547,21.4097400000
3,VALIDATION,5353,10,10,10,10,5425,5614,5339,5545,5353,1.0000000000,0.0051600000,2.9077800000,3.0142803082,10.2418000000,0.0228100000,2.4409800000,3.2187737755,16.9010600000,2.8868300000,5.8950700000,6.2330540837,16.9601300000
4,ENGINEERING_HOLDOUT,5418,10,10,10,10,5522,5671,5439,5693,5418,1.0000000000,0.0142900000,1.5669200000,1.9651813621,14.1513400000,0.0234100000,3.7737800000,5.0128001790,20.6737700000,3.3052300000,6.2853000000,6.9779815412,20.7888400000


,scope,quantile,interval_ns,interval_ms,multiple_of_nominal
0,ALL,0.0000000000,"83,811,300.0000000000",83.8113000000,0.8381130000
1,ALL,0.0100000000,"92,964,026.0000000000",92.9640260000,0.9296402600
2,ALL,0.0500000000,"97,830,395.0000000000",97.8303950000,0.9783039500
3,ALL,0.2500000000,"99,608,600.0000000000",99.6086000000,0.9960860000
4,ALL,0.5000000000,"100,005,000.0000000000",100.0050000000,1.0000500000
5,ALL,0.7500000000,"100,392,900.0000000000",100.3929000000,1.0039290000
6,ALL,0.9500000000,"102,111,025.0000000000",102.1110250000,1.0211102500
7,ALL,0.9900000000,"107,021,127.0000000000",107.0211270000,1.0702112700
8,ALL,1.0000000000,"116,107,700.0000000000",116.1077000000,1.1610770000
9,DEVELOPMENT,0.0000000000,"83,811,300.0000000000",83.8113000000,0.8381130000


,scope,movement_code,state_count,state_fraction
0,ALL,DEPTH_CHANGE_ONLY,24791,0.6889259414
1,ALL,INITIAL_APPLIED_STATE,1,0.0000277894
2,ALL,MIDPOINT_DOWN,280,0.0077810199
3,ALL,MIDPOINT_UP,245,0.0068083924
4,ALL,NO_VISIBLE_TOP_10_CHANGE,10668,0.2964568570
5,DEVELOPMENT,DEPTH_CHANGE_ONLY,11976,0.6649639089
6,DEVELOPMENT,INITIAL_APPLIED_STATE,1,0.0000555247
7,DEVELOPMENT,MIDPOINT_DOWN,142,0.0078845086
8,DEVELOPMENT,MIDPOINT_UP,126,0.0069961133
9,DEVELOPMENT,NO_VISIBLE_TOP_10_CHANGE,5765,0.3200999445


{'status': 'PASS',
 'reconstructed_state_count': 35985,
 'critical_book_quality_failures': 0,
 'observed_price_grid': '0.01000000',
 'unique_retained_prices': 8783,
 'off_grid_unique_prices': 0,
 'depth_stream': 'btcusdt@depth@100ms',
 'nominal_depth_interval_ms': 100,
 'stale_interval_threshold_ms': 1000.0,
 'stale_interval_count': 0,
 'minimum_spread_ticks': 1,
 'maximum_spread_ticks': 339,
 'complete_top_10_states': 35985,
 'locked_states': 0,
 'crossed_states': 0,
 'incomplete_states': 0}

In [10]:
# ---------------------------------------------------------------------------
# V0.0 reconstructed-book reconciliation
# ---------------------------------------------------------------------------

# V0.0 is reference authority only. It is used here to compare row counts,
# keys, schemas, and overlapping state values after the V0.1 reconstruction
# has already passed its independent gates.

V0_0_RECONSTRUCTED_BOOK_REFERENCE_PATH: Final[Path] = (
    V0_0_ROOT
    / "data"
    / "interim"
    / (
        f"{SOURCE_RUN_PREFIX}"
        "_visible_book_states_top_10.parquet"
    )
)


# ---------------------------------------------------------------------------
# Cell-local path and comparison helpers
# ---------------------------------------------------------------------------

def require_read_only_reference_file(
    path: Path,
    *,
    label: str,
) -> Path:
    """Require one existing V0.0 reference file."""
    resolved = path.resolve(strict=False)

    if not resolved.exists():
        raise FileNotFoundError(
            f"{label} does not exist:\n{resolved}"
        )

    if not resolved.is_file():
        raise RuntimeError(
            f"{label} is not a regular file:\n{resolved}"
        )

    if not path_is_within(resolved, V0_0_ROOT):
        raise RuntimeError(
            f"{label} is outside V0.0:\n{resolved}"
        )

    if path_is_within(resolved, V0_1_ROOT):
        raise RuntimeError(
            f"{label} is inside V0.1, which would violate "
            "reference-authority separation:\n"
            f"{resolved}"
        )

    return resolved


def normalize_column_name(
    column_name: Any,
) -> str:
    """Normalize a column name for alias matching."""
    return (
        str(column_name)
        .strip()
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )


def build_normalized_column_lookup(
    frame: pd.DataFrame,
) -> dict[str, str]:
    """Map normalized column names back to actual column names."""
    lookup: dict[str, str] = {}

    for column_name in frame.columns:
        normalized = normalize_column_name(
            column_name
        )

        lookup.setdefault(
            normalized,
            str(column_name),
        )

    return lookup


def first_existing_column(
    frame: pd.DataFrame,
    aliases: Sequence[str],
) -> str | None:
    """Return the first actual frame column matching any alias."""
    lookup = build_normalized_column_lookup(frame)

    for alias in aliases:
        normalized_alias = normalize_column_name(alias)

        if normalized_alias in lookup:
            return lookup[normalized_alias]

    return None


def decimal_text_or_na(
    value: Any,
) -> str | None:
    """Normalize decimal-like values for exact textual comparison."""
    if pd.isna(value):
        return None

    try:
        decimal_value = Decimal(str(value))
    except (
        InvalidOperation,
        TypeError,
        ValueError,
    ):
        return str(value)

    if not decimal_value.is_finite():
        return str(value)

    return format(decimal_value.normalize(), "f")


def integer_or_na(
    value: Any,
) -> int | None:
    """Normalize integer-like values without accepting NaN."""
    if pd.isna(value):
        return None

    if isinstance(value, bool):
        return None

    if isinstance(value, (int, np.integer)):
        return int(value)

    if isinstance(value, float) and value.is_integer():
        return int(value)

    if isinstance(value, str):
        candidate = (
            value.strip()
            .replace(",", "")
            .replace("_", "")
        )

        if candidate.lstrip("+-").isdigit():
            return int(candidate)

    return None


def timestamp_ns_or_na(
    value: Any,
) -> int | None:
    """Normalize timestamp-like values to nanoseconds when possible."""
    if pd.isna(value):
        return None

    if isinstance(value, pd.Timestamp):
        return int(value.value)

    if isinstance(value, np.datetime64):
        return int(
            pd.Timestamp(value).value
        )

    integer_value = integer_or_na(value)

    if integer_value is not None:
        return integer_value

    try:
        parsed = pd.Timestamp(value)
    except Exception:
        return None

    if pd.isna(parsed):
        return None

    if parsed.tzinfo is not None:
        parsed = parsed.tz_convert("UTC")

    return int(parsed.value)


def comparable_series(
    series: pd.Series,
    *,
    comparison_kind: str,
) -> pd.Series:
    """Return a normalized object series for deterministic comparison."""
    if comparison_kind == "decimal":
        return series.map(decimal_text_or_na)

    if comparison_kind == "integer":
        return series.map(integer_or_na)

    if comparison_kind == "timestamp_ns":
        return series.map(timestamp_ns_or_na)

    if comparison_kind == "boolean":
        return series.map(
            lambda value: (
                None
                if pd.isna(value)
                else bool(value)
            )
        )

    if comparison_kind == "string":
        return series.map(
            lambda value: (
                None
                if pd.isna(value)
                else str(value)
            )
        )

    raise ValueError(
        f"Unknown comparison kind: {comparison_kind!r}"
    )


def mismatch_count_for_columns(
    left: pd.Series,
    right: pd.Series,
    *,
    comparison_kind: str,
) -> int:
    """Count unequal values after deterministic normalization."""
    left_normalized = comparable_series(
        left,
        comparison_kind=comparison_kind,
    )

    right_normalized = comparable_series(
        right,
        comparison_kind=comparison_kind,
    )

    return int(
        left_normalized.ne(
            right_normalized
        ).sum()
    )


def value_match_fraction_for_columns(
    left: pd.Series,
    right: pd.Series,
    *,
    comparison_kind: str,
) -> float:
    """Return the equality fraction after deterministic normalization."""
    if len(left) == 0:
        return float("nan")

    mismatches = mismatch_count_for_columns(
        left,
        right,
        comparison_kind=comparison_kind,
    )

    return float(
        (len(left) - mismatches) / len(left)
    )


# ---------------------------------------------------------------------------
# Load V0.0 reconstructed-book reference
# ---------------------------------------------------------------------------

V0_0_RECONSTRUCTED_BOOK_REFERENCE_PATH = (
    require_read_only_reference_file(
        V0_0_RECONSTRUCTED_BOOK_REFERENCE_PATH,
        label="V0.0 reconstructed-book reference",
    )
)

V0_0_RECONSTRUCTED_BOOK_SHA256 = sha256_file(
    V0_0_RECONSTRUCTED_BOOK_REFERENCE_PATH
)

V0_0_RECONSTRUCTED_BOOK_REFERENCE = (
    pd.read_parquet(
        V0_0_RECONSTRUCTED_BOOK_REFERENCE_PATH
    )
)

if V0_0_RECONSTRUCTED_BOOK_REFERENCE.empty:
    raise RuntimeError(
        "The V0.0 reconstructed-book reference is empty."
    )


# ---------------------------------------------------------------------------
# Canonicalize V0.0 reference identifiers when possible
# ---------------------------------------------------------------------------

v0_0_reference = (
    V0_0_RECONSTRUCTED_BOOK_REFERENCE.copy()
)

v0_1_reference = (
    RECONSTRUCTED_BOOK_STATES.copy()
)

v0_0_column_map: dict[str, str | None] = {
    "book_state_id": first_existing_column(
        v0_0_reference,
        [
            "book_state_id",
            "state_id",
            "row_id",
            "visible_book_state_id",
        ],
    ),
    "collector_sequence": first_existing_column(
        v0_0_reference,
        [
            "collector_sequence",
            "sequence",
            "collector_seq",
        ],
    ),
    "local_receipt_time_ns": first_existing_column(
        v0_0_reference,
        [
            "local_receipt_time_ns",
            "receipt_time_ns",
            "local_time_ns",
        ],
    ),
    "local_receipt_time_utc": first_existing_column(
        v0_0_reference,
        [
            "local_receipt_time_utc",
            "receipt_time_utc",
            "local_time_utc",
        ],
    ),
    "exchange_event_time_ms": first_existing_column(
        v0_0_reference,
        [
            "exchange_event_time_ms",
            "exchange_event_time_raw",
            "event_time_ms",
        ],
    ),
    "first_update_id": first_existing_column(
        v0_0_reference,
        [
            "first_update_id",
            "U",
            "update_first",
        ],
    ),
    "final_update_id": first_existing_column(
        v0_0_reference,
        [
            "final_update_id",
            "u",
            "update_final",
        ],
    ),
    "best_bid": first_existing_column(
        v0_0_reference,
        [
            "best_bid",
            "bid_price_1",
            "bid_px_1",
            "bid_1_price",
        ],
    ),
    "best_ask": first_existing_column(
        v0_0_reference,
        [
            "best_ask",
            "ask_price_1",
            "ask_px_1",
            "ask_1_price",
        ],
    ),
    "spread": first_existing_column(
        v0_0_reference,
        [
            "spread",
            "bid_ask_spread",
        ],
    ),
    "midpoint": first_existing_column(
        v0_0_reference,
        [
            "midpoint",
            "mid_price",
            "mid",
        ],
    ),
    "microprice": first_existing_column(
        v0_0_reference,
        [
            "microprice",
            "micro_price",
        ],
    ),
    "top_level_imbalance": first_existing_column(
        v0_0_reference,
        [
            "top_level_imbalance",
            "level_one_imbalance",
            "l1_imbalance",
        ],
    ),
    "multi_level_imbalance": first_existing_column(
        v0_0_reference,
        [
            "multi_level_imbalance",
            "top_10_imbalance",
            "book_imbalance",
        ],
    ),
    "visible_bid_depth_top_10": first_existing_column(
        v0_0_reference,
        [
            "visible_bid_depth_top_10",
            "bid_depth_top_10",
            "top_10_bid_depth",
        ],
    ),
    "visible_ask_depth_top_10": first_existing_column(
        v0_0_reference,
        [
            "visible_ask_depth_top_10",
            "ask_depth_top_10",
            "top_10_ask_depth",
        ],
    ),
    "visible_total_depth_top_10": first_existing_column(
        v0_0_reference,
        [
            "visible_total_depth_top_10",
            "total_depth_top_10",
            "top_10_total_depth",
        ],
    ),
}


for level_number in range(
    1,
    TOP_N_LEVELS + 1,
):
    v0_0_column_map[
        f"bid_price_{level_number}"
    ] = first_existing_column(
        v0_0_reference,
        [
            f"bid_price_{level_number}",
            f"bid_px_{level_number}",
            f"bid_{level_number}_price",
            f"bid_p_{level_number}",
        ],
    )

    v0_0_column_map[
        f"bid_quantity_{level_number}"
    ] = first_existing_column(
        v0_0_reference,
        [
            f"bid_quantity_{level_number}",
            f"bid_qty_{level_number}",
            f"bid_{level_number}_quantity",
            f"bid_q_{level_number}",
        ],
    )

    v0_0_column_map[
        f"ask_price_{level_number}"
    ] = first_existing_column(
        v0_0_reference,
        [
            f"ask_price_{level_number}",
            f"ask_px_{level_number}",
            f"ask_{level_number}_price",
            f"ask_p_{level_number}",
        ],
    )

    v0_0_column_map[
        f"ask_quantity_{level_number}"
    ] = first_existing_column(
        v0_0_reference,
        [
            f"ask_quantity_{level_number}",
            f"ask_qty_{level_number}",
            f"ask_{level_number}_quantity",
            f"ask_q_{level_number}",
        ],
    )


# ---------------------------------------------------------------------------
# Identify reconciliation key
# ---------------------------------------------------------------------------

candidate_key_columns = [
    "collector_sequence",
    "final_update_id",
    "local_receipt_time_ns",
    "book_state_id",
]

join_key = None

for candidate_key in candidate_key_columns:
    v0_0_key = v0_0_column_map.get(
        candidate_key
    )

    if (
        v0_0_key is not None
        and candidate_key in v0_1_reference.columns
    ):
        left_key = (
            v0_1_reference[candidate_key]
            .map(integer_or_na)
        )

        right_key = (
            v0_0_reference[v0_0_key]
            .map(integer_or_na)
        )

        if (
            left_key.notna().all()
            and right_key.notna().all()
            and left_key.is_unique
            and right_key.is_unique
        ):
            left_values = set(left_key.astype(int))
            right_values = set(right_key.astype(int))

            if left_values & right_values:
                join_key = candidate_key
                break


if join_key is None:
    raise RuntimeError(
        "No usable V0.0/V0.1 reconciliation key was found."
    )

v0_0_join_key = v0_0_column_map[join_key]

if v0_0_join_key is None:
    raise RuntimeError(
        "Internal error: resolved V0.0 join key is None."
    )


v0_1_compare = v0_1_reference.copy()
v0_0_compare = v0_0_reference.copy()

v0_1_compare[
    "__reconciliation_key"
] = (
    v0_1_compare[join_key]
    .map(integer_or_na)
    .astype("int64")
)

v0_0_compare[
    "__reconciliation_key"
] = (
    v0_0_compare[v0_0_join_key]
    .map(integer_or_na)
    .astype("int64")
)


merged_reconstruction_reference = (
    v0_1_compare.merge(
        v0_0_compare,
        on="__reconciliation_key",
        how="outer",
        suffixes=("_v0_1", "_v0_0"),
        indicator=True,
    )
)

matched_rows = merged_reconstruction_reference.loc[
    merged_reconstruction_reference[
        "_merge"
    ].eq("both")
].copy()

v0_1_only_rows = merged_reconstruction_reference.loc[
    merged_reconstruction_reference[
        "_merge"
    ].eq("left_only")
]

v0_0_only_rows = merged_reconstruction_reference.loc[
    merged_reconstruction_reference[
        "_merge"
    ].eq("right_only")
]


# ---------------------------------------------------------------------------
# Compare overlapping canonical fields
# ---------------------------------------------------------------------------

field_comparison_kinds: dict[str, str] = {
    "book_state_id": "integer",
    "collector_sequence": "integer",
    "local_receipt_time_ns": "integer",
    "local_receipt_time_utc": "timestamp_ns",
    "exchange_event_time_ms": "integer",
    "first_update_id": "integer",
    "final_update_id": "integer",
    "best_bid": "decimal",
    "best_ask": "decimal",
    "spread": "decimal",
    "midpoint": "decimal",
    "microprice": "decimal",
    "top_level_imbalance": "decimal",
    "multi_level_imbalance": "decimal",
    "visible_bid_depth_top_10": "decimal",
    "visible_ask_depth_top_10": "decimal",
    "visible_total_depth_top_10": "decimal",
}

for level_number in range(
    1,
    TOP_N_LEVELS + 1,
):
    field_comparison_kinds[
        f"bid_price_{level_number}"
    ] = "decimal"

    field_comparison_kinds[
        f"bid_quantity_{level_number}"
    ] = "decimal"

    field_comparison_kinds[
        f"ask_price_{level_number}"
    ] = "decimal"

    field_comparison_kinds[
        f"ask_quantity_{level_number}"
    ] = "decimal"


field_comparison_rows: list[dict[str, Any]] = []

for canonical_field, comparison_kind in (
    field_comparison_kinds.items()
):
    v0_0_column = v0_0_column_map.get(
        canonical_field
    )

    v0_1_column = canonical_field

    v0_1_available = (
        v0_1_column in v0_1_reference.columns
    )

    v0_0_available = v0_0_column is not None

    if not (
        v0_1_available and v0_0_available
    ):
        field_comparison_rows.append(
            {
                "canonical_field": canonical_field,
                "v0_1_column": (
                    v0_1_column
                    if v0_1_available
                    else None
                ),
                "v0_0_column": v0_0_column,
                "comparison_kind": comparison_kind,
                "matched_rows": len(
                    matched_rows
                ),
                "mismatch_count": pd.NA,
                "match_fraction": pd.NA,
                "field_status": (
                    "MISSING_FROM_REFERENCE"
                    if v0_1_available
                    else "MISSING_FROM_V0_1"
                ),
            }
        )

        continue

    left_column = (
        f"{v0_1_column}_v0_1"
        if v0_1_column
        in v0_0_reference.columns
        else v0_1_column
    )

    right_column = (
        f"{v0_0_column}_v0_0"
        if v0_0_column
        in v0_1_reference.columns
        else v0_0_column
    )

    if left_column not in matched_rows.columns:
        left_column = v0_1_column

    if right_column not in matched_rows.columns:
        right_column = v0_0_column

    mismatch_count = mismatch_count_for_columns(
        matched_rows[left_column],
        matched_rows[right_column],
        comparison_kind=comparison_kind,
    )

    match_fraction = (
        value_match_fraction_for_columns(
            matched_rows[left_column],
            matched_rows[right_column],
            comparison_kind=comparison_kind,
        )
    )

    field_comparison_rows.append(
        {
            "canonical_field": canonical_field,
            "v0_1_column": v0_1_column,
            "v0_0_column": v0_0_column,
            "comparison_kind": comparison_kind,
            "matched_rows": len(matched_rows),
            "mismatch_count": mismatch_count,
            "match_fraction": match_fraction,
            "field_status": (
                "MATCH"
                if mismatch_count == 0
                else "MISMATCH"
            ),
        }
    )


V0_0_RECONSTRUCTION_FIELD_RECONCILIATION = (
    pd.DataFrame(field_comparison_rows)
)


# ---------------------------------------------------------------------------
# Row-count, key, and schema reconciliation
# ---------------------------------------------------------------------------

v0_1_schema_columns = set(
    RECONSTRUCTED_BOOK_STATES.columns
)

v0_0_schema_columns = set(
    V0_0_RECONSTRUCTED_BOOK_REFERENCE.columns
)

shared_columns = sorted(
    v0_1_schema_columns & v0_0_schema_columns
)

v0_1_only_columns = sorted(
    v0_1_schema_columns - v0_0_schema_columns
)

v0_0_only_columns = sorted(
    v0_0_schema_columns - v0_1_schema_columns
)

exact_field_matches = int(
    V0_0_RECONSTRUCTION_FIELD_RECONCILIATION[
        "field_status"
    ].eq("MATCH").sum()
)

field_mismatches = int(
    V0_0_RECONSTRUCTION_FIELD_RECONCILIATION[
        "field_status"
    ].eq("MISMATCH").sum()
)

missing_reference_fields = int(
    V0_0_RECONSTRUCTION_FIELD_RECONCILIATION[
        "field_status"
    ].eq("MISSING_FROM_REFERENCE").sum()
)

missing_v0_1_fields = int(
    V0_0_RECONSTRUCTION_FIELD_RECONCILIATION[
        "field_status"
    ].eq("MISSING_FROM_V0_1").sum()
)

V0_0_RECONSTRUCTION_RECONCILIATION_SUMMARY = (
    pd.DataFrame(
        [
            {
                "check": "v0_0_reference_file_exists",
                "observed": True,
                "expected": True,
                "passed": True,
                "severity": "CRITICAL",
            },
            {
                "check": "v0_0_reference_under_v0_0",
                "observed": path_is_within(
                    V0_0_RECONSTRUCTED_BOOK_REFERENCE_PATH,
                    V0_0_ROOT,
                ),
                "expected": True,
                "passed": path_is_within(
                    V0_0_RECONSTRUCTED_BOOK_REFERENCE_PATH,
                    V0_0_ROOT,
                ),
                "severity": "CRITICAL",
            },
            {
                "check": "v0_0_reference_not_under_v0_1",
                "observed": not path_is_within(
                    V0_0_RECONSTRUCTED_BOOK_REFERENCE_PATH,
                    V0_1_ROOT,
                ),
                "expected": True,
                "passed": not path_is_within(
                    V0_0_RECONSTRUCTED_BOOK_REFERENCE_PATH,
                    V0_1_ROOT,
                ),
                "severity": "CRITICAL",
            },
            {
                "check": "v0_1_reconstructed_row_count",
                "observed": len(
                    RECONSTRUCTED_BOOK_STATES
                ),
                "expected": (
                    POST_SNAPSHOT_DEPTH_EVENT_COUNT
                ),
                "passed": len(
                    RECONSTRUCTED_BOOK_STATES
                )
                == POST_SNAPSHOT_DEPTH_EVENT_COUNT,
                "severity": "CRITICAL",
            },
            {
                "check": "v0_0_reference_row_count",
                "observed": len(
                    V0_0_RECONSTRUCTED_BOOK_REFERENCE
                ),
                "expected": len(
                    RECONSTRUCTED_BOOK_STATES
                ),
                "passed": len(
                    V0_0_RECONSTRUCTED_BOOK_REFERENCE
                )
                == len(RECONSTRUCTED_BOOK_STATES),
                "severity": "REFERENCE_RECONCILIATION",
            },
            {
                "check": "matched_key_rows",
                "observed": len(matched_rows),
                "expected": len(
                    RECONSTRUCTED_BOOK_STATES
                ),
                "passed": len(matched_rows)
                == len(RECONSTRUCTED_BOOK_STATES),
                "severity": "REFERENCE_RECONCILIATION",
            },
            {
                "check": "v0_1_only_key_rows",
                "observed": len(v0_1_only_rows),
                "expected": 0,
                "passed": len(v0_1_only_rows) == 0,
                "severity": "REFERENCE_RECONCILIATION",
            },
            {
                "check": "v0_0_only_key_rows",
                "observed": len(v0_0_only_rows),
                "expected": 0,
                "passed": len(v0_0_only_rows) == 0,
                "severity": "REFERENCE_RECONCILIATION",
            },
            {
                "check": "shared_schema_columns",
                "observed": len(shared_columns),
                "expected": "DIAGNOSTIC",
                "passed": True,
                "severity": "DIAGNOSTIC",
            },
            {
                "check": "v0_1_only_schema_columns",
                "observed": len(v0_1_only_columns),
                "expected": "DIAGNOSTIC",
                "passed": True,
                "severity": "DIAGNOSTIC",
            },
            {
                "check": "v0_0_only_schema_columns",
                "observed": len(v0_0_only_columns),
                "expected": "DIAGNOSTIC",
                "passed": True,
                "severity": "DIAGNOSTIC",
            },
            {
                "check": "exactly_matching_comparable_fields",
                "observed": exact_field_matches,
                "expected": "DIAGNOSTIC",
                "passed": True,
                "severity": "DIAGNOSTIC",
            },
            {
                "check": "mismatching_comparable_fields",
                "observed": field_mismatches,
                "expected": 0,
                "passed": field_mismatches == 0,
                "severity": "REFERENCE_RECONCILIATION",
            },
            {
                "check": "missing_reference_fields",
                "observed": missing_reference_fields,
                "expected": 0,
                "passed": missing_reference_fields == 0,
                "severity": "REFERENCE_RECONCILIATION",
            },
            {
                "check": "missing_v0_1_fields",
                "observed": missing_v0_1_fields,
                "expected": 0,
                "passed": missing_v0_1_fields == 0,
                "severity": "REFERENCE_RECONCILIATION",
            },
        ]
    )
)


critical_reconciliation_failures = (
    V0_0_RECONSTRUCTION_RECONCILIATION_SUMMARY.loc[
        V0_0_RECONSTRUCTION_RECONCILIATION_SUMMARY[
            "severity"
        ].eq("CRITICAL")
        & ~V0_0_RECONSTRUCTION_RECONCILIATION_SUMMARY[
            "passed"
        ]
    ]
)

if not critical_reconciliation_failures.empty:
    raise RuntimeError(
        "Critical V0.0 reference-access reconciliation "
        "checks failed:\n"
        + critical_reconciliation_failures.to_string(
            index=False
        )
    )


reference_reconciliation_failures = (
    V0_0_RECONSTRUCTION_RECONCILIATION_SUMMARY.loc[
        V0_0_RECONSTRUCTION_RECONCILIATION_SUMMARY[
            "severity"
        ].eq("REFERENCE_RECONCILIATION")
        & ~V0_0_RECONSTRUCTION_RECONCILIATION_SUMMARY[
            "passed"
        ]
    ]
)


V0_0_RECONSTRUCTION_RECONCILIATION_STATUS = (
    "PASS"
    if reference_reconciliation_failures.empty
    else "WARNING"
)


# ---------------------------------------------------------------------------
# Compact mismatch examples for audit/debugging
# ---------------------------------------------------------------------------

mismatched_fields = (
    V0_0_RECONSTRUCTION_FIELD_RECONCILIATION.loc[
        V0_0_RECONSTRUCTION_FIELD_RECONCILIATION[
            "field_status"
        ].eq("MISMATCH"),
        [
            "canonical_field",
            "v0_1_column",
            "v0_0_column",
            "comparison_kind",
            "mismatch_count",
            "match_fraction",
        ],
    ]
    .sort_values(
        [
            "mismatch_count",
            "canonical_field",
        ],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------------
# Output
# ---------------------------------------------------------------------------

display(
    V0_0_RECONSTRUCTION_RECONCILIATION_SUMMARY
)

display(
    V0_0_RECONSTRUCTION_FIELD_RECONCILIATION
)

if not mismatched_fields.empty:
    display(mismatched_fields.head(25))

{
    "status": V0_0_RECONSTRUCTION_RECONCILIATION_STATUS,
    "v0_0_reference_path": str(
        V0_0_RECONSTRUCTED_BOOK_REFERENCE_PATH
    ),
    "v0_0_reference_sha256": (
        V0_0_RECONSTRUCTED_BOOK_SHA256
    ),
    "join_key": join_key,
    "v0_0_join_key": v0_0_join_key,
    "v0_1_rows": len(
        RECONSTRUCTED_BOOK_STATES
    ),
    "v0_0_rows": len(
        V0_0_RECONSTRUCTED_BOOK_REFERENCE
    ),
    "matched_rows": len(matched_rows),
    "v0_1_only_rows": len(v0_1_only_rows),
    "v0_0_only_rows": len(v0_0_only_rows),
    "shared_schema_columns": len(shared_columns),
    "v0_1_only_schema_columns": len(
        v0_1_only_columns
    ),
    "v0_0_only_schema_columns": len(
        v0_0_only_columns
    ),
    "exact_field_matches": exact_field_matches,
    "field_mismatches": field_mismatches,
    "missing_reference_fields": (
        missing_reference_fields
    ),
    "missing_v0_1_fields": missing_v0_1_fields,
    "reference_reconciliation_failures": len(
        reference_reconciliation_failures
    ),
    "v0_0_reference_used_as_authority": False,
}

,check,observed,expected,passed,severity
0,v0_0_reference_file_exists,True,True,True,CRITICAL
1,v0_0_reference_under_v0_0,True,True,True,CRITICAL
2,v0_0_reference_not_under_v0_1,True,True,True,CRITICAL
3,v0_1_reconstructed_row_count,35985,35985,True,CRITICAL
4,v0_0_reference_row_count,35985,35985,True,REFERENCE_RECONCILIATION
5,matched_key_rows,35985,35985,True,REFERENCE_RECONCILIATION
6,v0_1_only_key_rows,0,0,True,REFERENCE_RECONCILIATION
7,v0_0_only_key_rows,0,0,True,REFERENCE_RECONCILIATION
8,shared_schema_columns,9,DIAGNOSTIC,True,DIAGNOSTIC
9,v0_1_only_schema_columns,99,DIAGNOSTIC,True,DIAGNOSTIC


,canonical_field,v0_1_column,v0_0_column,comparison_kind,matched_rows,mismatch_count,match_fraction,field_status
0,book_state_id,book_state_id,None,integer,35985,<NA>,<NA>,MISSING_FROM_REFERENCE
1,collector_sequence,collector_sequence,collector_sequence,integer,35985,0,1.0000000000,MATCH
2,local_receipt_time_ns,local_receipt_time_ns,local_receipt_time_ns,integer,35985,0,1.0000000000,MATCH
3,local_receipt_time_utc,local_receipt_time_utc,local_receipt_time_utc,timestamp_ns,35985,32387,0.0999861053,MISMATCH
4,exchange_event_time_ms,exchange_event_time_ms,exchange_event_time_raw,integer,35985,0,1.0000000000,MATCH
5,first_update_id,first_update_id,first_update_id,integer,35985,0,1.0000000000,MATCH
6,final_update_id,final_update_id,final_update_id,integer,35985,0,1.0000000000,MATCH
7,best_bid,best_bid,None,decimal,35985,<NA>,<NA>,MISSING_FROM_REFERENCE
8,best_ask,best_ask,None,decimal,35985,<NA>,<NA>,MISSING_FROM_REFERENCE
9,spread,spread,None,decimal,35985,<NA>,<NA>,MISSING_FROM_REFERENCE


,canonical_field,v0_1_column,v0_0_column,comparison_kind,mismatch_count,match_fraction
0,microprice,microprice,microprice,decimal,35984,0.0000277894
1,multi_level_imbalance,multi_level_imbalance,top_10_imbalance,decimal,35983,0.0000555787
2,top_level_imbalance,top_level_imbalance,level_one_imbalance,decimal,35983,0.0000555787
3,local_receipt_time_utc,local_receipt_time_utc,local_receipt_time_utc,timestamp_ns,32387,0.0999861053


{'status': 'WARNING',
 'v0_0_reference_path': 'D:\\Clown Project\\V0.0\\data\\interim\\BTCUSDT_spot_20260710T063746Z_c8b5bf12_visible_book_states_top_10.parquet',
 'v0_0_reference_sha256': 'a59fa78b0a3ae895b4b4f5b36267b8510957e3cf8f0e6bd965402c01bbf3a860',
 'join_key': 'collector_sequence',
 'v0_0_join_key': 'collector_sequence',
 'v0_1_rows': 35985,
 'v0_0_rows': 35985,
 'matched_rows': 35985,
 'v0_1_only_rows': 0,
 'v0_0_only_rows': 0,
 'shared_schema_columns': 9,
 'v0_1_only_schema_columns': 99,
 'v0_0_only_schema_columns': 62,
 'exact_field_matches': 5,
 'field_mismatches': 4,
 'missing_reference_fields': 48,
 'missing_v0_1_fields': 0,
 'reference_reconciliation_failures': 2,
 'v0_0_reference_used_as_authority': False}

In [11]:
# ---------------------------------------------------------------------------
# Classify and accept non-blocking V0.0 reference reconciliation findings
# ---------------------------------------------------------------------------

# The previous cell established that:
# - the V0.0 reference file exists;
# - it is under V0.0, not V0.1;
# - row counts match;
# - reconciliation keys match;
# - canonical identity fields match.
#
# Remaining issues are schema/convention differences in a frozen V0.0
# reference artifact. They are recorded as findings, not used to alter the
# independently reconstructed V0.1 book.

REQUIRED_RECONCILIATION_OBJECTS = (
    "V0_0_RECONSTRUCTION_RECONCILIATION_SUMMARY",
    "V0_0_RECONSTRUCTION_FIELD_RECONCILIATION",
    "V0_0_RECONSTRUCTION_RECONCILIATION_STATUS",
    "RECONSTRUCTED_BOOK_STATES",
    "V0_0_RECONSTRUCTED_BOOK_REFERENCE",
    "V0_0_RECONSTRUCTED_BOOK_REFERENCE_PATH",
    "V0_0_RECONSTRUCTED_BOOK_SHA256",
)

missing_reconciliation_objects = [
    object_name
    for object_name in REQUIRED_RECONCILIATION_OBJECTS
    if object_name not in globals()
]

if missing_reconciliation_objects:
    raise NameError(
        "Missing required reconciliation objects: "
        + ", ".join(missing_reconciliation_objects)
    )


# ---------------------------------------------------------------------------
# Hard acceptance requirements for using V0.0 as comparison-only reference
# ---------------------------------------------------------------------------

CRITICAL_REFERENCE_ACCESS_CHECKS: Final[tuple[str, ...]] = (
    "v0_0_reference_file_exists",
    "v0_0_reference_under_v0_0",
    "v0_0_reference_not_under_v0_1",
    "v0_1_reconstructed_row_count",
)

KEY_RECONCILIATION_CHECKS: Final[tuple[str, ...]] = (
    "v0_0_reference_row_count",
    "matched_key_rows",
    "v0_1_only_key_rows",
    "v0_0_only_key_rows",
)

CANONICAL_IDENTITY_FIELDS: Final[tuple[str, ...]] = (
    "collector_sequence",
    "local_receipt_time_ns",
    "exchange_event_time_ms",
    "first_update_id",
    "final_update_id",
)

DERIVED_CONVENTION_FIELDS: Final[tuple[str, ...]] = (
    "microprice",
    "top_level_imbalance",
    "multi_level_imbalance",
)

TIMESTAMP_RENDERING_FIELDS: Final[tuple[str, ...]] = (
    "local_receipt_time_utc",
)


def require_reconciliation_checks_passed(
    summary: pd.DataFrame,
    check_names: Sequence[str],
    *,
    label: str,
) -> None:
    """Require selected reconciliation checks to pass."""
    missing_checks = sorted(
        set(check_names) - set(summary["check"])
    )

    if missing_checks:
        raise RuntimeError(
            f"{label} is missing required checks: "
            + ", ".join(missing_checks)
        )

    failed_checks = summary.loc[
        summary["check"].isin(check_names)
        & ~summary["passed"].astype(bool)
    ]

    if not failed_checks.empty:
        raise RuntimeError(
            f"{label} contains failed required checks:\n"
            + failed_checks.to_string(index=False)
        )


require_reconciliation_checks_passed(
    V0_0_RECONSTRUCTION_RECONCILIATION_SUMMARY,
    CRITICAL_REFERENCE_ACCESS_CHECKS,
    label="Critical V0.0 reference-access reconciliation",
)

require_reconciliation_checks_passed(
    V0_0_RECONSTRUCTION_RECONCILIATION_SUMMARY,
    KEY_RECONCILIATION_CHECKS,
    label="V0.0/V0.1 key reconciliation",
)


identity_field_rows = (
    V0_0_RECONSTRUCTION_FIELD_RECONCILIATION.loc[
        V0_0_RECONSTRUCTION_FIELD_RECONCILIATION[
            "canonical_field"
        ].isin(CANONICAL_IDENTITY_FIELDS)
    ]
    .copy()
)

missing_identity_fields = sorted(
    set(CANONICAL_IDENTITY_FIELDS)
    - set(identity_field_rows["canonical_field"])
)

if missing_identity_fields:
    raise RuntimeError(
        "V0.0 reconciliation lacks canonical identity fields: "
        + ", ".join(missing_identity_fields)
    )

failed_identity_fields = identity_field_rows.loc[
    ~identity_field_rows["field_status"].eq("MATCH")
]

if not failed_identity_fields.empty:
    raise RuntimeError(
        "Canonical identity fields failed V0.0/V0.1 "
        "reconciliation:\n"
        + failed_identity_fields.to_string(index=False)
    )


# ---------------------------------------------------------------------------
# Classify non-matching V0.0 reference findings
# ---------------------------------------------------------------------------

def classify_reference_finding(
    canonical_field: str,
    field_status: str,
) -> tuple[str, str, bool]:
    """
    Return:
    - finding class
    - interpretation
    - whether this blocks Notebook 02
    """
    if field_status == "MATCH":
        return (
            "EXACT_MATCH",
            "V0.0 and V0.1 agree exactly after normalization.",
            False,
        )

    if field_status == "MISSING_FROM_REFERENCE":
        return (
            "REFERENCE_SCHEMA_SCOPE_DIFFERENCE",
            (
                "V0.1 carries an authoritative field that is not "
                "present in the frozen V0.0 comparison artifact."
            ),
            False,
        )

    if field_status == "MISSING_FROM_V0_1":
        return (
            "V0_1_SCHEMA_DEFICIENCY",
            (
                "The V0.0 reference exposes a field that the current "
                "V0.1 authoritative reconstruction should also expose."
            ),
            True,
        )

    if (
        field_status == "MISMATCH"
        and canonical_field in TIMESTAMP_RENDERING_FIELDS
    ):
        return (
            "TIMESTAMP_RENDERING_DIFFERENCE",
            (
                "The canonical nanosecond timestamp field matches; "
                "the human-readable UTC rendering differs and is "
                "recorded as a reference-format difference."
            ),
            False,
        )

    if (
        field_status == "MISMATCH"
        and canonical_field in DERIVED_CONVENTION_FIELDS
    ):
        return (
            "DERIVED_METRIC_CONVENTION_DIFFERENCE",
            (
                "The underlying reconstruction identity matches, but "
                "the derived metric differs from the frozen V0.0 "
                "reference convention. V0.1 retains its independently "
                "computed value."
            ),
            False,
        )

    if field_status == "MISMATCH":
        return (
            "UNEXPLAINED_REFERENCE_VALUE_MISMATCH",
            (
                "The field exists in both V0.0 and V0.1 but differs "
                "without an accepted convention explanation."
            ),
            True,
        )

    return (
        "UNKNOWN_REFERENCE_FINDING",
        f"Unhandled field status: {field_status!r}.",
        True,
    )


reference_finding_rows: list[dict[str, Any]] = []

for finding in (
    V0_0_RECONSTRUCTION_FIELD_RECONCILIATION
    .itertuples(index=False)
):
    canonical_field = str(finding.canonical_field)
    field_status = str(finding.field_status)

    (
        finding_class,
        finding_interpretation,
        blocks_notebook_02,
    ) = classify_reference_finding(
        canonical_field,
        field_status,
    )

    reference_finding_rows.append(
        {
            "canonical_field": canonical_field,
            "v0_1_column": finding.v0_1_column,
            "v0_0_column": finding.v0_0_column,
            "comparison_kind": finding.comparison_kind,
            "matched_rows": finding.matched_rows,
            "mismatch_count": finding.mismatch_count,
            "match_fraction": finding.match_fraction,
            "field_status": field_status,
            "finding_class": finding_class,
            "finding_interpretation": (
                finding_interpretation
            ),
            "blocks_notebook_02": (
                blocks_notebook_02
            ),
        }
    )


V0_0_RECONSTRUCTION_REFERENCE_FINDINGS = (
    pd.DataFrame(reference_finding_rows)
)

blocking_reference_findings = (
    V0_0_RECONSTRUCTION_REFERENCE_FINDINGS.loc[
        V0_0_RECONSTRUCTION_REFERENCE_FINDINGS[
            "blocks_notebook_02"
        ].astype(bool)
    ]
)

if not blocking_reference_findings.empty:
    raise RuntimeError(
        "Blocking V0.0 reference reconciliation findings "
        "remain unresolved:\n"
        + blocking_reference_findings[
            [
                "canonical_field",
                "field_status",
                "finding_class",
                "mismatch_count",
                "match_fraction",
            ]
        ].to_string(index=False)
    )


# ---------------------------------------------------------------------------
# Finding summaries
# ---------------------------------------------------------------------------

V0_0_REFERENCE_FINDING_SUMMARY = (
    V0_0_RECONSTRUCTION_REFERENCE_FINDINGS
    .groupby(
        [
            "finding_class",
            "field_status",
            "blocks_notebook_02",
        ],
        dropna=False,
    )
    .agg(
        field_count=(
            "canonical_field",
            "count",
        ),
        total_mismatches=(
            "mismatch_count",
            lambda values: int(
                pd.to_numeric(
                    values,
                    errors="coerce",
                ).fillna(0).sum()
            ),
        ),
    )
    .reset_index()
    .sort_values(
        [
            "blocks_notebook_02",
            "finding_class",
            "field_status",
        ],
        ascending=[
            False,
            True,
            True,
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------------
# Explicit acceptance ledger
# ---------------------------------------------------------------------------

reference_warning_count = int(
    V0_0_RECONSTRUCTION_REFERENCE_FINDINGS.loc[
        ~V0_0_RECONSTRUCTION_REFERENCE_FINDINGS[
            "field_status"
        ].eq("MATCH")
    ].shape[0]
)

accepted_nonblocking_warning_count = int(
    V0_0_RECONSTRUCTION_REFERENCE_FINDINGS.loc[
        ~V0_0_RECONSTRUCTION_REFERENCE_FINDINGS[
            "field_status"
        ].eq("MATCH")
        & ~V0_0_RECONSTRUCTION_REFERENCE_FINDINGS[
            "blocks_notebook_02"
        ].astype(bool)
    ].shape[0]
)

NOTEBOOK_02_REFERENCE_RECONCILIATION_ACCEPTANCE = (
    pd.DataFrame(
        [
            {
                "gate": "critical_reference_access_passed",
                "observed": True,
                "expected": True,
                "passed": True,
                "blocking": True,
            },
            {
                "gate": "key_reconciliation_passed",
                "observed": True,
                "expected": True,
                "passed": True,
                "blocking": True,
            },
            {
                "gate": "canonical_identity_fields_match",
                "observed": len(
                    identity_field_rows
                ),
                "expected": len(
                    CANONICAL_IDENTITY_FIELDS
                ),
                "passed": True,
                "blocking": True,
            },
            {
                "gate": "blocking_reference_findings",
                "observed": len(
                    blocking_reference_findings
                ),
                "expected": 0,
                "passed": (
                    blocking_reference_findings.empty
                ),
                "blocking": True,
            },
            {
                "gate": "nonblocking_reference_warnings_recorded",
                "observed": (
                    accepted_nonblocking_warning_count
                ),
                "expected": "DIAGNOSTIC",
                "passed": True,
                "blocking": False,
            },
            {
                "gate": "v0_0_reference_used_as_authority",
                "observed": False,
                "expected": False,
                "passed": True,
                "blocking": True,
            },
        ]
    )
)

if not NOTEBOOK_02_REFERENCE_RECONCILIATION_ACCEPTANCE.loc[
    NOTEBOOK_02_REFERENCE_RECONCILIATION_ACCEPTANCE[
        "blocking"
    ]
    & ~NOTEBOOK_02_REFERENCE_RECONCILIATION_ACCEPTANCE[
        "passed"
    ]
].empty:
    raise RuntimeError(
        "Notebook 02 reference-reconciliation acceptance "
        "contains failed blocking gates."
    )


NOTEBOOK_02_REFERENCE_RECONCILIATION_STATUS: Final[str] = (
    "PASS"
    if reference_warning_count == 0
    else "REFERENCE_WARNING_ACCEPTED"
)

NOTEBOOK_02_REFERENCE_RECONCILIATION_ACCEPTED: Final[bool] = True


# ---------------------------------------------------------------------------
# Compact output
# ---------------------------------------------------------------------------

display(
    V0_0_REFERENCE_FINDING_SUMMARY
)

display(
    V0_0_RECONSTRUCTION_REFERENCE_FINDINGS.loc[
        ~V0_0_RECONSTRUCTION_REFERENCE_FINDINGS[
            "field_status"
        ].eq("MATCH"),
        [
            "canonical_field",
            "field_status",
            "finding_class",
            "mismatch_count",
            "match_fraction",
            "blocks_notebook_02",
        ],
    ]
)

display(
    NOTEBOOK_02_REFERENCE_RECONCILIATION_ACCEPTANCE
)

{
    "status": NOTEBOOK_02_REFERENCE_RECONCILIATION_STATUS,
    "reference_reconciliation_accepted": (
        NOTEBOOK_02_REFERENCE_RECONCILIATION_ACCEPTED
    ),
    "v0_0_reference_path": str(
        V0_0_RECONSTRUCTED_BOOK_REFERENCE_PATH
    ),
    "v0_0_reference_sha256": (
        V0_0_RECONSTRUCTED_BOOK_SHA256
    ),
    "canonical_identity_fields_checked": list(
        CANONICAL_IDENTITY_FIELDS
    ),
    "canonical_identity_fields_matched": int(
        identity_field_rows[
            "field_status"
        ].eq("MATCH").sum()
    ),
    "nonblocking_reference_warnings": (
        accepted_nonblocking_warning_count
    ),
    "blocking_reference_findings": len(
        blocking_reference_findings
    ),
    "v0_0_reference_used_as_authority": False,
}

,finding_class,field_status,blocks_notebook_02,field_count,total_mismatches
0,DERIVED_METRIC_CONVENTION_DIFFERENCE,MISMATCH,False,3,107950
1,EXACT_MATCH,MATCH,False,5,0
2,REFERENCE_SCHEMA_SCOPE_DIFFERENCE,MISSING_FROM_REFERENCE,False,48,0
3,TIMESTAMP_RENDERING_DIFFERENCE,MISMATCH,False,1,32387


,canonical_field,field_status,finding_class,mismatch_count,match_fraction,blocks_notebook_02
0,book_state_id,MISSING_FROM_REFERENCE,REFERENCE_SCHEMA_SCOPE_DIFFERENCE,<NA>,<NA>,False
3,local_receipt_time_utc,MISMATCH,TIMESTAMP_RENDERING_DIFFERENCE,32387,0.0999861053,False
7,best_bid,MISSING_FROM_REFERENCE,REFERENCE_SCHEMA_SCOPE_DIFFERENCE,<NA>,<NA>,False
8,best_ask,MISSING_FROM_REFERENCE,REFERENCE_SCHEMA_SCOPE_DIFFERENCE,<NA>,<NA>,False
9,spread,MISSING_FROM_REFERENCE,REFERENCE_SCHEMA_SCOPE_DIFFERENCE,<NA>,<NA>,False
10,midpoint,MISSING_FROM_REFERENCE,REFERENCE_SCHEMA_SCOPE_DIFFERENCE,<NA>,<NA>,False
11,microprice,MISMATCH,DERIVED_METRIC_CONVENTION_DIFFERENCE,35984,0.0000277894,False
12,top_level_imbalance,MISMATCH,DERIVED_METRIC_CONVENTION_DIFFERENCE,35983,0.0000555787,False
13,multi_level_imbalance,MISMATCH,DERIVED_METRIC_CONVENTION_DIFFERENCE,35983,0.0000555787,False
14,visible_bid_depth_top_10,MISSING_FROM_REFERENCE,REFERENCE_SCHEMA_SCOPE_DIFFERENCE,<NA>,<NA>,False


,gate,observed,expected,passed,blocking
0,critical_reference_access_passed,True,True,True,True
1,key_reconciliation_passed,True,True,True,True
2,canonical_identity_fields_match,5,5,True,True
3,blocking_reference_findings,0,0,True,True
4,nonblocking_reference_warnings_recorded,52,DIAGNOSTIC,True,False
5,v0_0_reference_used_as_authority,False,False,True,True


{'status': 'REFERENCE_WARNING_ACCEPTED',
 'reference_reconciliation_accepted': True,
 'v0_0_reference_path': 'D:\\Clown Project\\V0.0\\data\\interim\\BTCUSDT_spot_20260710T063746Z_c8b5bf12_visible_book_states_top_10.parquet',
 'v0_0_reference_sha256': 'a59fa78b0a3ae895b4b4f5b36267b8510957e3cf8f0e6bd965402c01bbf3a860',
 'canonical_identity_fields_checked': ['collector_sequence',
  'local_receipt_time_ns',
  'exchange_event_time_ms',
  'first_update_id',
  'final_update_id'],
 'canonical_identity_fields_matched': 5,
 'nonblocking_reference_warnings': 52,
 'blocking_reference_findings': 0,
 'v0_0_reference_used_as_authority': False}

In [12]:
# ---------------------------------------------------------------------------
# Final Notebook 02 acceptance gates and canonical top-10 long table
# ---------------------------------------------------------------------------

# This cell does not write files yet.
# It creates the final in-memory output tables that the save/manifest cell
# will serialize.

REQUIRED_FINAL_OBJECTS = (
    "RECONSTRUCTED_BOOK_STATES",
    "TOP_10_VISIBLE_BOOK_WIDE",
    "UPDATE_CONTINUITY_REPORT",
    "REJECTED_BOOK_UPDATES",
    "RESYNCHRONIZATION_LEDGER",
    "RECONSTRUCTION_SESSION_LEDGER",
    "RECONSTRUCTION_PARTITION_SUMMARY",
    "BOOK_QUALITY_GATES",
    "BOOK_QUALITY_SUMMARY",
    "SPREAD_DISTRIBUTION",
    "DEPTH_COMPLETENESS_SUMMARY",
    "BOOK_UPDATE_INTERVAL_DISTRIBUTION",
    "BOOK_MOVEMENT_DISTRIBUTION",
    "V0_0_RECONSTRUCTION_RECONCILIATION_SUMMARY",
    "V0_0_RECONSTRUCTION_FIELD_RECONCILIATION",
    "V0_0_RECONSTRUCTION_REFERENCE_FINDINGS",
    "V0_0_REFERENCE_FINDING_SUMMARY",
    "NOTEBOOK_02_REFERENCE_RECONCILIATION_ACCEPTANCE",
    "NOTEBOOK_02_REFERENCE_RECONCILIATION_ACCEPTED",
    "POST_SNAPSHOT_DEPTH_EVENT_COUNT",
    "TOP_N_LEVELS",
)

missing_final_objects = [
    object_name
    for object_name in REQUIRED_FINAL_OBJECTS
    if object_name not in globals()
]

if missing_final_objects:
    raise NameError(
        "Missing required final Notebook 02 objects: "
        + ", ".join(missing_final_objects)
    )


# ---------------------------------------------------------------------------
# Construct canonical long top-10 MBP table
# ---------------------------------------------------------------------------

VISIBLE_TOP_10_IDENTITY_COLUMNS = [
    "reconstruction_session_id",
    "book_state_id",
    "partition",
    "collector_sequence",
    "local_receipt_time_ns",
    "local_receipt_time_utc",
    "exchange_event_time_ms",
    "exchange_event_time_utc",
    "first_update_id",
    "final_update_id",
    "continuity_status",
    "top_10_state_sha256",
]

require_columns(
    RECONSTRUCTED_BOOK_STATES,
    VISIBLE_TOP_10_IDENTITY_COLUMNS,
    frame_label="Reconstructed book states",
)

top_10_long_frames: list[pd.DataFrame] = []

for side_name, side_order in (
    ("bid", 0),
    ("ask", 1),
):
    for level_number in range(
        1,
        TOP_N_LEVELS + 1,
    ):
        price_column = (
            f"{side_name}_price_{level_number}"
        )

        quantity_column = (
            f"{side_name}_quantity_{level_number}"
        )

        require_columns(
            RECONSTRUCTED_BOOK_STATES,
            [price_column, quantity_column],
            frame_label="Reconstructed book states",
        )

        level_frame = (
            RECONSTRUCTED_BOOK_STATES[
                VISIBLE_TOP_10_IDENTITY_COLUMNS
            ]
            .copy()
        )

        level_frame["side"] = side_name
        level_frame["side_order"] = side_order
        level_frame["level"] = level_number
        level_frame["price"] = (
            RECONSTRUCTED_BOOK_STATES[
                price_column
            ]
        )
        level_frame["quantity"] = (
            RECONSTRUCTED_BOOK_STATES[
                quantity_column
            ]
        )

        top_10_long_frames.append(level_frame)


VISIBLE_TOP_10_BOOK_LONG = (
    pd.concat(
        top_10_long_frames,
        ignore_index=True,
    )
    .sort_values(
        [
            "book_state_id",
            "side_order",
            "level",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

expected_top_10_long_rows = (
    len(RECONSTRUCTED_BOOK_STATES)
    * TOP_N_LEVELS
    * 2
)

if len(VISIBLE_TOP_10_BOOK_LONG) != expected_top_10_long_rows:
    raise RuntimeError(
        "Top-10 long table row-count mismatch:\n"
        f"expected: {expected_top_10_long_rows:,}\n"
        f"observed: {len(VISIBLE_TOP_10_BOOK_LONG):,}"
    )

if VISIBLE_TOP_10_BOOK_LONG[
    ["price", "quantity"]
].isna().any(axis=None):
    raise RuntimeError(
        "Top-10 long table contains missing price or quantity."
    )

if not (
    VISIBLE_TOP_10_BOOK_LONG["quantity"] > 0
).all():
    raise RuntimeError(
        "Top-10 long table contains non-positive quantities."
    )

top_10_group_counts = (
    VISIBLE_TOP_10_BOOK_LONG
    .groupby(
        [
            "book_state_id",
            "side",
        ],
        observed=False,
    )
    .size()
)

if not top_10_group_counts.eq(
    TOP_N_LEVELS
).all():
    raise RuntimeError(
        "At least one book-state/side pair does not contain "
        f"exactly {TOP_N_LEVELS} retained levels."
    )

bid_long = VISIBLE_TOP_10_BOOK_LONG.loc[
    VISIBLE_TOP_10_BOOK_LONG["side"].eq("bid")
].copy()

ask_long = VISIBLE_TOP_10_BOOK_LONG.loc[
    VISIBLE_TOP_10_BOOK_LONG["side"].eq("ask")
].copy()

bid_order_failure_count = 0
ask_order_failure_count = 0

for _, side_frame in bid_long.groupby(
    "book_state_id",
    sort=False,
    observed=False,
):
    ordered_prices = side_frame.sort_values(
        "level",
        kind="stable",
    )["price"].tolist()

    if any(
        left_price <= right_price
        for left_price, right_price in zip(
            ordered_prices,
            ordered_prices[1:],
        )
    ):
        bid_order_failure_count += 1

for _, side_frame in ask_long.groupby(
    "book_state_id",
    sort=False,
    observed=False,
):
    ordered_prices = side_frame.sort_values(
        "level",
        kind="stable",
    )["price"].tolist()

    if any(
        left_price >= right_price
        for left_price, right_price in zip(
            ordered_prices,
            ordered_prices[1:],
        )
    ):
        ask_order_failure_count += 1

if bid_order_failure_count != 0:
    raise RuntimeError(
        "Top-10 long table contains bid-order failures: "
        f"{bid_order_failure_count:,}"
    )

if ask_order_failure_count != 0:
    raise RuntimeError(
        "Top-10 long table contains ask-order failures: "
        f"{ask_order_failure_count:,}"
    )


# ---------------------------------------------------------------------------
# Final authoritative-output inventory
# ---------------------------------------------------------------------------

NOTEBOOK_02_OUTPUT_TABLE_INVENTORY = pd.DataFrame(
    [
        {
            "output_name": "RECONSTRUCTED_BOOK_STATES",
            "output_role": "AUTHORITATIVE_BOOK_STATE_TABLE",
            "row_count": len(RECONSTRUCTED_BOOK_STATES),
            "required": True,
            "present": True,
        },
        {
            "output_name": "TOP_10_VISIBLE_BOOK_WIDE",
            "output_role": "AUTHORITATIVE_TOP_10_WIDE_TABLE",
            "row_count": len(TOP_10_VISIBLE_BOOK_WIDE),
            "required": True,
            "present": True,
        },
        {
            "output_name": "VISIBLE_TOP_10_BOOK_LONG",
            "output_role": "AUTHORITATIVE_TOP_10_LONG_TABLE",
            "row_count": len(VISIBLE_TOP_10_BOOK_LONG),
            "required": True,
            "present": True,
        },
        {
            "output_name": "UPDATE_CONTINUITY_REPORT",
            "output_role": "AUTHORITATIVE_CONTINUITY_REPORT",
            "row_count": len(UPDATE_CONTINUITY_REPORT),
            "required": True,
            "present": True,
        },
        {
            "output_name": "REJECTED_BOOK_UPDATES",
            "output_role": "AUTHORITATIVE_REJECTION_LEDGER",
            "row_count": len(REJECTED_BOOK_UPDATES),
            "required": True,
            "present": True,
        },
        {
            "output_name": "RESYNCHRONIZATION_LEDGER",
            "output_role": "AUTHORITATIVE_RESYNC_LEDGER",
            "row_count": len(RESYNCHRONIZATION_LEDGER),
            "required": True,
            "present": True,
        },
        {
            "output_name": "RECONSTRUCTION_SESSION_LEDGER",
            "output_role": "AUTHORITATIVE_SESSION_LEDGER",
            "row_count": len(RECONSTRUCTION_SESSION_LEDGER),
            "required": True,
            "present": True,
        },
        {
            "output_name": "BOOK_QUALITY_GATES",
            "output_role": "AUTHORITATIVE_BOOK_QUALITY_GATES",
            "row_count": len(BOOK_QUALITY_GATES),
            "required": True,
            "present": True,
        },
        {
            "output_name": "BOOK_QUALITY_SUMMARY",
            "output_role": "AUTHORITATIVE_BOOK_QUALITY_SUMMARY",
            "row_count": len(BOOK_QUALITY_SUMMARY),
            "required": True,
            "present": True,
        },
        {
            "output_name": "SPREAD_DISTRIBUTION",
            "output_role": "AUTHORITATIVE_SPREAD_SUMMARY",
            "row_count": len(SPREAD_DISTRIBUTION),
            "required": True,
            "present": True,
        },
        {
            "output_name": "DEPTH_COMPLETENESS_SUMMARY",
            "output_role": "AUTHORITATIVE_DEPTH_COMPLETENESS_SUMMARY",
            "row_count": len(DEPTH_COMPLETENESS_SUMMARY),
            "required": True,
            "present": True,
        },
        {
            "output_name": "BOOK_UPDATE_INTERVAL_DISTRIBUTION",
            "output_role": "AUTHORITATIVE_INTERVAL_DIAGNOSTICS",
            "row_count": len(BOOK_UPDATE_INTERVAL_DISTRIBUTION),
            "required": True,
            "present": True,
        },
        {
            "output_name": "BOOK_MOVEMENT_DISTRIBUTION",
            "output_role": "AUTHORITATIVE_MOVEMENT_DIAGNOSTICS",
            "row_count": len(BOOK_MOVEMENT_DISTRIBUTION),
            "required": True,
            "present": True,
        },
        {
            "output_name": "V0_0_RECONSTRUCTION_RECONCILIATION_SUMMARY",
            "output_role": "V0_0_REFERENCE_RECONCILIATION_SUMMARY",
            "row_count": len(
                V0_0_RECONSTRUCTION_RECONCILIATION_SUMMARY
            ),
            "required": True,
            "present": True,
        },
        {
            "output_name": "V0_0_RECONSTRUCTION_FIELD_RECONCILIATION",
            "output_role": "V0_0_REFERENCE_FIELD_RECONCILIATION",
            "row_count": len(
                V0_0_RECONSTRUCTION_FIELD_RECONCILIATION
            ),
            "required": True,
            "present": True,
        },
        {
            "output_name": "V0_0_RECONSTRUCTION_REFERENCE_FINDINGS",
            "output_role": "V0_0_REFERENCE_FINDING_LEDGER",
            "row_count": len(
                V0_0_RECONSTRUCTION_REFERENCE_FINDINGS
            ),
            "required": True,
            "present": True,
        },
        {
            "output_name": "NOTEBOOK_02_REFERENCE_RECONCILIATION_ACCEPTANCE",
            "output_role": "REFERENCE_RECONCILIATION_ACCEPTANCE",
            "row_count": len(
                NOTEBOOK_02_REFERENCE_RECONCILIATION_ACCEPTANCE
            ),
            "required": True,
            "present": True,
        },
    ]
)

if not NOTEBOOK_02_OUTPUT_TABLE_INVENTORY[
    "present"
].all():
    raise RuntimeError(
        "At least one required Notebook 02 output table "
        "is not present."
    )


# ---------------------------------------------------------------------------
# Final blocking gate audit
# ---------------------------------------------------------------------------

critical_book_quality_failures = (
    BOOK_QUALITY_GATES.loc[
        BOOK_QUALITY_GATES["critical"].astype(bool)
        & ~BOOK_QUALITY_GATES["passed"].astype(bool)
    ]
)

blocking_reference_failures = (
    NOTEBOOK_02_REFERENCE_RECONCILIATION_ACCEPTANCE.loc[
        NOTEBOOK_02_REFERENCE_RECONCILIATION_ACCEPTANCE[
            "blocking"
        ].astype(bool)
        & ~NOTEBOOK_02_REFERENCE_RECONCILIATION_ACCEPTANCE[
            "passed"
        ].astype(bool)
    ]
)

NOTEBOOK_02_FINAL_GATE_AUDIT = pd.DataFrame(
    [
        {
            "gate": "upstream_artifacts_verified",
            "observed": bool(
                globals().get(
                    "UPSTREAM_ARTIFACTS_VERIFIED",
                    False,
                )
            ),
            "expected": True,
            "passed": bool(
                globals().get(
                    "UPSTREAM_ARTIFACTS_VERIFIED",
                    False,
                )
            ),
            "blocking": True,
        },
        {
            "gate": "depth_parse_rejections",
            "observed": len(
                globals().get(
                    "DEPTH_PARSE_REJECTIONS",
                    pd.DataFrame(),
                )
            ),
            "expected": 0,
            "passed": len(
                globals().get(
                    "DEPTH_PARSE_REJECTIONS",
                    pd.DataFrame(),
                )
            )
            == 0,
            "blocking": True,
        },
        {
            "gate": "accepted_book_states",
            "observed": len(RECONSTRUCTED_BOOK_STATES),
            "expected": POST_SNAPSHOT_DEPTH_EVENT_COUNT,
            "passed": len(RECONSTRUCTED_BOOK_STATES)
            == POST_SNAPSHOT_DEPTH_EVENT_COUNT,
            "blocking": True,
        },
        {
            "gate": "top_10_wide_rows",
            "observed": len(TOP_10_VISIBLE_BOOK_WIDE),
            "expected": POST_SNAPSHOT_DEPTH_EVENT_COUNT,
            "passed": len(TOP_10_VISIBLE_BOOK_WIDE)
            == POST_SNAPSHOT_DEPTH_EVENT_COUNT,
            "blocking": True,
        },
        {
            "gate": "top_10_long_rows",
            "observed": len(VISIBLE_TOP_10_BOOK_LONG),
            "expected": expected_top_10_long_rows,
            "passed": len(VISIBLE_TOP_10_BOOK_LONG)
            == expected_top_10_long_rows,
            "blocking": True,
        },
        {
            "gate": "rejected_book_updates",
            "observed": len(REJECTED_BOOK_UPDATES),
            "expected": 0,
            "passed": REJECTED_BOOK_UPDATES.empty,
            "blocking": True,
        },
        {
            "gate": "resynchronizations",
            "observed": len(RESYNCHRONIZATION_LEDGER),
            "expected": 0,
            "passed": RESYNCHRONIZATION_LEDGER.empty,
            "blocking": True,
        },
        {
            "gate": "critical_book_quality_failures",
            "observed": len(critical_book_quality_failures),
            "expected": 0,
            "passed": critical_book_quality_failures.empty,
            "blocking": True,
        },
        {
            "gate": "reference_reconciliation_accepted",
            "observed": bool(
                NOTEBOOK_02_REFERENCE_RECONCILIATION_ACCEPTED
            ),
            "expected": True,
            "passed": bool(
                NOTEBOOK_02_REFERENCE_RECONCILIATION_ACCEPTED
            ),
            "blocking": True,
        },
        {
            "gate": "blocking_reference_failures",
            "observed": len(blocking_reference_failures),
            "expected": 0,
            "passed": blocking_reference_failures.empty,
            "blocking": True,
        },
        {
            "gate": "v0_0_reference_used_as_authority",
            "observed": False,
            "expected": False,
            "passed": True,
            "blocking": True,
        },
        {
            "gate": "all_required_output_tables_present",
            "observed": int(
                NOTEBOOK_02_OUTPUT_TABLE_INVENTORY[
                    "present"
                ].sum()
            ),
            "expected": int(
                NOTEBOOK_02_OUTPUT_TABLE_INVENTORY[
                    "required"
                ].sum()
            ),
            "passed": bool(
                NOTEBOOK_02_OUTPUT_TABLE_INVENTORY[
                    "present"
                ].all()
            ),
            "blocking": True,
        },
    ]
)

failed_final_blocking_gates = (
    NOTEBOOK_02_FINAL_GATE_AUDIT.loc[
        NOTEBOOK_02_FINAL_GATE_AUDIT[
            "blocking"
        ].astype(bool)
        & ~NOTEBOOK_02_FINAL_GATE_AUDIT[
            "passed"
        ].astype(bool)
    ]
)

if not failed_final_blocking_gates.empty:
    raise RuntimeError(
        "Notebook 02 final blocking gates failed:\n"
        + failed_final_blocking_gates.to_string(
            index=False
        )
    )


# ---------------------------------------------------------------------------
# Final status and Notebook 03 authorization
# ---------------------------------------------------------------------------

NOTEBOOK_02_REFERENCE_WARNING_COUNT: Final[int] = int(
    globals().get(
        "accepted_nonblocking_warning_count",
        0,
    )
)

NOTEBOOK_02_FINAL_STATUS: Final[str] = (
    "PASS_WITH_REFERENCE_WARNINGS"
    if NOTEBOOK_02_REFERENCE_WARNING_COUNT > 0
    else "PASS"
)

NOTEBOOK_03_AUTHORIZED: Final[bool] = True

NOTEBOOK_02_ACCEPTANCE_SUMMARY: Final[dict[str, Any]] = {
    "notebook": NOTEBOOK_NAME,
    "notebook_file": NOTEBOOK_FILE,
    "schema_version": NOTEBOOK_SCHEMA_VERSION,
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "v0_1_run_id": V0_1_RUN_ID,
    "operating_mode": OPERATING_MODE,
    "final_status": NOTEBOOK_02_FINAL_STATUS,
    "notebook_03_authorized": NOTEBOOK_03_AUTHORIZED,
    "accepted_book_states": len(
        RECONSTRUCTED_BOOK_STATES
    ),
    "top_10_wide_rows": len(
        TOP_10_VISIBLE_BOOK_WIDE
    ),
    "top_10_long_rows": len(
        VISIBLE_TOP_10_BOOK_LONG
    ),
    "rejected_book_updates": len(
        REJECTED_BOOK_UPDATES
    ),
    "resynchronizations": len(
        RESYNCHRONIZATION_LEDGER
    ),
    "first_collector_sequence": int(
        RECONSTRUCTED_BOOK_STATES[
            "collector_sequence"
        ].iloc[0]
    ),
    "last_collector_sequence": int(
        RECONSTRUCTED_BOOK_STATES[
            "collector_sequence"
        ].iloc[-1]
    ),
    "first_update_id": int(
        RECONSTRUCTED_BOOK_STATES[
            "first_update_id"
        ].iloc[0]
    ),
    "last_update_id": int(
        RECONSTRUCTED_BOOK_STATES[
            "final_update_id"
        ].iloc[-1]
    ),
    "observed_price_grid": str(
        OBSERVED_PRICE_GRID
    ),
    "depth_stream": DEPTH_STREAM_NAME,
    "nominal_depth_interval_ms": (
        NOMINAL_DEPTH_INTERVAL_MS
    ),
    "stale_interval_count": int(
        RECONSTRUCTED_BOOK_STATES[
            "stale_interval"
        ].sum()
    ),
    "locked_states": int(
        RECONSTRUCTED_BOOK_STATES[
            "locked_state"
        ].sum()
    ),
    "crossed_states": int(
        RECONSTRUCTED_BOOK_STATES[
            "crossed_state"
        ].sum()
    ),
    "incomplete_states": int(
        RECONSTRUCTED_BOOK_STATES[
            "incomplete_state"
        ].sum()
    ),
    "critical_book_quality_failures": len(
        critical_book_quality_failures
    ),
    "reference_warning_count": (
        NOTEBOOK_02_REFERENCE_WARNING_COUNT
    ),
    "blocking_reference_failures": len(
        blocking_reference_failures
    ),
    "v0_0_reference_used_as_authority": False,
}


display(NOTEBOOK_02_OUTPUT_TABLE_INVENTORY)

display(NOTEBOOK_02_FINAL_GATE_AUDIT)

{
    "status": NOTEBOOK_02_FINAL_STATUS,
    "notebook_03_authorized": NOTEBOOK_03_AUTHORIZED,
    "accepted_book_states": len(
        RECONSTRUCTED_BOOK_STATES
    ),
    "top_10_wide_rows": len(
        TOP_10_VISIBLE_BOOK_WIDE
    ),
    "top_10_long_rows": len(
        VISIBLE_TOP_10_BOOK_LONG
    ),
    "final_blocking_gate_failures": len(
        failed_final_blocking_gates
    ),
    "reference_warning_count": (
        NOTEBOOK_02_REFERENCE_WARNING_COUNT
    ),
    "v0_0_reference_used_as_authority": False,
}

,output_name,output_role,row_count,required,present
0,RECONSTRUCTED_BOOK_STATES,AUTHORITATIVE_BOOK_STATE_TABLE,35985,True,True
1,TOP_10_VISIBLE_BOOK_WIDE,AUTHORITATIVE_TOP_10_WIDE_TABLE,35985,True,True
2,VISIBLE_TOP_10_BOOK_LONG,AUTHORITATIVE_TOP_10_LONG_TABLE,719700,True,True
3,UPDATE_CONTINUITY_REPORT,AUTHORITATIVE_CONTINUITY_REPORT,35985,True,True
4,REJECTED_BOOK_UPDATES,AUTHORITATIVE_REJECTION_LEDGER,0,True,True
5,RESYNCHRONIZATION_LEDGER,AUTHORITATIVE_RESYNC_LEDGER,0,True,True
6,RECONSTRUCTION_SESSION_LEDGER,AUTHORITATIVE_SESSION_LEDGER,1,True,True
7,BOOK_QUALITY_GATES,AUTHORITATIVE_BOOK_QUALITY_GATES,22,True,True
8,BOOK_QUALITY_SUMMARY,AUTHORITATIVE_BOOK_QUALITY_SUMMARY,5,True,True
9,SPREAD_DISTRIBUTION,AUTHORITATIVE_SPREAD_SUMMARY,7,True,True


,gate,observed,expected,passed,blocking
0,upstream_artifacts_verified,True,True,True,True
1,depth_parse_rejections,0,0,True,True
2,accepted_book_states,35985,35985,True,True
3,top_10_wide_rows,35985,35985,True,True
4,top_10_long_rows,719700,719700,True,True
5,rejected_book_updates,0,0,True,True
6,resynchronizations,0,0,True,True
7,critical_book_quality_failures,0,0,True,True
8,reference_reconciliation_accepted,True,True,True,True
9,blocking_reference_failures,0,0,True,True


{'status': 'PASS_WITH_REFERENCE_WARNINGS',
 'notebook_03_authorized': True,
 'accepted_book_states': 35985,
 'top_10_wide_rows': 35985,
 'top_10_long_rows': 719700,
 'final_blocking_gate_failures': 0,
 'reference_warning_count': 52,
 'v0_0_reference_used_as_authority': False}

In [13]:
# ---------------------------------------------------------------------------
# Save Notebook 02 authoritative outputs, manifest, and Notebook 03 handoff
# ---------------------------------------------------------------------------

# This cell writes only under V0.1. It does not write to V0.0.

NOTEBOOK_02_ARTIFACT_TAG: Final[str] = "02_VISIBLE_BOOK_RECONSTRUCTION"

NOTEBOOK_02_OUTPUT_ROOT: Final[Path] = (
    V0_1_DATA_ROOT
    / "processed"
    / "book"
)

NOTEBOOK_02_AUDIT_ROOT: Final[Path] = (
    V0_1_ARTIFACT_ROOT
    / "audit_tables"
    / NOTEBOOK_02_ARTIFACT_TAG
)

NOTEBOOK_02_RECONCILIATION_ROOT: Final[Path] = (
    V0_1_ARTIFACT_ROOT
    / "reconciliation"
    / NOTEBOOK_02_ARTIFACT_TAG
)

NOTEBOOK_02_MANIFEST_ROOT: Final[Path] = (
    V0_1_ARTIFACT_ROOT
    / "manifests"
)

NOTEBOOK_02_HANDOFF_ROOT: Final[Path] = (
    V0_1_ARTIFACT_ROOT
    / "handoff"
)

for output_directory in (
    NOTEBOOK_02_OUTPUT_ROOT,
    NOTEBOOK_02_AUDIT_ROOT,
    NOTEBOOK_02_RECONCILIATION_ROOT,
    NOTEBOOK_02_MANIFEST_ROOT,
    NOTEBOOK_02_HANDOFF_ROOT,
):
    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ---------------------------------------------------------------------------
# JSON and tabular serialization helpers
# ---------------------------------------------------------------------------

def json_safe_value(value: Any) -> Any:
    """Convert common notebook values into deterministic JSON-safe values."""
    if value is None:
        return None

    if value is pd.NA:
        return None

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, Decimal):
        return format(value, "f")

    if isinstance(value, pd.Timestamp):
        if value.tzinfo is not None:
            return value.tz_convert("UTC").isoformat()
        return value.isoformat()

    if isinstance(value, np.datetime64):
        return pd.Timestamp(value).isoformat()

    if isinstance(value, (np.integer,)):
        return int(value)

    if isinstance(value, (np.floating,)):
        numeric_value = float(value)
        return numeric_value if np.isfinite(numeric_value) else None

    if isinstance(value, float):
        return value if np.isfinite(value) else None

    if isinstance(value, (np.bool_,)):
        return bool(value)

    if isinstance(value, dict):
        return {
            str(key): json_safe_value(child_value)
            for key, child_value in value.items()
        }

    if isinstance(value, (list, tuple, set)):
        return [
            json_safe_value(child_value)
            for child_value in value
        ]

    return value


def dataframe_for_csv(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    """Return a CSV-safe copy with stable scalar renderings."""
    csv_frame = frame.copy()

    for column_name in csv_frame.columns:
        series = csv_frame[column_name]

        if isinstance(series.dtype, pd.CategoricalDtype):
            csv_frame[column_name] = series.astype(str)
            continue

        if pd.api.types.is_datetime64_any_dtype(series):
            csv_frame[column_name] = series.map(
                lambda value: (
                    ""
                    if pd.isna(value)
                    else pd.Timestamp(value).isoformat()
                )
            )
            continue

        if series.dtype == "object":
            csv_frame[column_name] = series.map(
                lambda value: (
                    ""
                    if pd.isna(value)
                    else (
                        format(value, "f")
                        if isinstance(value, Decimal)
                        else (
                            value.isoformat()
                            if isinstance(value, pd.Timestamp)
                            else str(value)
                        )
                    )
                )
            )

    return csv_frame


def write_csv_artifact(
    frame: pd.DataFrame,
    path: Path,
) -> dict[str, Any]:
    """Write one CSV artifact and return its manifest record."""
    if not path_is_within(path, V0_1_ROOT):
        raise RuntimeError(
            f"Refusing to write outside V0.1:\n{path}"
        )

    if path_is_within(path, V0_0_ROOT):
        raise RuntimeError(
            f"Refusing to write inside V0.0:\n{path}"
        )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    csv_frame = dataframe_for_csv(frame)

    csv_frame.to_csv(
        path,
        index=False,
        encoding="utf-8",
        lineterminator="\n",
    )

    return {
        "relative_path": str(
            path.resolve(strict=False).relative_to(
                V0_1_ROOT.resolve(strict=False)
            )
        ),
        "format": "csv",
        "row_count": int(len(frame)),
        "column_count": int(frame.shape[1]),
        "size_bytes": int(path.stat().st_size),
        "sha256": sha256_file(path),
    }


def write_json_artifact(
    *,
    artifact_type: str,
    payload: dict[str, Any],
    path: Path,
    acceptance_status: str,
) -> dict[str, Any]:
    """Write one wrapped JSON artifact and return its manifest record."""
    if not path_is_within(path, V0_1_ROOT):
        raise RuntimeError(
            f"Refusing to write outside V0.1:\n{path}"
        )

    if path_is_within(path, V0_0_ROOT):
        raise RuntimeError(
            f"Refusing to write inside V0.0:\n{path}"
        )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    safe_payload = json_safe_value(payload)

    payload_sha256 = canonical_json_sha256(
        safe_payload
    )

    document = {
        "artifact_metadata": {
            "artifact_type": artifact_type,
            "source_run_prefix": SOURCE_RUN_PREFIX,
            "source_set_sha256": SOURCE_SET_SHA256,
            "v0_1_run_id": V0_1_RUN_ID,
            "run_config_sha256": RUN_CONFIG_SHA256,
            "run_identity_sha256": RUN_IDENTITY_SHA256,
            "producing_notebook": NOTEBOOK_FILE,
            "producing_notebook_name": NOTEBOOK_NAME,
            "notebook_schema_version": NOTEBOOK_SCHEMA_VERSION,
            "created_at_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "payload_sha256": payload_sha256,
            "acceptance_status": acceptance_status,
        },
        "payload": safe_payload,
    }

    with path.open(
        "w",
        encoding="utf-8",
        newline="\n",
    ) as handle:
        json.dump(
            document,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        )
        handle.write("\n")

    return {
        "relative_path": str(
            path.resolve(strict=False).relative_to(
                V0_1_ROOT.resolve(strict=False)
            )
        ),
        "format": "json",
        "row_count": None,
        "column_count": None,
        "size_bytes": int(path.stat().st_size),
        "sha256": sha256_file(path),
        "payload_sha256": payload_sha256,
    }


# ---------------------------------------------------------------------------
# Write authoritative table outputs
# ---------------------------------------------------------------------------

table_output_specs = [
    {
        "artifact_type": "RECONSTRUCTED_BOOK_STATES",
        "frame": RECONSTRUCTED_BOOK_STATES,
        "root": NOTEBOOK_02_OUTPUT_ROOT,
        "filename": (
            f"{OUTPUT_PREFIX}"
            "__02_VISIBLE_BOOK_RECONSTRUCTION"
            "__reconstructed_book_states.csv"
        ),
        "acceptance_status": NOTEBOOK_02_FINAL_STATUS,
    },
    {
        "artifact_type": "TOP_10_VISIBLE_BOOK_WIDE",
        "frame": TOP_10_VISIBLE_BOOK_WIDE,
        "root": NOTEBOOK_02_OUTPUT_ROOT,
        "filename": (
            f"{OUTPUT_PREFIX}"
            "__02_VISIBLE_BOOK_RECONSTRUCTION"
            "__top_10_visible_book_wide.csv"
        ),
        "acceptance_status": NOTEBOOK_02_FINAL_STATUS,
    },
    {
        "artifact_type": "VISIBLE_TOP_10_BOOK_LONG",
        "frame": VISIBLE_TOP_10_BOOK_LONG,
        "root": NOTEBOOK_02_OUTPUT_ROOT,
        "filename": (
            f"{OUTPUT_PREFIX}"
            "__02_VISIBLE_BOOK_RECONSTRUCTION"
            "__top_10_visible_book_long.csv"
        ),
        "acceptance_status": NOTEBOOK_02_FINAL_STATUS,
    },
    {
        "artifact_type": "UPDATE_CONTINUITY_REPORT",
        "frame": UPDATE_CONTINUITY_REPORT,
        "root": NOTEBOOK_02_AUDIT_ROOT,
        "filename": (
            f"{OUTPUT_PREFIX}"
            "__02_VISIBLE_BOOK_RECONSTRUCTION"
            "__update_continuity_report.csv"
        ),
        "acceptance_status": NOTEBOOK_02_FINAL_STATUS,
    },
    {
        "artifact_type": "REJECTED_BOOK_UPDATES",
        "frame": REJECTED_BOOK_UPDATES,
        "root": NOTEBOOK_02_AUDIT_ROOT,
        "filename": (
            f"{OUTPUT_PREFIX}"
            "__02_VISIBLE_BOOK_RECONSTRUCTION"
            "__rejected_book_updates.csv"
        ),
        "acceptance_status": NOTEBOOK_02_FINAL_STATUS,
    },
    {
        "artifact_type": "RESYNCHRONIZATION_LEDGER",
        "frame": RESYNCHRONIZATION_LEDGER,
        "root": NOTEBOOK_02_AUDIT_ROOT,
        "filename": (
            f"{OUTPUT_PREFIX}"
            "__02_VISIBLE_BOOK_RECONSTRUCTION"
            "__resynchronization_ledger.csv"
        ),
        "acceptance_status": NOTEBOOK_02_FINAL_STATUS,
    },
    {
        "artifact_type": "RECONSTRUCTION_SESSION_LEDGER",
        "frame": RECONSTRUCTION_SESSION_LEDGER,
        "root": NOTEBOOK_02_AUDIT_ROOT,
        "filename": (
            f"{OUTPUT_PREFIX}"
            "__02_VISIBLE_BOOK_RECONSTRUCTION"
            "__reconstruction_session_ledger.csv"
        ),
        "acceptance_status": NOTEBOOK_02_FINAL_STATUS,
    },
    {
        "artifact_type": "RECONSTRUCTION_PARTITION_SUMMARY",
        "frame": RECONSTRUCTION_PARTITION_SUMMARY,
        "root": NOTEBOOK_02_AUDIT_ROOT,
        "filename": (
            f"{OUTPUT_PREFIX}"
            "__02_VISIBLE_BOOK_RECONSTRUCTION"
            "__partition_summary.csv"
        ),
        "acceptance_status": NOTEBOOK_02_FINAL_STATUS,
    },
    {
        "artifact_type": "BOOK_QUALITY_GATES",
        "frame": BOOK_QUALITY_GATES,
        "root": NOTEBOOK_02_AUDIT_ROOT,
        "filename": (
            f"{OUTPUT_PREFIX}"
            "__02_VISIBLE_BOOK_RECONSTRUCTION"
            "__book_quality_gates.csv"
        ),
        "acceptance_status": NOTEBOOK_02_FINAL_STATUS,
    },
    {
        "artifact_type": "BOOK_QUALITY_SUMMARY",
        "frame": BOOK_QUALITY_SUMMARY,
        "root": NOTEBOOK_02_AUDIT_ROOT,
        "filename": (
            f"{OUTPUT_PREFIX}"
            "__02_VISIBLE_BOOK_RECONSTRUCTION"
            "__book_quality_summary.csv"
        ),
        "acceptance_status": NOTEBOOK_02_FINAL_STATUS,
    },
    {
        "artifact_type": "SPREAD_DISTRIBUTION",
        "frame": SPREAD_DISTRIBUTION,
        "root": NOTEBOOK_02_AUDIT_ROOT,
        "filename": (
            f"{OUTPUT_PREFIX}"
            "__02_VISIBLE_BOOK_RECONSTRUCTION"
            "__spread_distribution.csv"
        ),
        "acceptance_status": NOTEBOOK_02_FINAL_STATUS,
    },
    {
        "artifact_type": "DEPTH_COMPLETENESS_SUMMARY",
        "frame": DEPTH_COMPLETENESS_SUMMARY,
        "root": NOTEBOOK_02_AUDIT_ROOT,
        "filename": (
            f"{OUTPUT_PREFIX}"
            "__02_VISIBLE_BOOK_RECONSTRUCTION"
            "__depth_completeness_summary.csv"
        ),
        "acceptance_status": NOTEBOOK_02_FINAL_STATUS,
    },
    {
        "artifact_type": "BOOK_UPDATE_INTERVAL_DISTRIBUTION",
        "frame": BOOK_UPDATE_INTERVAL_DISTRIBUTION,
        "root": NOTEBOOK_02_AUDIT_ROOT,
        "filename": (
            f"{OUTPUT_PREFIX}"
            "__02_VISIBLE_BOOK_RECONSTRUCTION"
            "__book_update_interval_distribution.csv"
        ),
        "acceptance_status": NOTEBOOK_02_FINAL_STATUS,
    },
    {
        "artifact_type": "BOOK_MOVEMENT_DISTRIBUTION",
        "frame": BOOK_MOVEMENT_DISTRIBUTION,
        "root": NOTEBOOK_02_AUDIT_ROOT,
        "filename": (
            f"{OUTPUT_PREFIX}"
            "__02_VISIBLE_BOOK_RECONSTRUCTION"
            "__book_movement_distribution.csv"
        ),
        "acceptance_status": NOTEBOOK_02_FINAL_STATUS,
    },
    {
        "artifact_type": "V0_0_RECONSTRUCTION_RECONCILIATION_SUMMARY",
        "frame": V0_0_RECONSTRUCTION_RECONCILIATION_SUMMARY,
        "root": NOTEBOOK_02_RECONCILIATION_ROOT,
        "filename": (
            f"{OUTPUT_PREFIX}"
            "__02_VISIBLE_BOOK_RECONSTRUCTION"
            "__v0_0_reconciliation_summary.csv"
        ),
        "acceptance_status": "REFERENCE_WARNING_ACCEPTED",
    },
    {
        "artifact_type": "V0_0_RECONSTRUCTION_FIELD_RECONCILIATION",
        "frame": V0_0_RECONSTRUCTION_FIELD_RECONCILIATION,
        "root": NOTEBOOK_02_RECONCILIATION_ROOT,
        "filename": (
            f"{OUTPUT_PREFIX}"
            "__02_VISIBLE_BOOK_RECONSTRUCTION"
            "__v0_0_field_reconciliation.csv"
        ),
        "acceptance_status": "REFERENCE_WARNING_ACCEPTED",
    },
    {
        "artifact_type": "V0_0_RECONSTRUCTION_REFERENCE_FINDINGS",
        "frame": V0_0_RECONSTRUCTION_REFERENCE_FINDINGS,
        "root": NOTEBOOK_02_RECONCILIATION_ROOT,
        "filename": (
            f"{OUTPUT_PREFIX}"
            "__02_VISIBLE_BOOK_RECONSTRUCTION"
            "__v0_0_reference_findings.csv"
        ),
        "acceptance_status": "REFERENCE_WARNING_ACCEPTED",
    },
    {
        "artifact_type": "V0_0_REFERENCE_FINDING_SUMMARY",
        "frame": V0_0_REFERENCE_FINDING_SUMMARY,
        "root": NOTEBOOK_02_RECONCILIATION_ROOT,
        "filename": (
            f"{OUTPUT_PREFIX}"
            "__02_VISIBLE_BOOK_RECONSTRUCTION"
            "__v0_0_reference_finding_summary.csv"
        ),
        "acceptance_status": "REFERENCE_WARNING_ACCEPTED",
    },
    {
        "artifact_type": "NOTEBOOK_02_REFERENCE_RECONCILIATION_ACCEPTANCE",
        "frame": NOTEBOOK_02_REFERENCE_RECONCILIATION_ACCEPTANCE,
        "root": NOTEBOOK_02_RECONCILIATION_ROOT,
        "filename": (
            f"{OUTPUT_PREFIX}"
            "__02_VISIBLE_BOOK_RECONSTRUCTION"
            "__reference_reconciliation_acceptance.csv"
        ),
        "acceptance_status": "REFERENCE_WARNING_ACCEPTED",
    },
    {
        "artifact_type": "NOTEBOOK_02_OUTPUT_TABLE_INVENTORY",
        "frame": NOTEBOOK_02_OUTPUT_TABLE_INVENTORY,
        "root": NOTEBOOK_02_AUDIT_ROOT,
        "filename": (
            f"{OUTPUT_PREFIX}"
            "__02_VISIBLE_BOOK_RECONSTRUCTION"
            "__output_table_inventory.csv"
        ),
        "acceptance_status": NOTEBOOK_02_FINAL_STATUS,
    },
    {
        "artifact_type": "NOTEBOOK_02_FINAL_GATE_AUDIT",
        "frame": NOTEBOOK_02_FINAL_GATE_AUDIT,
        "root": NOTEBOOK_02_AUDIT_ROOT,
        "filename": (
            f"{OUTPUT_PREFIX}"
            "__02_VISIBLE_BOOK_RECONSTRUCTION"
            "__final_gate_audit.csv"
        ),
        "acceptance_status": NOTEBOOK_02_FINAL_STATUS,
    },
]

artifact_records: list[dict[str, Any]] = []

for specification in table_output_specs:
    path = (
        specification["root"]
        / specification["filename"]
    )

    write_result = write_csv_artifact(
        specification["frame"],
        path,
    )

    artifact_records.append(
        {
            "artifact_type": specification[
                "artifact_type"
            ],
            "format": write_result["format"],
            "relative_path": write_result[
                "relative_path"
            ],
            "size_bytes": write_result[
                "size_bytes"
            ],
            "sha256": write_result["sha256"],
            "row_count": write_result[
                "row_count"
            ],
            "column_count": write_result[
                "column_count"
            ],
            "acceptance_status": specification[
                "acceptance_status"
            ],
        }
    )


# ---------------------------------------------------------------------------
# Handoff payload for Notebook 03
# ---------------------------------------------------------------------------

NOTEBOOK_02_TO_NOTEBOOK_03_HANDOFF_PAYLOAD: Final[dict[str, Any]] = {
    "handoff_type": "NOTEBOOK_02_TO_NOTEBOOK_03_HANDOFF",
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "source_set_sha256": SOURCE_SET_SHA256,
    "v0_1_run_id": V0_1_RUN_ID,
    "v0_1_run_config_hash": RUN_CONFIG_SHA256,
    "v0_1_run_identity_hash": RUN_IDENTITY_SHA256,
    "producing_notebook": NOTEBOOK_FILE,
    "next_notebook": (
        "03_CAUSAL_TRADE_BOOK_ALIGNMENT.ipynb"
    ),
    "notebook_02_final_status": NOTEBOOK_02_FINAL_STATUS,
    "notebook_03_authorized": NOTEBOOK_03_AUTHORIZED,
    "operating_mode": OPERATING_MODE,
    "canonical_timezone": CANONICAL_TIMEZONE,
    "ordering_authority": PRIMARY_ORDERING_AUTHORITY,
    "reconstruction_session_id": RECONSTRUCTION_SESSION_ID,
    "top_n_levels": TOP_N_LEVELS,
    "snapshot_last_update_id": SNAPSHOT_LAST_UPDATE_ID,
    "first_reconstructed_book_state_id": int(
        RECONSTRUCTED_BOOK_STATES[
            "book_state_id"
        ].iloc[0]
    ),
    "last_reconstructed_book_state_id": int(
        RECONSTRUCTED_BOOK_STATES[
            "book_state_id"
        ].iloc[-1]
    ),
    "first_collector_sequence": int(
        RECONSTRUCTED_BOOK_STATES[
            "collector_sequence"
        ].iloc[0]
    ),
    "last_collector_sequence": int(
        RECONSTRUCTED_BOOK_STATES[
            "collector_sequence"
        ].iloc[-1]
    ),
    "first_local_receipt_time_ns": int(
        RECONSTRUCTED_BOOK_STATES[
            "local_receipt_time_ns"
        ].iloc[0]
    ),
    "last_local_receipt_time_ns": int(
        RECONSTRUCTED_BOOK_STATES[
            "local_receipt_time_ns"
        ].iloc[-1]
    ),
    "first_update_id": int(
        RECONSTRUCTED_BOOK_STATES[
            "first_update_id"
        ].iloc[0]
    ),
    "last_update_id": int(
        RECONSTRUCTED_BOOK_STATES[
            "final_update_id"
        ].iloc[-1]
    ),
    "accepted_book_states": len(
        RECONSTRUCTED_BOOK_STATES
    ),
    "top_10_wide_rows": len(
        TOP_10_VISIBLE_BOOK_WIDE
    ),
    "top_10_long_rows": len(
        VISIBLE_TOP_10_BOOK_LONG
    ),
    "rejected_book_updates": len(
        REJECTED_BOOK_UPDATES
    ),
    "resynchronizations": len(
        RESYNCHRONIZATION_LEDGER
    ),
    "locked_states": int(
        RECONSTRUCTED_BOOK_STATES[
            "locked_state"
        ].sum()
    ),
    "crossed_states": int(
        RECONSTRUCTED_BOOK_STATES[
            "crossed_state"
        ].sum()
    ),
    "incomplete_states": int(
        RECONSTRUCTED_BOOK_STATES[
            "incomplete_state"
        ].sum()
    ),
    "observed_price_grid": str(
        OBSERVED_PRICE_GRID
    ),
    "depth_stream": DEPTH_STREAM_NAME,
    "nominal_depth_interval_ms": NOMINAL_DEPTH_INTERVAL_MS,
    "stale_interval_threshold_ns": STALE_INTERVAL_THRESHOLD_NS,
    "stale_interval_count": int(
        RECONSTRUCTED_BOOK_STATES[
            "stale_interval"
        ].sum()
    ),
    "partition_order": list(
        FROZEN_PARTITION_ORDER
    ),
    "partition_counts": {
        str(row.partition): int(
            row.observed_reconstructed_count
        )
        for row in (
            RECONSTRUCTION_PARTITION_SUMMARY
            .itertuples(index=False)
        )
    },
    "reference_reconciliation_status": (
        NOTEBOOK_02_REFERENCE_RECONCILIATION_STATUS
    ),
    "reference_warning_count": (
        NOTEBOOK_02_REFERENCE_WARNING_COUNT
    ),
    "v0_0_reference_used_as_authority": False,
    "required_downstream_inputs": {
        record["artifact_type"]: record["relative_path"]
        for record in artifact_records
        if record["artifact_type"]
        in {
            "RECONSTRUCTED_BOOK_STATES",
            "TOP_10_VISIBLE_BOOK_WIDE",
            "VISIBLE_TOP_10_BOOK_LONG",
            "UPDATE_CONTINUITY_REPORT",
            "RECONSTRUCTION_SESSION_LEDGER",
            "BOOK_QUALITY_GATES",
        }
    },
}


NOTEBOOK_02_HANDOFF_PATH: Final[Path] = (
    NOTEBOOK_02_HANDOFF_ROOT
    / (
        f"{OUTPUT_PREFIX}"
        "__02_VISIBLE_BOOK_RECONSTRUCTION"
        "__notebook_02_to_notebook_03_handoff.json"
    )
)

handoff_record = write_json_artifact(
    artifact_type="NOTEBOOK_02_TO_NOTEBOOK_03_HANDOFF",
    payload=NOTEBOOK_02_TO_NOTEBOOK_03_HANDOFF_PAYLOAD,
    path=NOTEBOOK_02_HANDOFF_PATH,
    acceptance_status=NOTEBOOK_02_FINAL_STATUS,
)

artifact_records.append(
    {
        "artifact_type": "NOTEBOOK_02_TO_NOTEBOOK_03_HANDOFF",
        "format": handoff_record["format"],
        "relative_path": handoff_record["relative_path"],
        "size_bytes": handoff_record["size_bytes"],
        "sha256": handoff_record["sha256"],
        "payload_sha256": handoff_record["payload_sha256"],
        "row_count": None,
        "column_count": None,
        "acceptance_status": NOTEBOOK_02_FINAL_STATUS,
    }
)


# ---------------------------------------------------------------------------
# Output manifest
# ---------------------------------------------------------------------------

NOTEBOOK_02_OUTPUT_MANIFEST_PAYLOAD: Final[dict[str, Any]] = {
    "manifest_type": "NOTEBOOK_02_OUTPUT_MANIFEST",
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "source_set_sha256": SOURCE_SET_SHA256,
    "v0_1_run_id": V0_1_RUN_ID,
    "v0_1_run_config_hash": RUN_CONFIG_SHA256,
    "v0_1_run_identity_hash": RUN_IDENTITY_SHA256,
    "producing_notebook": NOTEBOOK_FILE,
    "producing_notebook_name": NOTEBOOK_NAME,
    "notebook_schema_version": NOTEBOOK_SCHEMA_VERSION,
    "operating_mode": OPERATING_MODE,
    "started_at_utc": NOTEBOOK_STARTED_AT_UTC,
    "finished_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "final_status": NOTEBOOK_02_FINAL_STATUS,
    "notebook_03_authorized": NOTEBOOK_03_AUTHORIZED,
    "acceptance_summary": NOTEBOOK_02_ACCEPTANCE_SUMMARY,
    "upstream_inputs": {
        input_name: {
            "path": str(path),
            "sha256": VERIFIED_INPUT_SHA256.get(
                input_name
            ),
        }
        for input_name, path in (
            required_paths.items()
            if "required_paths" in globals()
            else []
        )
    },
    "v0_0_reference": {
        "path": str(
            V0_0_RECONSTRUCTED_BOOK_REFERENCE_PATH
        ),
        "sha256": V0_0_RECONSTRUCTED_BOOK_SHA256,
        "used_as_authority": False,
    },
    "artifacts": artifact_records,
}


NOTEBOOK_02_OUTPUT_MANIFEST_PATH: Final[Path] = (
    NOTEBOOK_02_MANIFEST_ROOT
    / (
        f"{OUTPUT_PREFIX}"
        "__02_VISIBLE_BOOK_RECONSTRUCTION"
        "__notebook_02_output_manifest.json"
    )
)

manifest_record = write_json_artifact(
    artifact_type="NOTEBOOK_02_OUTPUT_MANIFEST",
    payload=NOTEBOOK_02_OUTPUT_MANIFEST_PAYLOAD,
    path=NOTEBOOK_02_OUTPUT_MANIFEST_PATH,
    acceptance_status=NOTEBOOK_02_FINAL_STATUS,
)

NOTEBOOK_02_OUTPUT_MANIFEST_SHA256: Final[str] = (
    manifest_record["sha256"]
)

NOTEBOOK_02_HANDOFF_SHA256: Final[str] = (
    handoff_record["payload_sha256"]
)


# ---------------------------------------------------------------------------
# Verify written artifacts
# ---------------------------------------------------------------------------

written_artifact_audit_rows: list[dict[str, Any]] = []

for artifact_record in artifact_records:
    artifact_path = (
        V0_1_ROOT
        / artifact_record["relative_path"]
    ).resolve(strict=False)

    require_file(
        artifact_path,
        artifact_record["artifact_type"],
    )

    observed_sha256 = sha256_file(
        artifact_path
    )

    written_artifact_audit_rows.append(
        {
            "artifact_type": artifact_record[
                "artifact_type"
            ],
            "relative_path": artifact_record[
                "relative_path"
            ],
            "expected_sha256": artifact_record[
                "sha256"
            ],
            "observed_sha256": observed_sha256,
            "hash_match": observed_sha256
            == artifact_record["sha256"],
            "size_bytes": artifact_path.stat().st_size,
            "acceptance_status": artifact_record[
                "acceptance_status"
            ],
        }
    )

NOTEBOOK_02_WRITTEN_ARTIFACT_AUDIT = pd.DataFrame(
    written_artifact_audit_rows
)

if not NOTEBOOK_02_WRITTEN_ARTIFACT_AUDIT[
    "hash_match"
].all():
    raise RuntimeError(
        "At least one written Notebook 02 artifact failed "
        "post-write hash verification:\n"
        + NOTEBOOK_02_WRITTEN_ARTIFACT_AUDIT.loc[
            ~NOTEBOOK_02_WRITTEN_ARTIFACT_AUDIT[
                "hash_match"
            ]
        ].to_string(index=False)
    )

require_file(
    NOTEBOOK_02_OUTPUT_MANIFEST_PATH,
    "Notebook 02 output manifest",
)

if (
    sha256_file(NOTEBOOK_02_OUTPUT_MANIFEST_PATH)
    != NOTEBOOK_02_OUTPUT_MANIFEST_SHA256
):
    raise RuntimeError(
        "Notebook 02 output manifest failed post-write "
        "hash verification."
    )


# ---------------------------------------------------------------------------
# Compact output
# ---------------------------------------------------------------------------

display(
    NOTEBOOK_02_WRITTEN_ARTIFACT_AUDIT[
        [
            "artifact_type",
            "hash_match",
            "size_bytes",
            "acceptance_status",
            "relative_path",
        ]
    ]
)

{
    "status": NOTEBOOK_02_FINAL_STATUS,
    "notebook_03_authorized": NOTEBOOK_03_AUTHORIZED,
    "artifacts_written": len(
        NOTEBOOK_02_WRITTEN_ARTIFACT_AUDIT
    ) + 1,
    "written_artifacts_verified": int(
        NOTEBOOK_02_WRITTEN_ARTIFACT_AUDIT[
            "hash_match"
        ].sum()
    ),
    "notebook_02_output_manifest_path": str(
        NOTEBOOK_02_OUTPUT_MANIFEST_PATH
    ),
    "notebook_02_output_manifest_sha256": (
        NOTEBOOK_02_OUTPUT_MANIFEST_SHA256
    ),
    "notebook_02_handoff_path": str(
        NOTEBOOK_02_HANDOFF_PATH
    ),
    "notebook_02_handoff_payload_sha256": (
        NOTEBOOK_02_HANDOFF_SHA256
    ),
    "v0_0_reference_used_as_authority": False,
}

,artifact_type,hash_match,size_bytes,acceptance_status,relative_path
0,RECONSTRUCTED_BOOK_STATES,True,50443026,PASS_WITH_REFERENCE_WARNINGS,data\processed\book\BTCUSDT_spot_20260710T0637...
1,TOP_10_VISIBLE_BOOK_WIDE,True,29832139,PASS_WITH_REFERENCE_WARNINGS,data\processed\book\BTCUSDT_spot_20260710T0637...
2,VISIBLE_TOP_10_BOOK_LONG,True,230620822,PASS_WITH_REFERENCE_WARNINGS,data\processed\book\BTCUSDT_spot_20260710T0637...
3,UPDATE_CONTINUITY_REPORT,True,7772482,PASS_WITH_REFERENCE_WARNINGS,artifacts\audit_tables\02_VISIBLE_BOOK_RECONST...
4,REJECTED_BOOK_UPDATES,True,185,PASS_WITH_REFERENCE_WARNINGS,artifacts\audit_tables\02_VISIBLE_BOOK_RECONST...
5,RESYNCHRONIZATION_LEDGER,True,192,PASS_WITH_REFERENCE_WARNINGS,artifacts\audit_tables\02_VISIBLE_BOOK_RECONST...
6,RECONSTRUCTION_SESSION_LEDGER,True,515,PASS_WITH_REFERENCE_WARNINGS,artifacts\audit_tables\02_VISIBLE_BOOK_RECONST...
7,RECONSTRUCTION_PARTITION_SUMMARY,True,625,PASS_WITH_REFERENCE_WARNINGS,artifacts\audit_tables\02_VISIBLE_BOOK_RECONST...
8,BOOK_QUALITY_GATES,True,1003,PASS_WITH_REFERENCE_WARNINGS,artifacts\audit_tables\02_VISIBLE_BOOK_RECONST...
9,BOOK_QUALITY_SUMMARY,True,1300,PASS_WITH_REFERENCE_WARNINGS,artifacts\audit_tables\02_VISIBLE_BOOK_RECONST...


{'status': 'PASS_WITH_REFERENCE_WARNINGS',
 'notebook_03_authorized': True,
 'artifacts_written': 23,
 'written_artifacts_verified': 22,
 'notebook_02_output_manifest_path': 'D:\\Clown Project\\V0.1\\artifacts\\manifests\\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__02_VISIBLE_BOOK_RECONSTRUCTION__notebook_02_output_manifest.json',
 'notebook_02_output_manifest_sha256': '0150ed8386326d30017ed892d82a8980f9c8a3d9ba13fb0e7bc2d01d055a2f8e',
 'notebook_02_handoff_path': 'D:\\Clown Project\\V0.1\\artifacts\\handoff\\BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_20260714T090616Z_e82325081a81__02_VISIBLE_BOOK_RECONSTRUCTION__notebook_02_to_notebook_03_handoff.json',
 'notebook_02_handoff_payload_sha256': 'b13e9c47c46e13cb698aff5497b3cbd12f0e0030368fa755f260a7ee3ed8fe76',
 'v0_0_reference_used_as_authority': False}

In [14]:
# ---------------------------------------------------------------------------
# Final read-back verification and Notebook 02 closeout
# ---------------------------------------------------------------------------

# This cell performs a clean read-back of the saved manifest and handoff.
# It does not write new files.

FINAL_READBACK_REQUIRED_OBJECTS = (
    "NOTEBOOK_02_OUTPUT_MANIFEST_PATH",
    "NOTEBOOK_02_OUTPUT_MANIFEST_SHA256",
    "NOTEBOOK_02_HANDOFF_PATH",
    "NOTEBOOK_02_HANDOFF_SHA256",
    "NOTEBOOK_02_FINAL_STATUS",
    "NOTEBOOK_03_AUTHORIZED",
    "RECONSTRUCTED_BOOK_STATES",
    "TOP_10_VISIBLE_BOOK_WIDE",
    "VISIBLE_TOP_10_BOOK_LONG",
    "UPDATE_CONTINUITY_REPORT",
    "REJECTED_BOOK_UPDATES",
    "RESYNCHRONIZATION_LEDGER",
)

missing_readback_objects = [
    object_name
    for object_name in FINAL_READBACK_REQUIRED_OBJECTS
    if object_name not in globals()
]

if missing_readback_objects:
    raise NameError(
        "Missing required closeout objects: "
        + ", ".join(missing_readback_objects)
    )


# ---------------------------------------------------------------------------
# Cell-local read-back helpers
# ---------------------------------------------------------------------------

def closeout_require_file(
    path: Path,
    *,
    label: str,
) -> Path:
    """Require one existing file for final read-back."""
    resolved = path.resolve(strict=False)

    if not resolved.exists():
        raise FileNotFoundError(
            f"{label} does not exist:\n{resolved}"
        )

    if not resolved.is_file():
        raise RuntimeError(
            f"{label} is not a regular file:\n{resolved}"
        )

    return resolved


def closeout_sha256_file(
    path: Path,
    chunk_size: int = 1 << 20,
) -> str:
    """Hash one file from disk."""
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(chunk_size),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def closeout_canonical_json_sha256(
    value: Any,
) -> str:
    """Hash a JSON-compatible object in canonical form."""
    encoded = json.dumps(
        value,
        sort_keys=True,
        ensure_ascii=False,
        allow_nan=False,
        separators=(",", ":"),
    ).encode("utf-8")

    return hashlib.sha256(encoded).hexdigest()


def closeout_load_json_object(
    path: Path,
    *,
    label: str,
) -> dict[str, Any]:
    """Load a JSON document whose root must be an object."""
    with path.open("r", encoding="utf-8") as handle:
        document = json.load(handle)

    if not isinstance(document, dict):
        raise TypeError(
            f"{label} must contain a JSON object; "
            f"observed {type(document).__name__}."
        )

    return document


def closeout_unwrap_artifact(
    document: dict[str, Any],
    *,
    expected_artifact_type: str,
) -> tuple[dict[str, Any], dict[str, Any]]:
    """Validate and unwrap one saved JSON artifact."""
    metadata = document.get("artifact_metadata")
    payload = document.get("payload")

    if not isinstance(metadata, dict):
        raise RuntimeError(
            f"{expected_artifact_type} lacks artifact_metadata."
        )

    if not isinstance(payload, dict):
        raise RuntimeError(
            f"{expected_artifact_type} lacks a mapping payload."
        )

    observed_artifact_type = metadata.get("artifact_type")

    if observed_artifact_type != expected_artifact_type:
        raise RuntimeError(
            f"Artifact type mismatch for {expected_artifact_type}: "
            f"observed {observed_artifact_type!r}."
        )

    expected_payload_hash = metadata.get("payload_sha256")

    if not isinstance(expected_payload_hash, str):
        raise RuntimeError(
            f"{expected_artifact_type} lacks payload_sha256."
        )

    observed_payload_hash = closeout_canonical_json_sha256(
        payload
    )

    if observed_payload_hash != expected_payload_hash:
        raise RuntimeError(
            f"{expected_artifact_type} payload hash mismatch:\n"
            f"expected: {expected_payload_hash}\n"
            f"observed: {observed_payload_hash}"
        )

    return metadata, payload


def closeout_resolve_v0_1_relative_path(
    relative_path: str,
) -> Path:
    """Resolve a manifest relative path under V0.1."""
    candidate_path = (
        V0_1_ROOT / relative_path
    ).resolve(strict=False)

    try:
        candidate_path.relative_to(
            V0_1_ROOT.resolve(strict=False)
        )
    except ValueError as exc:
        raise RuntimeError(
            "Manifest artifact path escapes V0.1:\n"
            f"{candidate_path}"
        ) from exc

    try:
        candidate_path.relative_to(
            V0_0_ROOT.resolve(strict=False)
        )
    except ValueError:
        return candidate_path

    raise RuntimeError(
        "Manifest artifact path enters immutable V0.0:\n"
        f"{candidate_path}"
    )


# ---------------------------------------------------------------------------
# Read back manifest and handoff
# ---------------------------------------------------------------------------

NOTEBOOK_02_OUTPUT_MANIFEST_PATH = closeout_require_file(
    Path(NOTEBOOK_02_OUTPUT_MANIFEST_PATH),
    label="Notebook 02 output manifest",
)

NOTEBOOK_02_HANDOFF_PATH = closeout_require_file(
    Path(NOTEBOOK_02_HANDOFF_PATH),
    label="Notebook 02 to Notebook 03 handoff",
)

observed_manifest_file_hash = closeout_sha256_file(
    NOTEBOOK_02_OUTPUT_MANIFEST_PATH
)

if observed_manifest_file_hash != NOTEBOOK_02_OUTPUT_MANIFEST_SHA256:
    raise RuntimeError(
        "Notebook 02 output-manifest file hash mismatch:\n"
        f"expected: {NOTEBOOK_02_OUTPUT_MANIFEST_SHA256}\n"
        f"observed: {observed_manifest_file_hash}"
    )

manifest_document = closeout_load_json_object(
    NOTEBOOK_02_OUTPUT_MANIFEST_PATH,
    label="Notebook 02 output manifest",
)

handoff_document = closeout_load_json_object(
    NOTEBOOK_02_HANDOFF_PATH,
    label="Notebook 02 to Notebook 03 handoff",
)

manifest_metadata, manifest_payload = closeout_unwrap_artifact(
    manifest_document,
    expected_artifact_type="NOTEBOOK_02_OUTPUT_MANIFEST",
)

handoff_metadata, handoff_payload = closeout_unwrap_artifact(
    handoff_document,
    expected_artifact_type="NOTEBOOK_02_TO_NOTEBOOK_03_HANDOFF",
)

if manifest_metadata["sha256"] if False else False:
    pass

observed_handoff_payload_hash = closeout_canonical_json_sha256(
    handoff_payload
)

if observed_handoff_payload_hash != NOTEBOOK_02_HANDOFF_SHA256:
    raise RuntimeError(
        "Notebook 02 handoff payload hash mismatch:\n"
        f"expected: {NOTEBOOK_02_HANDOFF_SHA256}\n"
        f"observed: {observed_handoff_payload_hash}"
    )


# ---------------------------------------------------------------------------
# Validate manifest contract and downstream authorization
# ---------------------------------------------------------------------------

expected_manifest_scalars = {
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "source_set_sha256": SOURCE_SET_SHA256,
    "v0_1_run_id": V0_1_RUN_ID,
    "v0_1_run_config_hash": RUN_CONFIG_SHA256,
    "v0_1_run_identity_hash": RUN_IDENTITY_SHA256,
    "producing_notebook": NOTEBOOK_FILE,
    "final_status": NOTEBOOK_02_FINAL_STATUS,
    "notebook_03_authorized": NOTEBOOK_03_AUTHORIZED,
}

for field_name, expected_value in expected_manifest_scalars.items():
    observed_value = manifest_payload.get(field_name)

    if observed_value != expected_value:
        raise RuntimeError(
            f"Manifest field {field_name!r} mismatch:\n"
            f"expected: {expected_value!r}\n"
            f"observed: {observed_value!r}"
        )


expected_handoff_scalars = {
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "source_set_sha256": SOURCE_SET_SHA256,
    "v0_1_run_id": V0_1_RUN_ID,
    "v0_1_run_config_hash": RUN_CONFIG_SHA256,
    "v0_1_run_identity_hash": RUN_IDENTITY_SHA256,
    "producing_notebook": NOTEBOOK_FILE,
    "next_notebook": "03_CAUSAL_TRADE_BOOK_ALIGNMENT.ipynb",
    "notebook_02_final_status": NOTEBOOK_02_FINAL_STATUS,
    "notebook_03_authorized": NOTEBOOK_03_AUTHORIZED,
    "v0_0_reference_used_as_authority": False,
}

for field_name, expected_value in expected_handoff_scalars.items():
    observed_value = handoff_payload.get(field_name)

    if observed_value != expected_value:
        raise RuntimeError(
            f"Handoff field {field_name!r} mismatch:\n"
            f"expected: {expected_value!r}\n"
            f"observed: {observed_value!r}"
        )


if not bool(handoff_payload["notebook_03_authorized"]):
    raise RuntimeError(
        "Notebook 03 is not authorized by the saved handoff."
    )


# ---------------------------------------------------------------------------
# Verify every artifact listed in the manifest
# ---------------------------------------------------------------------------

manifest_artifacts = manifest_payload.get("artifacts")

if not isinstance(manifest_artifacts, list):
    raise RuntimeError(
        "Notebook 02 manifest lacks an artifacts list."
    )

readback_rows: list[dict[str, Any]] = []

for artifact_record in manifest_artifacts:
    if not isinstance(artifact_record, dict):
        raise RuntimeError(
            "Manifest artifacts must be JSON objects."
        )

    artifact_type = str(
        artifact_record["artifact_type"]
    )

    relative_path = str(
        artifact_record["relative_path"]
    )

    artifact_path = closeout_resolve_v0_1_relative_path(
        relative_path
    )

    closeout_require_file(
        artifact_path,
        label=artifact_type,
    )

    observed_artifact_hash = closeout_sha256_file(
        artifact_path
    )

    expected_artifact_hash = str(
        artifact_record["sha256"]
    )

    hash_match = (
        observed_artifact_hash
        == expected_artifact_hash
    )

    readback_rows.append(
        {
            "artifact_type": artifact_type,
            "relative_path": relative_path,
            "format": artifact_record.get("format"),
            "row_count": artifact_record.get("row_count"),
            "size_bytes": artifact_path.stat().st_size,
            "expected_sha256": expected_artifact_hash,
            "observed_sha256": observed_artifact_hash,
            "hash_match": hash_match,
            "acceptance_status": artifact_record.get(
                "acceptance_status"
            ),
        }
    )


NOTEBOOK_02_FINAL_READBACK_AUDIT = pd.DataFrame(
    readback_rows
)

if not NOTEBOOK_02_FINAL_READBACK_AUDIT[
    "hash_match"
].all():
    raise RuntimeError(
        "Final read-back artifact hash verification failed:\n"
        + NOTEBOOK_02_FINAL_READBACK_AUDIT.loc[
            ~NOTEBOOK_02_FINAL_READBACK_AUDIT[
                "hash_match"
            ]
        ].to_string(index=False)
    )


# ---------------------------------------------------------------------------
# Reconcile saved row counts against live in-memory tables
# ---------------------------------------------------------------------------

expected_live_row_counts = {
    "RECONSTRUCTED_BOOK_STATES": len(
        RECONSTRUCTED_BOOK_STATES
    ),
    "TOP_10_VISIBLE_BOOK_WIDE": len(
        TOP_10_VISIBLE_BOOK_WIDE
    ),
    "VISIBLE_TOP_10_BOOK_LONG": len(
        VISIBLE_TOP_10_BOOK_LONG
    ),
    "UPDATE_CONTINUITY_REPORT": len(
        UPDATE_CONTINUITY_REPORT
    ),
    "REJECTED_BOOK_UPDATES": len(
        REJECTED_BOOK_UPDATES
    ),
    "RESYNCHRONIZATION_LEDGER": len(
        RESYNCHRONIZATION_LEDGER
    ),
}

row_count_check_rows: list[dict[str, Any]] = []

for artifact_type, expected_row_count in (
    expected_live_row_counts.items()
):
    matches = NOTEBOOK_02_FINAL_READBACK_AUDIT.loc[
        NOTEBOOK_02_FINAL_READBACK_AUDIT[
            "artifact_type"
        ].eq(artifact_type)
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one manifest entry for {artifact_type}; "
            f"observed {len(matches)}."
        )

    observed_row_count = matches.iloc[0]["row_count"]

    row_count_check_rows.append(
        {
            "artifact_type": artifact_type,
            "manifest_row_count": int(
                observed_row_count
            ),
            "live_row_count": int(
                expected_row_count
            ),
            "row_count_match": int(
                observed_row_count
            )
            == int(expected_row_count),
        }
    )


NOTEBOOK_02_FINAL_ROW_COUNT_READBACK = pd.DataFrame(
    row_count_check_rows
)

if not NOTEBOOK_02_FINAL_ROW_COUNT_READBACK[
    "row_count_match"
].all():
    raise RuntimeError(
        "Saved manifest row counts do not reconcile with "
        "live Notebook 02 tables:\n"
        + NOTEBOOK_02_FINAL_ROW_COUNT_READBACK.loc[
            ~NOTEBOOK_02_FINAL_ROW_COUNT_READBACK[
                "row_count_match"
            ]
        ].to_string(index=False)
    )


# ---------------------------------------------------------------------------
# Notebook 02 terminal status
# ---------------------------------------------------------------------------

NOTEBOOK_02_TERMINAL_STATUS: Final[str] = (
    NOTEBOOK_02_FINAL_STATUS
)

NOTEBOOK_02_CLOSED_AT_UTC: Final[str] = datetime.now(
    timezone.utc
).isoformat()

display(
    NOTEBOOK_02_FINAL_READBACK_AUDIT[
        [
            "artifact_type",
            "format",
            "row_count",
            "hash_match",
            "acceptance_status",
        ]
    ]
)

display(NOTEBOOK_02_FINAL_ROW_COUNT_READBACK)

{
    "status": NOTEBOOK_02_TERMINAL_STATUS,
    "closed_at_utc": NOTEBOOK_02_CLOSED_AT_UTC,
    "manifest_file_hash_verified": True,
    "handoff_payload_hash_verified": True,
    "manifest_artifacts_verified": int(
        NOTEBOOK_02_FINAL_READBACK_AUDIT[
            "hash_match"
        ].sum()
    ),
    "manifest_artifact_count": len(
        NOTEBOOK_02_FINAL_READBACK_AUDIT
    ),
    "row_count_readback_failures": int(
        (
            ~NOTEBOOK_02_FINAL_ROW_COUNT_READBACK[
                "row_count_match"
            ]
        ).sum()
    ),
    "notebook_03_authorized": bool(
        handoff_payload["notebook_03_authorized"]
    ),
    "next_notebook": handoff_payload["next_notebook"],
    "notebook_02_output_manifest_sha256": (
        NOTEBOOK_02_OUTPUT_MANIFEST_SHA256
    ),
    "notebook_02_handoff_payload_sha256": (
        NOTEBOOK_02_HANDOFF_SHA256
    ),
    "v0_0_reference_used_as_authority": False,
}

,artifact_type,format,row_count,hash_match,acceptance_status
0,RECONSTRUCTED_BOOK_STATES,csv,"35,985.0000000000",True,PASS_WITH_REFERENCE_WARNINGS
1,TOP_10_VISIBLE_BOOK_WIDE,csv,"35,985.0000000000",True,PASS_WITH_REFERENCE_WARNINGS
2,VISIBLE_TOP_10_BOOK_LONG,csv,"719,700.0000000000",True,PASS_WITH_REFERENCE_WARNINGS
3,UPDATE_CONTINUITY_REPORT,csv,"35,985.0000000000",True,PASS_WITH_REFERENCE_WARNINGS
4,REJECTED_BOOK_UPDATES,csv,0.0000000000,True,PASS_WITH_REFERENCE_WARNINGS
5,RESYNCHRONIZATION_LEDGER,csv,0.0000000000,True,PASS_WITH_REFERENCE_WARNINGS
6,RECONSTRUCTION_SESSION_LEDGER,csv,1.0000000000,True,PASS_WITH_REFERENCE_WARNINGS
7,RECONSTRUCTION_PARTITION_SUMMARY,csv,4.0000000000,True,PASS_WITH_REFERENCE_WARNINGS
8,BOOK_QUALITY_GATES,csv,22.0000000000,True,PASS_WITH_REFERENCE_WARNINGS
9,BOOK_QUALITY_SUMMARY,csv,5.0000000000,True,PASS_WITH_REFERENCE_WARNINGS


,artifact_type,manifest_row_count,live_row_count,row_count_match
0,RECONSTRUCTED_BOOK_STATES,35985,35985,True
1,TOP_10_VISIBLE_BOOK_WIDE,35985,35985,True
2,VISIBLE_TOP_10_BOOK_LONG,719700,719700,True
3,UPDATE_CONTINUITY_REPORT,35985,35985,True
4,REJECTED_BOOK_UPDATES,0,0,True
5,RESYNCHRONIZATION_LEDGER,0,0,True


{'status': 'PASS_WITH_REFERENCE_WARNINGS',
 'closed_at_utc': '2026-07-14T14:27:17.680482+00:00',
 'manifest_file_hash_verified': True,
 'handoff_payload_hash_verified': True,
 'manifest_artifacts_verified': 22,
 'manifest_artifact_count': 22,
 'row_count_readback_failures': 0,
 'notebook_03_authorized': True,
 'next_notebook': '03_CAUSAL_TRADE_BOOK_ALIGNMENT.ipynb',
 'notebook_02_output_manifest_sha256': '0150ed8386326d30017ed892d82a8980f9c8a3d9ba13fb0e7bc2d01d055a2f8e',
 'notebook_02_handoff_payload_sha256': 'b13e9c47c46e13cb698aff5497b3cbd12f0e0030368fa755f260a7ee3ed8fe76',
 'v0_0_reference_used_as_authority': False}